# FINOR – Index Tracking do S&P 100 e do Ibovespa

**Notebook único do projeto.** Reúne, em um só arquivo, a coleta e a limpeza dos dados, a análise exploratória (EDA), o modelo de otimização (heurística + MIQP) e a Parte 2 do projeto (Guia 2): sete modelos comparados, controle de giro, custos, crises e testes de robustez.

## Objetivo

Construir carteiras *long-only* que repliquem o retorno de um índice com no máximo `K` ações. O modelo principal é um MIQP com restrição de cardinalidade, resolvido no Gurobi a partir de uma solução heurística. O erro de acompanhamento é medido sempre fora da amostra (*walk-forward*, 20 trimestres, set/2021 a set/2026) por três medidas: **MSE** (o erro total que o modelo minimiza; mostramos o RMSE anualizado), **TE** (*tracking error*, a oscilação em torno do índice) e **TD** (*tracking difference*, a direção): RMSE² = TE² + TD²/252.

## Como executar (rápido por padrão)

O notebook foi feito para rodar em **poucos minutos**: os dados limpos e os resultados pesados ficam em cache dentro do repositório, e as importações estão todas na célula de *setup*. Rode as células de cima para baixo, uma vez por sessão.

| Chave (célula de setup) | Padrão | O que faz quando `True` | Tempo |
|---|---|---|---|
| `RECALCULAR_DADOS` | `False` | Refaz a limpeza e as correções a partir de `data/yahoo_raw.pkl` | cerca de 1 min |
| `BAIXAR_YAHOO` | `False` | Baixa os preços de novo do Yahoo Finance (use com `RECALCULAR_DADOS`) | 25 a 30 min |
| `RECALCULAR_MODELO` | `False` | Refaz o *walk-forward* da Parte 1 (heurística + MIQP, Gurobi) | cerca de 25 min |
| `RECALCULAR_PARTE2` | `False` | Refaz a comparação dos 7 modelos, a curva de Pareto e os testes de robustez | cerca de 10 min |

Com tudo em `False`, o notebook lê os caches (`data/`) e os resultados (`results/`) e só refaz tabelas e figuras. O `gurobipy` só é instalado se uma das chaves de modelo estiver ligada.

## Estrutura

1. Dados: coleta, correções auditadas e contrato dos dados (seções 2 a 5)
2. EDA: qualidade, outliers, viés de sobrevivência, comportamento e estrutura dos índices (seções 6 e 7)
3. Apêndice de auditoria (apêndice A)
4. Da análise ao modelo e o modelo de otimização (seções 8 a 13)
5. Parte 2 – Guia 2: sete modelos, giro, custos, crises e robustez (seção 14)
6. Conclusões, limitações, próximos passos e exportação (seções 15 a 19)

## Decisão de escopo

Usamos uma fotografia dos componentes dos índices em **30/09/2026** e sete anos de preços do Yahoo Finance. Isso introduz **viés de sobrevivência**: empresas presentes em 2026 são tratadas como candidatas nos anos anteriores. A seção 6.3 mede o tamanho e a direção desse viés. **Nenhuma ação é descartada**: o modelo usa todas as ações disponíveis, e a análise descritiva também.

## 1. Setup único

Uma única célula com todas as importações, os caminhos e os modos de execução. Em Colab, ela clona o repositório automaticamente.

In [ ]:
# SETUP ÚNICO: importações, caminhos e modos de execução. Rode uma vez por sessão.
import sys, os, json, math, time, warnings, textwrap, subprocess, urllib.request
from pathlib import Path
from itertools import combinations
from math import erfc, sqrt
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter, MaxNLocator, FixedLocator, NullLocator
from matplotlib.colors import LinearSegmentedColormap
from scipy.linalg import solve_triangular
from scipy.optimize import nnls
from scipy.stats import norm
from IPython.display import display, Markdown, Image

# ---------------------------------------------------------------- modos de execução
RECALCULAR_DADOS = False    # refaz as correções (a partir de data/yahoo_raw.pkl)
BAIXAR_YAHOO = False        # baixa o Yahoo Finance de novo (25-30 min; use junto com RECALCULAR_DADOS)
RECALCULAR_MODELO = False   # refaz o walk-forward da Parte 1 (~25 min, precisa do gurobipy)
RECALCULAR_PARTE2 = False   # refaz a comparação dos 7 modelos, o Pareto e a robustez (~10 min)
RECALCULAR_REBALANCEAMENTO = False   # roda o experimento da etapa 13 (rebalanceamento e janela, ~35 min; retoma de onde parou)
RECALCULAR_VALIDACAO_LAMBDA = False  # roda a validação de λ da etapa 12 (48 execuções, ~20 min; retoma de onde parou)

# ---------------------------------------------------------------- caminhos (local ou Google Colab)
REPO_URL = "https://github.com/podorize/index-tracking.git"
EM_COLAB = "google.colab" in sys.modules


def achar_raiz():
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / "src" / "data_loader.py").exists():
            return pasta
    if EM_COLAB:
        destino = Path("/content/index-tracking-finor")
        if not destino.exists():
            subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("Abra o notebook dentro do repositório (a pasta com src/ e data/).")


RAIZ = achar_raiz()
os.chdir(RAIZ)                      # os módulos de src/ usam caminhos relativos (data/, results/, figures/)
sys.path.insert(0, str(RAIZ))
DATA = RAIZ / "data"
PASTA_SAIDA = RAIZ / "results"
PASTA_FIGURAS = RAIZ / "figures"
for pasta in (DATA, PASTA_SAIDA, PASTA_FIGURAS):
    pasta.mkdir(exist_ok=True)

# ---------------------------------------------------------------- bibliotecas opcionais
try:
    import yfinance as yf
except ImportError:
    yf = None
try:
    import gurobipy as gp
    from gurobipy import GRB
except ImportError:
    gp = GRB = None
if (RECALCULAR_MODELO or RECALCULAR_PARTE2 or RECALCULAR_REBALANCEAMENTO or RECALCULAR_VALIDACAO_LAMBDA) and gp is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gurobipy"], check=True)
    import gurobipy as gp
    from gurobipy import GRB
if (BAIXAR_YAHOO or EM_COLAB) and yf is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance", "lxml"], check=True)
    import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

ANUAL = 252


def mostrar_png(nome, largura=1100):
    """Mostra uma figura já salva em figures/ (sem redesenhar)."""
    display(Image(filename=str(PASTA_FIGURAS / nome), width=largura))


print(f"Raiz do projeto: {RAIZ}")
print(f"Modos: dados={RECALCULAR_DADOS}, yahoo={BAIXAR_YAHOO}, modelo={RECALCULAR_MODELO}, parte2={RECALCULAR_PARTE2}, rebalanceamento={RECALCULAR_REBALANCEAMENTO}, validação_lambda={RECALCULAR_VALIDACAO_LAMBDA}")
print("Gurobi:", ".".join(map(str, gp.gurobi.version())) if gp else "não instalado (só é necessário ao recalcular os modelos)")

## 1.1 Estilo dos gráficos

Os gráficos seguem a linguagem visual dos gráficos do *New York Times*, porque serão usados na apresentação:

- **o título diz a conclusão**, e o subtítulo explica o que está sendo mostrado e em que unidade;
- **rótulos diretos** nas linhas e nos pontos que importam, em vez de depender só da legenda;
- **cinza para o contexto e cor só para o que conta a história**: azul para o S&P 100 e laranja para o Ibovespa em todo o notebook, vermelho apenas para destacar extremos negativos;
- grade horizontal discreta, sem molduras, números no padrão brasileiro (vírgula decimal) e a fonte dos dados no rodapé.

A fonte tipográfica é a **Libre Franklin**, uma versão livre da família usada pelo jornal, baixada do repositório do projeto. Se o download falhar, o notebook usa a fonte padrão do matplotlib. A paleta azul e laranja passou nos testes de daltonismo (protanopia e deuteranopia) e de contraste sobre fundo branco.

Cada figura é salva em PNG com 200 dpi, pronta para os slides. Para salvar também em SVG (vetorial), defina `SALVAR_SVG = True`.

In [ ]:
SALVAR_SVG = False
PASTA_FONTES = DATA / "fontes"
URL_FONTES = "https://raw.githubusercontent.com/impallari/Libre-Franklin/master/fonts/TTF/"


def preparar_fonte():
    # Baixa a Libre Franklin (licença SIL Open Font) e registra no matplotlib.
    PASTA_FONTES.mkdir(exist_ok=True)
    try:
        for peso in ["Regular", "Medium", "SemiBold", "Bold"]:
            arquivo = PASTA_FONTES / f"LibreFranklin-{peso}.ttf"
            if not arquivo.exists():
                urllib.request.urlretrieve(URL_FONTES + arquivo.name, arquivo)
            fm.fontManager.addfont(str(arquivo))
        return "Libre Franklin"
    except Exception as exc:
        print(f"Libre Franklin indisponível ({type(exc).__name__}); usando DejaVu Sans.")
        return "DejaVu Sans"


FONTE = preparar_fonte()

# Identidade fixa de cada índice (paleta validada para daltonismo e contraste)
CORES = {"S&P100": "#2a78d6", "IBOV": "#eb6834"}
CORES_CLARAS = {"S&P100": "#9ec5f4", "IBOV": "#f6b79c"}
NOMES = {"S&P100": "S&P 100", "IBOV": "Ibovespa"}
TINTA, TINTA_2, APAGADO = "#121212", "#52514e", "#898781"
GRADE, BASE, FUNDO = "#e1e0d9", "#c3c2b7", "#ffffff"
CONTEXTO, CONTEXTO_ESC = "#d6d4cd", "#a3a19a"
ALERTA = "#d03b3b"
FAIXA = "#f3f2ee"
RAMPA_AZUL = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
              "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
FONTE_DADOS = "Fonte: Yahoo Finance, out/2019 a set/2026. Elaboração própria."

mpl.rcParams.update({
    "font.family": FONTE, "font.size": 11.5,
    "figure.facecolor": FUNDO, "axes.facecolor": FUNDO, "savefig.facecolor": FUNDO,
    "axes.edgecolor": BASE, "axes.linewidth": 0.8, "axes.labelcolor": TINTA_2, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRADE, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.axisbelow": True, "axes.titlesize": 12.5, "axes.titleweight": "semibold",
    "axes.titlelocation": "left", "axes.titlepad": 12, "axes.titlecolor": TINTA,
    "xtick.color": BASE, "ytick.color": BASE, "xtick.labelcolor": TINTA_2, "ytick.labelcolor": TINTA_2,
    "xtick.labelsize": 10.5, "ytick.labelsize": 10.5, "xtick.major.size": 0, "ytick.major.size": 0,
    "xtick.minor.size": 0, "ytick.minor.size": 0, "xtick.major.pad": 6, "ytick.major.pad": 4,
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
    "legend.frameon": False, "legend.fontsize": 10.5, "legend.handlelength": 1.6, "legend.labelcolor": TINTA,
    "figure.dpi": 110, "savefig.dpi": 200, "svg.fonttype": "path",
    "text.parse_math": False,   # "US$" e "R$" aparecem como texto, nunca como fórmula
})


def num(x, casas=1, sinal=False):
    # Número no padrão brasileiro: 1.234,5
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "–"
    texto = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return ("+" + texto) if (sinal and x > 0) else texto.replace("-", "−")


def pct(x, casas=0, sinal=False):
    return num(x, casas, sinal) + "%"


def eixo_pct(ax, eixo="y", casas=None, escala=1.0):
    # Sem "casas", usa 0 casas quando todas as marcas são inteiras e 1 caso contrário (evita rótulos repetidos).
    alvo = ax.yaxis if eixo == "y" else ax.xaxis

    def formatar(v, _):
        c = casas
        if c is None:
            c = 0 if all(abs(m * escala - round(m * escala)) < 1e-9 for m in alvo.get_majorticklocs()) else 1
        return pct(v * escala, c)
    alvo.set_major_formatter(FuncFormatter(formatar))


def eixo_anos(ax):
    import matplotlib.dates as mdates
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))


def rotular_pontos(ax, xs, ys, textos, tam=9.5, cor=TINTA):
    # Rótulos de pontos sem sobreposição: tenta posições ao redor do ponto até achar uma livre.
    # Chamar depois de titulos(), quando o layout já está definido.
    renderer = ax.figure.canvas.get_renderer()
    ocupadas = []
    opcoes = [(8, 5), (8, -9), (-8, 5), (-8, -9), (8, 17), (-8, 17), (8, -21), (-8, -21), (0, 26), (0, -30)]
    for x, y, texto in zip(xs, ys, textos):
        for dx, dy in opcoes:
            rot = ax.annotate(texto, (x, y), xytext=(dx, dy), textcoords="offset points", ha="left" if dx > 0 else ("right" if dx < 0 else "center"),
                              va="center", fontsize=tam, color=cor, zorder=6)
            caixa = rot.get_window_extent(renderer).expanded(1.04, 1.2)
            if not any(caixa.overlaps(o) for o in ocupadas):
                ocupadas.append(caixa)
                break
            rot.remove()
        else:
            ax.annotate(texto, (x, y), xytext=(8, 5), textcoords="offset points", fontsize=tam, color=cor, zorder=6)


def eixo_num(ax, eixo="y", casas=1):
    formato = FuncFormatter(lambda v, _: num(v, casas))
    (ax.yaxis if eixo == "y" else ax.xaxis).set_major_formatter(formato)


def quebrar(texto, largura_pol, tamanho, fator):
    # Quebra linhas longas para que o texto caiba na largura da figura.
    maximo = max(int(largura_pol * 72 / (tamanho * fator)), 30)
    return "\n".join(textwrap.fill(p, maximo) if len(p) > maximo else p for p in texto.split("\n"))


def titulos(fig, titulo, subtitulo=None, fonte=FONTE_DADOS, base_pol=0.75, esquerda=0.07, direita=0.97, espaco_h=None, espaco_w=None):
    # Título (a conclusão), subtítulo (o que se vê) e fonte, alinhados à esquerda, como no NYT.
    largura, altura = fig.get_size_inches()
    titulo = quebrar(titulo, 0.97 * largura, 17, 0.56)
    if subtitulo:
        subtitulo = quebrar(subtitulo, 0.97 * largura, 12, 0.50)
    n_t = titulo.count("\n") + 1
    n_s = 0 if not subtitulo else subtitulo.count("\n") + 1
    topo = 0.22 + 0.34 * n_t + (0.1 + 0.25 * n_s if subtitulo else 0) + 0.3
    ajustes = dict(top=1 - topo / altura, bottom=base_pol / altura, left=esquerda, right=direita)
    if espaco_h is not None:
        ajustes["hspace"] = espaco_h
    if espaco_w is not None:
        ajustes["wspace"] = espaco_w
    fig.subplots_adjust(**ajustes)
    y0 = 1 - 0.2 / altura
    fig.text(0.01, y0, titulo, ha="left", va="top", fontsize=17, fontweight="bold", color=TINTA, linespacing=1.15)
    if subtitulo:
        fig.text(0.01, y0 - (0.34 * n_t + 0.1) / altura, subtitulo, ha="left", va="top", fontsize=12,
                 color=TINTA_2, linespacing=1.35)
    if fonte:
        fig.text(0.01, 0.1 / altura, fonte, ha="left", va="bottom", fontsize=9, color=APAGADO)


def rotulo(ax, x, y, texto, cor=None, dx=8, dy=0, ha="left", va="center", peso="semibold", tam=11, marcador=True, cor_texto=TINTA):
    # Rótulo direto: a marca carrega a cor; o texto fica na cor de texto.
    if marcador and cor is not None:
        ax.plot([x], [y], "o", ms=7, color=cor, mec=FUNDO, mew=2, zorder=8, clip_on=False)
    ax.annotate(texto, (x, y), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, fontsize=tam,
                fontweight=peso, color=cor_texto, zorder=9, annotation_clip=False)


def nota(ax, x, y, texto, dx, dy, ha="left", va="center", tam=10.5, cor=TINTA_2, linha=True):
    # Anotação explicativa com linha-guia fina.
    ax.annotate(texto, (x, y), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, fontsize=tam, color=cor,
                linespacing=1.3, zorder=9, annotation_clip=False,
                arrowprops=dict(arrowstyle="-", color=APAGADO, lw=0.8, shrinkA=3, shrinkB=4) if linha else None)


def legenda(ax, **kw):
    opcoes = dict(loc="upper left", ncol=4, handlelength=1.4, columnspacing=1.4, borderaxespad=0, handletextpad=0.6)
    opcoes.update(kw)
    return ax.legend(**opcoes)


def finalizar_figura(nome, fig=None):
    fig = fig or plt.gcf()
    fig.savefig(PASTA_FIGURAS / nome, dpi=200, bbox_inches="tight", pad_inches=0.25)
    if SALVAR_SVG:
        fig.savefig(PASTA_FIGURAS / nome.replace(".png", ".svg"), bbox_inches="tight", pad_inches=0.25)
    plt.show()
    plt.close(fig)


print(f"Estilo pronto. Fonte: {FONTE}.")

## 2. Configuração declarada antes dos resultados

O período termina em 30/09/2026 e começa exatamente sete anos antes. Como o parâmetro `end` do Yahoo é exclusivo, a consulta termina em 01/10/2026.

O notebook usa retornos simples, porque o retorno de uma carteira é a soma ponderada dos retornos simples dos ativos. As convenções são:

- **S&P 100:** `Close` dos componentes e `Close` de `^OEX`, isto é, retorno de preços;
- **IBOV:** `Adj Close` dos componentes e `Close` de `^BVSP`, aproximando retorno total.

A diferença é intencional: `^OEX` representa o índice de preços, enquanto o IBOV é metodologicamente um índice de retorno total. Assim, cada carteira é comparada a um benchmark internamente coerente. Os níveis de TE dos dois mercados não devem ser interpretados como perfeitamente comparáveis entre si.

In [ ]:
DATA_INICIAL = pd.Timestamp("2019-09-30")
DATA_FINAL = pd.Timestamp("2026-09-30")
DATA_FINAL_EXCLUSIVA = DATA_FINAL + pd.Timedelta(days=1)

MODO_EXECUCAO = "preliminar"  # "rapido", "preliminar" ou "completo"
PERFIS = {
    "rapido":     {"n_janelas": 6,    "time_limit_s": 5,  "K_MIQP": (5, 10, 20)},
    "preliminar": {"n_janelas": None, "time_limit_s": 10, "K_MIQP": (5, 10, 15, 20, 30)},
    "completo":   {"n_janelas": None, "time_limit_s": 60, "K_MIQP": (5, 10, 15, 20, 30)},
}
if MODO_EXECUCAO not in PERFIS:
    raise ValueError("MODO_EXECUCAO inválido.")
CFG = PERFIS[MODO_EXECUCAO]

K_HEURISTICA = (5, 10, 15, 20, 30)   # a heurística cobre a fronteira inteira
K_PRINCIPAL = 10                     # carteira usada nos gráficos de acompanhamento
JANELA_TREINO = 504                  # referência: cerca de 2 anos de pregões
JANELA_ALTERNATIVA = 252             # cerca de 1 ano, testada sem MIQP (seção 8)
HORIZONTE_TESTE = 63                 # cerca de 1 trimestre
COBERTURA_MINIMA_TREINO = 0.98
LIMITE_FFILL_DIAS = 2
MIP_GAP = 0.01
SEED = 42
THREADS_GUROBI = 2
LIMIAR_PESO = 1e-6
USAR_CACHE_MODELO = not RECALCULAR_MODELO   # o cache do modelo fica em data/

print(json.dumps({
    "período": [str(DATA_INICIAL.date()), str(DATA_FINAL.date())],
    "modo": MODO_EXECUCAO,
    "trimestres fora da amostra": CFG["n_janelas"] or "todos",
    "K no MIQP": list(CFG["K_MIQP"]),
    "K na heurística": list(K_HEURISTICA),
    "treino/teste em pregões": [JANELA_TREINO, HORIZONTE_TESTE],
    "janela alternativa (só heurística)": JANELA_ALTERNATIVA,
    "tempo máximo por MIQP (s)": CFG["time_limit_s"],
}, ensure_ascii=False, indent=2))

# Constantes das correções de dados (Etapas 2 e 3), usadas na limpeza e nas tabelas de auditoria
LACUNA_REUTILIZACAO = 60       # Etapa 2: pregões sem dados que caracterizam código (ticker) reaproveitado
LIMIAR_SALTO_PRECO = 0.40      # Etapa 3, teste 1: retorno diário de 40% ou mais...
LIMIAR_INDICE_ESTAVEL = 0.02   # ...com o índice variando menos de 2%

# Decisões da triagem da Etapa 3 (seção 6.2), aplicadas aqui para que todo o notebook use os dados corrigidos.
# Só entram casos apontados por um teste automático e investigados um a um.
# Etapa 3 – correções pela cotação oficial, decididas na triagem (seção 6.2). Para cada caso, os fechamentos oficiais em volta
# da data (B3, séries históricas COTAHIST), já no padrão do preço ajustado do Yahoo: proventos descontados dos preços anteriores
# à data ex. Entre duas datas seguidas do calendário do índice, o retorno do Yahoo é trocado pelo oficial quando os dois diferem
# mais de 2 p.p.; os preços anteriores são reescalados, e o resto da série não muda.
DIVIDENDO_CSMG3 = 6.4876594827            # R$ por ação, dividendo extraordinário com data ex em 20/11/2020
F_CSMG3 = 1 - DIVIDENDO_CSMG3 / 52.80     # fator de ajuste dos preços anteriores à data ex (fechamento de 19/11: R$ 52,80)
COTACOES_OFICIAIS = {
    "UGPA3.SA": {"fonte": "B3 (COTAHIST jun/2021)",
                 "fechamentos": {"2021-06-25": 18.99, "2021-06-28": 19.19}},
    "CSMG3.SA": {"fonte": "B3 (COTAHIST nov/2020), com o dividendo de R$ 6,49 na data ex de 20/11/2020",
                 "fechamentos": {"2020-11-18": 51.49 * F_CSMG3, "2020-11-19": 52.80 * F_CSMG3, "2020-11-20": 44.86, "2020-11-23": 46.23}},
}
TOLERANCIA_OFICIAL = 0.02

## 3. Universo atual congelado

A lista abaixo é uma fotografia de 30/09/2026. Ela fica gravada no notebook para que a amostra não mude quando páginas externas forem atualizadas.

O S&P 100 pode possuir 101 papéis porque uma companhia pode ter duas classes de ações no índice. A quantidade de componentes do IBOV também não é fixada em 100.

A lista define o universo candidato. Ela não afirma que cada papel possui sete anos completos de histórico; essa condição será auditada antes do modelo.

In [ ]:
DATA_FOTOGRAFIA_UNIVERSO = "2026-09-30"
TICKERS_SP100 = ['AAPL', 'ABBV', 'ABT', 'ACN', 'ADBE', 'AMAT', 'AMD', 'AMGN', 'AMT', 'AMZN', 'ANET', 'AVGO', 'AXP', 'BA', 'BAC', 'BKNG', 'BLK', 'BMY', 'BNY', 'BRK-B', 'C', 'CAT', 'CMCSA', 'COF', 'COP', 'COST', 'CRM', 'CSCO', 'CVS', 'CVX', 'DE', 'DELL', 'DHR', 'DIS', 'DUK', 'EMR', 'FDX', 'GD', 'GE', 'GEV', 'GILD', 'GM', 'GOOG', 'GOOGL', 'GS', 'HD', 'IBM', 'INTC', 'INTU', 'ISRG', 'JNJ', 'JPM', 'KO', 'LIN', 'LLY', 'LMT', 'LOW', 'LRCX', 'MA', 'MCD', 'MDLZ', 'MDT', 'META', 'MMM', 'MO', 'MRK', 'MS', 'MSFT', 'MU', 'NEE', 'NFLX', 'NOW', 'NVDA', 'ORCL', 'PANW', 'PEP', 'PFE', 'PG', 'PLTR', 'PM', 'QCOM', 'RTX', 'SBUX', 'SCHW', 'SNDK', 'SO', 'T', 'TMO', 'TMUS', 'TSLA', 'TXN', 'UBER', 'UNH', 'UNP', 'UPS', 'USB', 'V', 'VZ', 'WFC', 'WMT', 'XOM']
TICKERS_IBOV = ['ALOS3.SA', 'ABEV3.SA', 'ASAI3.SA', 'AURE3.SA', 'AXIA3.SA', 'AZZA3.SA', 'B3SA3.SA', 'BBSE3.SA', 'BBDC3.SA', 'BBDC4.SA', 'BRAP4.SA', 'BBAS3.SA', 'BRAV3.SA', 'BPAC11.SA', 'CXSE3.SA', 'CEAB3.SA', 'CMIG4.SA', 'COGN3.SA', 'CSMG3.SA', 'CPLE3.SA', 'CSAN3.SA', 'CPFE3.SA', 'CMIN3.SA', 'CURY3.SA', 'CYRE3.SA', 'DIRR3.SA', 'EMBJ3.SA', 'ENGI11.SA', 'ENEV3.SA', 'EGIE3.SA', 'EQTL3.SA', 'FLRY3.SA', 'GGBR4.SA', 'GOAU4.SA', 'HAPV3.SA', 'HYPE3.SA', 'IGTI11.SA', 'ISAE4.SA', 'ITSA4.SA', 'ITUB4.SA', 'KLBN11.SA', 'RENT3.SA', 'LREN3.SA', 'MGLU3.SA', 'POMO4.SA', 'MBRF3.SA', 'BEEF3.SA', 'MOTV3.SA', 'MRVE3.SA', 'MULT3.SA', 'NATU3.SA', 'PETR3.SA', 'PETR4.SA', 'PSSA3.SA', 'PRIO3.SA', 'RADL3.SA', 'RDOR3.SA', 'RAIL3.SA', 'SBSP3.SA', 'SANB11.SA', 'CSNA3.SA', 'SMFT3.SA', 'SUZB3.SA', 'TAEE11.SA', 'VIVT3.SA', 'TEND3.SA', 'TIMS3.SA', 'TOTS3.SA', 'UGPA3.SA', 'USIM5.SA', 'VALE3.SA', 'VAMO3.SA', 'VBBR3.SA', 'VIVA3.SA', 'WEGE3.SA', 'YDUQ3.SA']
BENCHMARKS = {"S&P100": "^OEX", "IBOV": "^BVSP"}

componentes_atuais = pd.DataFrame(
    [("S&P100", t) for t in TICKERS_SP100] + [("IBOV", t) for t in TICKERS_IBOV],
    columns=["indice", "ticker"],
)
display(componentes_atuais.groupby("indice")["ticker"].agg(["count", "nunique"]))
assert componentes_atuais.duplicated(["indice", "ticker"]).sum() == 0

## 4. Dados: coleta, correções e contrato dos dados (com cache)

Os tickers são baixados em lotes, com uma segunda tentativa apenas para séries que vierem vazias. Mantemos `auto_adjust=False` para obter simultaneamente `Close`, `Adj Close` e `Volume` e escolher conscientemente a convenção adequada a cada índice.

O Yahoo pode apresentar indisponibilidade temporária. Uma série vazia não é removida silenciosamente: ela permanece na auditoria com cobertura zero.

As correções são aplicadas aqui, antes de calcular os retornos, para que todo o notebook use os dados corrigidos. A triagem que as justifica está nas seções 6.1 (Etapa 2) e 6.2 (Etapa 3).

| Etapa | Problema | Regra | Caso nesta base |
|---|---|---|---|
| 2 – séries | Código reaproveitado: a série some por mais de 60 pregões e volta | Manter só o trecho recente, sem as cotações repetidas iniciais | NATU3 (Natura antiga até 2019 e nova listagem em 2025) |
| 3 – retornos | Fator de ajuste (`Adj Close` / `Close`) salta mais de 25% **e** o retorno ajustado passa de 40% com índice estável | Neutralizar o salto do fator, preservando os proventos | nenhum |
| 3 – retornos | Fator salta mais de 25%, mas o retorno ajustado é normal | **Manter**: é um provento grande (o `Close` cai na data ex) | BRAP4, dividendo de R$ 6,04 em 17/12/2021 |
| 3 – retornos | Retorno do Yahoo diferente da cotação oficial da B3 (arquivo COTAHIST) em mais de 2 p.p., confirmado na investigação | Trocar o retorno pelo oficial e reescalar os preços anteriores (`COTACOES_OFICIAIS`) | UGPA3: +119% em 28/06/2021 no Yahoo, +1,05% na B3 (erro de escala). CSMG3: +16,9% em 19/11/2020 no Yahoo, +2,5% na B3 (dividendo de R$ 6,49 descontado um pregão antes da data ex) |

Retornos extremos **reais** não são removidos nem winsorizados: o índice também os sofreu, na proporção do peso de cada ação.

O calendário de cada mercado é o do seu benchmark. Lacunas internas de até dois pregões recebem *forward fill*; o período anterior ao IPO não é preenchido.

Os retornos são calculados sobre os preços **corrigidos** (seção 4.1). Guardamos também os preços originais (`precos_originais`, usados na cobertura) e os preços depois da Etapa 2 (`precos_etapa2`), que geram `R_bruto`: são as séries íntegras, ainda sem as correções da Etapa 3, e é sobre elas que a triagem de erros da seção 6.2 roda.

**Cache:** o resultado desta seção (`bases` e as tabelas de auditoria) fica em `data/cache_dados_finor.pkl`. Os preços brutos ficam em `data/yahoo_raw.pkl`. Com `RECALCULAR_DADOS = False` (padrão), a célula só lê o cache.

In [ ]:
# Dados limpos em cache: carrega em ~1 s. Só refaz a coleta e as correções se RECALCULAR_DADOS = True.
CACHE_DADOS = DATA / "cache_dados_finor.pkl"
YAHOO_BRUTO = DATA / "yahoo_raw.pkl"
NOMES_CACHE_DADOS = ["bases", "auditoria_fator_ajuste", "cenario_ugpa3", "contrato_dados", "diag_ugpa3", "log_correcoes"]

if CACHE_DADOS.exists() and not RECALCULAR_DADOS:
    globals().update(pd.read_pickle(CACHE_DADOS))
    print("Dados carregados do cache:", CACHE_DADOS.name)
    display(contrato_dados)
else:
    def extrair_campo(raw, campo, tickers_lote):
        if raw is None or raw.empty:
            return pd.DataFrame(index=pd.DatetimeIndex([], name="Date"), columns=tickers_lote, dtype=float)
        if isinstance(raw.columns, pd.MultiIndex):
            if campo in raw.columns.get_level_values(0):
                out = raw[campo].copy()
            elif campo in raw.columns.get_level_values(1):
                out = raw.xs(campo, axis=1, level=1).copy()
            else:
                out = pd.DataFrame(index=raw.index)
        else:
            if campo in raw.columns and len(tickers_lote) == 1:
                out = raw[[campo]].rename(columns={campo: tickers_lote[0]})
            else:
                out = pd.DataFrame(index=raw.index)
        if isinstance(out, pd.Series):
            out = out.to_frame(tickers_lote[0])
        out.columns = out.columns.astype(str)
        return out.reindex(columns=tickers_lote)


    def baixar_yahoo(tickers, tamanho_lote=40, pausa=0.5):
        tickers = list(dict.fromkeys(tickers))
        campos = {"Close": [], "Adj Close": [], "Volume": []}
        for inicio in range(0, len(tickers), tamanho_lote):
            lote = tickers[inicio:inicio + tamanho_lote]
            print(f"Baixando lote {inicio // tamanho_lote + 1}: {len(lote)} símbolos...", flush=True)
            raw = yf.download(
                lote, start=DATA_INICIAL, end=DATA_FINAL_EXCLUSIVA,
                auto_adjust=False, actions=False, progress=False,
                group_by="column", threads=True, timeout=40,
            )
            for campo in campos:
                campos[campo].append(extrair_campo(raw, campo, lote))
            time.sleep(pausa)

        paineis = {}
        for campo, partes in campos.items():
            painel = pd.concat(partes, axis=1).sort_index()
            painel = painel.loc[:, ~painel.columns.duplicated()].reindex(columns=tickers)
            painel.index = pd.to_datetime(painel.index).tz_localize(None)
            painel = painel.loc[(painel.index >= DATA_INICIAL) & (painel.index <= DATA_FINAL)]
            paineis[campo] = painel.apply(pd.to_numeric, errors="coerce")

        vazios = [t for t in tickers if paineis["Close"][t].notna().sum() == 0]
        if vazios:
            print(f"Segunda tentativa para {len(vazios)} séries vazias: {vazios}")
        for ticker in vazios:
            try:
                raw = yf.download(
                    ticker, start=DATA_INICIAL, end=DATA_FINAL_EXCLUSIVA,
                    auto_adjust=False, actions=False, progress=False,
                    threads=False, timeout=40,
                )
                for campo in paineis:
                    serie = extrair_campo(raw, campo, [ticker])[ticker]
                    paineis[campo].loc[serie.index, ticker] = serie
            except Exception as exc:
                print(f"Falha persistente em {ticker}: {type(exc).__name__}: {exc}")
        return paineis

    def obter_yahoo():
        # Preços brutos: usa data/yahoo_raw.pkl se existir; baixa do Yahoo só se faltar ou se BAIXAR_YAHOO = True.
        if YAHOO_BRUTO.exists() and not BAIXAR_YAHOO:
            print("Preços brutos carregados de", YAHOO_BRUTO.name)
            return pd.read_pickle(YAHOO_BRUTO)
        painel = baixar_yahoo(TICKERS_SP100 + TICKERS_IBOV + list(BENCHMARKS.values()))
        pd.to_pickle(painel, YAHOO_BRUTO)
        return painel


    yahoo = obter_yahoo()
    print("Dimensões:", {campo: df.shape for campo, df in yahoo.items()})

    DATA_SUSPEITA_UGPA3 = pd.Timestamp("2021-06-28")
    LIMIAR_SALTO_FATOR = 0.25   # variação diária do fator > 25% não é compatível com dividendo usual

    close_ib = yahoo["Close"][TICKERS_IBOV]
    adj_ib = yahoo["Adj Close"][TICKERS_IBOV]
    fator_ajuste = adj_ib / close_ib

    janela_diag = slice(DATA_SUSPEITA_UGPA3 - pd.Timedelta(days=10), DATA_SUSPEITA_UGPA3 + pd.Timedelta(days=10))
    diag_ugpa3 = pd.DataFrame({
        "Close": close_ib.loc[janela_diag, "UGPA3.SA"],
        "Adj Close": adj_ib.loc[janela_diag, "UGPA3.SA"],
        "fator Adj/Close": fator_ajuste.loc[janela_diag, "UGPA3.SA"],
        "retorno Close (%)": 100 * close_ib["UGPA3.SA"].pct_change(fill_method=None).loc[janela_diag],
        "retorno Adj Close (%)": 100 * adj_ib["UGPA3.SA"].pct_change(fill_method=None).loc[janela_diag],
        "retorno ^BVSP (%)": 100 * yahoo["Close"]["^BVSP"].pct_change(fill_method=None).loc[janela_diag],
    })
    r_close_dia = diag_ugpa3["retorno Close (%)"].get(DATA_SUSPEITA_UGPA3, np.nan)
    r_adj_dia = diag_ugpa3["retorno Adj Close (%)"].get(DATA_SUSPEITA_UGPA3, np.nan)
    if abs(r_adj_dia) > 40 and abs(r_close_dia) < 15:
        cenario_ugpa3 = "A: o salto está só no fator de ajuste"
    elif abs(r_close_dia) > 40:
        cenario_ugpa3 = "B: o próprio Close salta (erro no preço)"
    else:
        cenario_ugpa3 = "salto ausente nesta extração (o Yahoo pode ter corrigido a série)"

    # Auditoria do fator de ajuste em todos os componentes do IBOV
    dlog_fator = np.log(fator_ajuste).diff()
    auditoria_fator_ajuste = pd.DataFrame(
        [{"data": d, "ticker": t, "variação log do fator": v}
         for (d, t), v in dlog_fator.stack().items() if abs(v) > np.log1p(LIMIAR_SALTO_FATOR)],
        columns=["data", "ticker", "variação log do fator"],
    )
    auditoria_fator_ajuste["variação do fator (%)"] = 100 * np.expm1(auditoria_fator_ajuste["variação log do fator"])
    r_close_ib = close_ib.pct_change(fill_method=None)
    r_adj_ib = adj_ib.pct_change(fill_method=None)
    auditoria_fator_ajuste["retorno Close (%)"] = [100 * r_close_ib.loc[d, t] for d, t in zip(auditoria_fator_ajuste["data"], auditoria_fator_ajuste["ticker"])]
    auditoria_fator_ajuste["retorno Adj Close (%)"] = [100 * r_adj_ib.loc[d, t] for d, t in zip(auditoria_fator_ajuste["data"], auditoria_fator_ajuste["ticker"])]
    auditoria_fator_ajuste["leitura"] = np.where(
        auditoria_fator_ajuste["retorno Adj Close (%)"].abs() > 40,
        "suspeita de erro de ajuste", "provento: Close cai na data ex, retorno ajustado normal")

    print(f"Diagnóstico UGPA3 em {DATA_SUSPEITA_UGPA3.date()}: cenário {cenario_ugpa3}.")
    print(f"Saltos do fator de ajuste > {LIMIAR_SALTO_FATOR:.0%} no IBOV: {len(auditoria_fator_ajuste)} (detalhes no Apêndice A).")

    yahoo_bruto = {campo: painel.copy() for campo, painel in yahoo.items()}
    log_correcoes = []
    RETORNOS_ANULADOS = []  # (índice, ticker, data) -> retorno vira NaN na preparação dos retornos (sem casos nesta versão)


    def corrigir_fator_ajuste(close, adj, r_indice, limiar=LIMIAR_SALTO_FATOR):
        # Neutraliza saltos do fator Adj/Close que NÃO correspondem a proventos.
        # Provento real: o Close cai na data ex e o retorno ajustado fica normal.
        # Erro de ajuste: o Close fica normal e o retorno ajustado é extremo com índice estável.
        corrigido = adj.copy()
        r_adj = adj.pct_change(fill_method=None)
        for ticker in adj.columns:
            f = (adj[ticker] / close[ticker]).dropna()
            if len(f) < 2:
                continue
            dlog = np.log(f).diff().fillna(0.0)
            saltos = dlog.index[dlog.abs() > np.log1p(limiar)]
            anomalos = [d for d in saltos
                        if abs(r_adj[ticker].get(d, 0.0)) > LIMIAR_SALTO_PRECO
                        and abs(r_indice.get(d, 0.0)) < LIMIAR_INDICE_ESTAVEL]
            for d in saltos:
                if d not in anomalos:
                    log_correcoes.append({
                        "etapa": "3 (retornos)", "indice": "IBOV", "ticker": ticker, "data": d,
                        "regra": "salto do fator mantido (provento)",
                        "detalhe": f"fator variou {100 * np.expm1(dlog.loc[d]):.1f}%; retorno ajustado do dia {100 * r_adj[ticker].get(d, np.nan):.1f}% (normal)",
                    })
            if len(anomalos) == 0:
                continue
            dlog_corr = dlog.copy()
            dlog_corr.loc[anomalos] = 0.0
            soma_futura = dlog_corr[::-1].cumsum()[::-1] - dlog_corr   # Σ das variações posteriores a t
            log_f_corr = np.log(f.iloc[-1]) - soma_futura
            corrigido.loc[f.index, ticker] = close.loc[f.index, ticker] * np.exp(log_f_corr)
            for data in anomalos:
                log_correcoes.append({
                    "etapa": "3 (retornos)", "indice": "IBOV", "ticker": ticker, "data": data,
                    "regra": "salto do fator de ajuste neutralizado",
                    "detalhe": f"fator variou {100 * np.expm1(dlog.loc[data]):.1f}% em um dia",
                })
        return corrigido


    def recortar_tickers_reutilizados(painel, nome_indice, calendario, lacuna=LACUNA_REUTILIZACAO):
        # a lacuna é medida no calendário de pregões do próprio mercado
        saida = painel.copy()
        for ticker in painel.columns:
            s = painel[ticker].reindex(calendario)
            if s.notna().sum() == 0:
                continue
            trecho = s.loc[s.first_valid_index():s.last_valid_index()]
            grupos = trecho.isna().ne(trecho.isna().shift()).cumsum()
            tamanhos = trecho.isna().groupby(grupos).agg(["all", "size"])
            lacunas = tamanhos[tamanhos["all"] & (tamanhos["size"] > lacuna)]
            if lacunas.empty:
                continue
            ultimo_grupo = lacunas.index.max()
            inicio_novo = trecho[grupos > ultimo_grupo].first_valid_index()
            novo = trecho.loc[inicio_novo:].dropna()
            # cotações residuais: sequência inicial de preços idênticos antes da negociação efetiva
            repetidos_iniciais = int(novo.eq(novo.iloc[0]).cummin().sum())
            primeiro_real = novo.index[repetidos_iniciais] if 1 < repetidos_iniciais < len(novo) else novo.index[0]
            saida.loc[:primeiro_real - pd.Timedelta(days=1), ticker] = np.nan
            log_correcoes.append({
                "etapa": "2 (séries)", "indice": nome_indice, "ticker": ticker, "data": primeiro_real,
                "regra": "ticker reutilizado: mantido apenas o trecho recente",
                "detalhe": f"lacuna de {int(lacunas['size'].max())} pregões; série recomeça em {primeiro_real.date()}",
            })
        return saida


    # Etapa 2 – integridade das séries: códigos reaproveitados, no campo usado por cada índice
    yahoo["Adj Close"].loc[:, TICKERS_IBOV] = recortar_tickers_reutilizados(yahoo["Adj Close"][TICKERS_IBOV], "IBOV", yahoo["Close"]["^BVSP"].dropna().index)
    yahoo["Close"].loc[:, TICKERS_SP100] = recortar_tickers_reutilizados(yahoo["Close"][TICKERS_SP100], "S&P100", yahoo["Close"]["^OEX"].dropna().index)
    yahoo_etapa2 = {campo: painel.copy() for campo, painel in yahoo.items()}   # séries íntegras; retornos ainda sem correção

    # Etapa 3 – retornos: (a) fator de ajuste (somente o IBOV usa Adj Close)
    r_ibov_bench = yahoo["Close"]["^BVSP"].pct_change(fill_method=None)
    yahoo["Adj Close"].loc[:, TICKERS_IBOV] = corrigir_fator_ajuste(
        yahoo_bruto["Close"][TICKERS_IBOV], yahoo["Adj Close"][TICKERS_IBOV], r_ibov_bench
    )

    # (b) retornos trocados pela cotação oficial (só se a divergência ainda existir nesta extração do Yahoo)
    calendario_ibov = yahoo["Close"]["^BVSP"].dropna().index
    for ticker, caso in COTACOES_OFICIAIS.items():
        niveis = pd.Series(caso["fechamentos"])
        niveis.index = pd.to_datetime(niveis.index)
        datas = [d for d in niveis.index if d in calendario_ibov]
        for d0, d1 in zip(datas[:-1], datas[1:]):
            serie = yahoo["Adj Close"][ticker]
            r_oficial = niveis[d1] / niveis[d0] - 1
            r_yahoo = serie.get(d1, np.nan) / serie.get(d0, np.nan) - 1
            if pd.notna(r_yahoo) and abs(r_yahoo - r_oficial) > TOLERANCIA_OFICIAL:
                yahoo["Adj Close"].loc[:d1 - pd.Timedelta(days=1), ticker] *= (1 + r_yahoo) / (1 + r_oficial)
                log_correcoes.append({
                    "etapa": "3 (retornos)", "indice": "IBOV", "ticker": ticker, "data": d1,
                    "regra": "retorno trocado pelo da cotação oficial",
                    "detalhe": f"{caso['fonte']}: {100 * r_yahoo:+.1f}% no Yahoo, {100 * r_oficial:+.1f}% na cotação oficial",
                })

    log_correcoes = pd.DataFrame(log_correcoes, columns=["etapa", "indice", "ticker", "data", "regra", "detalhe"])
    n_mantidos = int(log_correcoes["regra"].str.contains("mantido (provento)", regex=False).sum())
    print(f"Registros da auditoria: {len(log_correcoes)} ({len(log_correcoes) - n_mantidos} correções aplicadas, {n_mantidos} proventos mantidos).")
    display(log_correcoes[["etapa", "indice", "ticker", "data", "regra"]])

    def ffill_interno_limitado(painel, limite=2):
        saida = painel.copy()
        for col in saida:
            primeiro = saida[col].first_valid_index()
            ultimo = saida[col].last_valid_index()
            if primeiro is not None and ultimo is not None:
                saida.loc[primeiro:ultimo, col] = saida.loc[primeiro:ultimo, col].ffill(limit=limite)
        return saida


    bases = {}
    especificacao = {
        "S&P100": {"tickers": TICKERS_SP100, "campo_componentes": "Close", "benchmark": "^OEX", "tipo_retorno": "preço"},
        "IBOV": {"tickers": TICKERS_IBOV, "campo_componentes": "Adj Close", "benchmark": "^BVSP", "tipo_retorno": "retorno total aproximado"},
    }

    for nome, spec in especificacao.items():
        p_bench = yahoo["Close"][spec["benchmark"]].dropna().rename(nome)
        calendario = p_bench.index
        # preços originais (cobertura), depois da Etapa 2 (séries íntegras, base da Etapa 3) e corrigidos (análise e modelo)
        p_ativos_original = yahoo_bruto[spec["campo_componentes"]][spec["tickers"]].reindex(calendario)
        p_ativos_etapa2 = yahoo_etapa2[spec["campo_componentes"]][spec["tickers"]].reindex(calendario)
        p_ativos = ffill_interno_limitado(yahoo[spec["campo_componentes"]][spec["tickers"]].reindex(calendario), LIMITE_FFILL_DIAS)
        r_ativos = p_ativos.pct_change(fill_method=None)
        for indice_anulado, ticker, data in RETORNOS_ANULADOS:
            if indice_anulado == nome and data in r_ativos.index:
                r_ativos.loc[data, ticker] = np.nan
        r_bruto = ffill_interno_limitado(p_ativos_etapa2, LIMITE_FFILL_DIAS).pct_change(fill_method=None)
        r_bench = p_bench.pct_change(fill_method=None).rename(nome)
        bases[nome] = {
            "precos_brutos": p_ativos_original,
            "precos_originais": p_ativos_original,
            "precos_etapa2": p_ativos_etapa2,
            "precos": p_ativos,
            "R_bruto": r_bruto,
            "R": r_ativos,
            "y": r_bench,
            "volume": yahoo["Volume"][spec["tickers"]].reindex(calendario),
            "close_negociado": yahoo_bruto["Close"][spec["tickers"]].reindex(calendario),
            "tipo_retorno": spec["tipo_retorno"],
            "campo_yahoo": spec["campo_componentes"],
        }

    contrato_dados = pd.DataFrame([
        {
            "indice": nome,
            "universo": "fotografia fixa em 30/09/2026",
            "fonte preços": "Yahoo Finance",
            "campo componentes": b["campo_yahoo"],
            "benchmark": BENCHMARKS[nome],
            "campo benchmark": "Close",
            "convenção": b["tipo_retorno"],
            "início observado": b["y"].dropna().index.min(),
            "fim observado": b["y"].dropna().index.max(),
            "pregões benchmark": b["y"].notna().sum(),
        }
        for nome, b in bases.items()
    ])
    display(contrato_dados)

    pd.to_pickle({nome_cache: globals()[nome_cache] for nome_cache in NOMES_CACHE_DADOS}, CACHE_DADOS)
    print("Cache de dados gravado em", CACHE_DADOS)

## 6. EDA: roteiro

O EDA responde a seis perguntas, na ordem do guia do projeto. A síntese da seção 7.15 retoma cada uma.

| # | Pergunta | Onde |
|---|---|---|
| 1 | Os dados são confiáveis? O que falta, o que estava errado e o que foi feito? | 6.1 a 6.3 |
| 2 | Como os índices se comportaram e quais crises aparecem nos dados? | 7.2 a 7.4 |
| 3 | Quais ações se parecem mais com o índice, e quais andam juntas? | 7.1, 7.5, 7.6 e 7.8 |
| 4 | Quais setores são mais fáceis de replicar? | 7.7 |
| 5 | Qual índice parece mais difícil de replicar com poucas ações, e por quê? | 7.9 a 7.12 e 7.14 |
| 6 | A carteira seria implementável na prática? | 7.13 |

**Convenções:** retornos simples diários; anualização com 252 pregões; S&P 100 em USD e sem dividendos (como o ^OEX); IBOV em BRL com proventos (como o ^BVSP).

In [ ]:
# Funções e tabelas auxiliares do EDA
SETORES_PT = {
    "Financial Services": "Financeiro", "Technology": "Tecnologia", "Industrials": "Indústria",
    "Healthcare": "Saúde", "Consumer Cyclical": "Consumo cíclico", "Consumer Defensive": "Consumo básico",
    "Utilities": "Utilidades públicas", "Communication Services": "Comunicações", "Energy": "Energia",
    "Basic Materials": "Materiais básicos", "Real Estate": "Imobiliário", "Não informado": "Não informado",
}

def tabela_pct(df, casas=1):
    # Arredonda apenas as colunas numéricas para exibição.
    return df.round({c: casas for c in df.select_dtypes("number").columns})


def drawdown_maximo(r):
    nivel = (1 + r.dropna()).cumprod()
    return (nivel / nivel.cummax() - 1).min()


print("Funções auxiliares prontas.")

## 6.1 Qualidade e cobertura

A cobertura é medida no calendário do respectivo índice. Cobertura baixa pode indicar IPO recente, cisão, mudança de ticker ou falha do provedor. Nenhum ativo é removido aqui: o EDA usa ações com pelo menos 80% de cobertura, e o modelo decide a elegibilidade janela a janela.

In [ ]:
def tabela_qualidade(nome, b):
    precos = b["precos_brutos"]
    linhas = []
    for ticker in precos:
        s = precos[ticker]
        linhas.append({
            "indice": nome,
            "ticker": ticker,
            "observações": int(s.notna().sum()),
            "pregões esperados": len(s),
            "cobertura (%)": 100 * s.notna().mean(),
            "início": s.first_valid_index(),
            "fim": s.last_valid_index(),
            "ausências internas": int(s.loc[s.first_valid_index():s.last_valid_index()].isna().sum()) if s.notna().any() else len(s),
            "volume mediano": b["volume"][ticker].median(),
        })
    return pd.DataFrame(linhas).sort_values(["cobertura (%)", "ticker"])


qualidade = pd.concat([tabela_qualidade(nome, b) for nome, b in bases.items()], ignore_index=True)
display(qualidade.groupby("indice").agg(
    tickers=("ticker", "nunique"),
    cobertura_mediana=("cobertura (%)", "median"),
    com_cobertura_99=("cobertura (%)", lambda s: int((s >= 99).sum())),
    abaixo_de_80=("cobertura (%)", lambda s: int((s < 80).sum())),
    series_vazias=("observações", lambda s: int((s == 0).sum())),
).round(2))
print("Séries com cobertura inferior a 99%:")
display(tabela_pct(qualidade[qualidade["cobertura (%)"] < 99][["indice", "ticker", "cobertura (%)", "início", "ausências internas"]].reset_index(drop=True)))

# Gráfico: quando começa a série de cada ação com histórico incompleto
linhas_graf = []
for nome in ["S&P100", "IBOV"]:
    q = qualidade[(qualidade["indice"] == nome) & (qualidade["cobertura (%)"] < 99)]
    for t in q["ticker"]:
        inicio = bases[nome]["precos"][t].first_valid_index()
        linhas_graf.append({"indice": nome, "ticker": t.replace(".SA", ""), "inicio": inicio})
incompletas = pd.DataFrame(linhas_graf).sort_values(["indice", "inicio"], ascending=[False, True])
n_sp = int((incompletas["indice"] == "S&P100").sum())
n_ib = int((incompletas["indice"] == "IBOV").sum())
completas = qualidade[qualidade["cobertura (%)"] >= 99].groupby("indice")["ticker"].size()

altura = 1.6 + 0.32 * (len(incompletas) + 2)
fig, ax = plt.subplots(figsize=(12, max(5.5, altura)))
y = 0
marcas_y, nomes_y = [], []
for nome in ["S&P100", "IBOV"]:
    g = incompletas[incompletas["indice"] == nome]
    ax.text(DATA_INICIAL, y + 0.15, NOMES[nome], fontsize=12, fontweight="bold", color=TINTA, va="bottom")
    y -= 1
    for _, l in g.iterrows():
        ax.plot([DATA_INICIAL, DATA_FINAL], [y, y], color=FAIXA, lw=7, solid_capstyle="butt", zorder=1)
        ax.plot([l["inicio"], DATA_FINAL], [y, y], color=CORES[nome], lw=7, solid_capstyle="butt", zorder=2)
        ax.text(l["inicio"] - pd.Timedelta(days=25), y, f"{l['ticker']}  ", ha="right", va="center", fontsize=10.5, color=TINTA)
        ax.text(DATA_FINAL + pd.Timedelta(days=20), y, l["inicio"].strftime("%m/%Y"), ha="left", va="center", fontsize=10, color=TINTA_2)
        y -= 1
    y -= 0.6
ax.set_ylim(y + 0.3, 1.2)
ax.set_xlim(DATA_INICIAL - pd.Timedelta(days=200), DATA_FINAL + pd.Timedelta(days=170))
ax.yaxis.set_major_locator(NullLocator())
ax.grid(axis="x", color=GRADE)
ax.grid(axis="y", visible=False)
ax.spines["bottom"].set_visible(False)
ax.text(DATA_FINAL + pd.Timedelta(days=20), 1.0, "início da\nsérie", ha="left", va="bottom", fontsize=9.5, color=APAGADO)
titulos(fig, f"{n_ib} ações do Ibovespa e {n_sp} do S&P 100 não têm os sete anos completos de histórico",
        f"Período com preços de cada ação incompleta (barra colorida). As demais ({completas.get('S&P100', 0)} no S&P 100 e "
        f"{completas.get('IBOV', 0)} no Ibovespa)\ncobrem todo o período. A maioria das incompletas abriu capital entre 2020 e 2022.",
        esquerda=0.12)
finalizar_figura("01_cobertura_dados.png", fig)

In [ ]:
# Etapa 2 – integridade das séries: a série existe, é contínua e é sempre da mesma empresa?
def trechos_parados(precos, minimo=5):
    # Sequências de pelo menos `minimo` pregões seguidos com preço idêntico (retorno exatamente zero),
    # sinal típico de cotação congelada pelo provedor.
    linhas = []
    for ticker in precos.columns:
        s = precos[ticker].dropna()
        if len(s) <= minimo:
            continue
        igual = s.eq(s.shift())
        grupo = (~igual).cumsum()
        tamanho = igual.groupby(grupo).sum()
        for g, n in tamanho[tamanho >= minimo].items():
            datas = s.index[(grupo == g).to_numpy() & igual.to_numpy()]
            linhas.append({"ticker": ticker, "início": datas[0], "fim": datas[-1], "pregões parados": int(n)})
    return pd.DataFrame(linhas, columns=["ticker", "início", "fim", "pregões parados"])


reaproveitados = log_correcoes[log_correcoes["regra"].str.contains("reutilizado", regex=False)]
integridade_series, trechos_parados_etapa2 = [], []
for nome, b in bases.items():
    originais, etapa2 = b["precos_originais"], b["precos_etapa2"]
    parados_orig, parados_et2 = trechos_parados(originais), trechos_parados(etapa2)
    trechos_parados_etapa2.append(parados_et2.assign(indice=nome))
    integridade_series.append({
        "indice": nome,
        "datas duplicadas": int(originais.index.duplicated().sum()),
        "preços nulos ou negativos": int((originais <= 0).sum().sum()),
        "códigos reaproveitados (lacuna > 60 pregões)": ", ".join(reaproveitados.loc[reaproveitados["indice"] == nome, "ticker"]) or "nenhum",
        "trechos parados ≥ 5 pregões (dados originais)": len(parados_orig),
        "trechos parados ≥ 5 pregões (após a Etapa 2)": len(parados_et2),
    })
integridade_series = pd.DataFrame(integridade_series)
trechos_parados_etapa2 = pd.concat(trechos_parados_etapa2, ignore_index=True)
print("Etapa 2 – integridade das séries:")
display(integridade_series)
if len(reaproveitados):
    display(reaproveitados[["indice", "ticker", "data", "detalhe"]])
if len(trechos_parados_etapa2):
    print("Trechos com preço parado depois da Etapa 2 (investigar):")
    display(trechos_parados_etapa2)

**O que a cobertura e a integridade mostram**

- **O S&P 100 é praticamente completo:** 98 dos 101 papéis têm cobertura acima de 99%. As exceções são listagens recentes: SNDK (cisão da Western Digital, desde fev/2025), GEV (cisão da GE, desde mar/2024) e PLTR (IPO em set/2020).
- **O IBOV tem renovação muito maior:** só 64 dos 76 papéis têm histórico completo. Os demais são IPOs e cisões da onda de 2020–2021 (AURE3, IGTI11, SMFT3, CXSE3, ASAI3, CMIN3, VAMO3, RDOR3, BRAV3, CURY3, CEAB3) e a NATU3. **O índice brasileiro de 2026 é bem diferente do de 2019.**
- **Um problema de série:** a NATU3 tem duas empresas sob o mesmo código. A Natura antiga deixou de ser negociada no fim de 2019 e a nova listagem começou em 2025; ligar as duas cria uma queda de 72% em 02/07/2025 que nunca aconteceu. A regra de código reaproveitado (lacuna de mais de 60 pregões) mantém só a série nova.
- **Sem datas duplicadas, preços nulos ou cotações congeladas:** depois do recorte da Natura, nenhuma ação tem 5 ou mais pregões seguidos com preço idêntico. Não há lacunas pontuais relevantes: o *forward fill* de até 2 pregões quase não é acionado.

**Decisão:** nenhum ativo é removido, nem na análise descritiva. O guia do projeto sugere descartar ações com mais de 20% de dados faltantes (cobertura abaixo de 80%), mas não aplicamos esse corte: o modelo usa todas as ações disponíveis e escolhe, trimestre a trimestre, as que têm histórico suficiente (pelo menos 98% dos 504 pregões de treino). Por coerência, a análise descritiva também mantém todas (101 no S&P 100 e 76 no Ibovespa), inclusive as 7 com menos de 80% do histórico (2 no S&P 100 e 5 no Ibovespa), assim como a análise de outliers (seção 6.2) cobre todas as séries. O custo é que as estatísticas de séries curtas, como a SNDK (19 meses), são menos confiáveis.

## 6.2 Etapa 3 – Outliers e erros nos retornos

**Pergunta:** quais retornos diários são erros de dados, quais são movimentos reais extremos e o quanto sabemos sobre cada um?

O ponto de partida é a regra do guia (retorno diário de ±15% ou mais), mas ela sozinha não basta:
1. um erro de dados pode ser menor que 15%;
2. 15% é um dia comum para uma ação muito volátil e raríssimo para uma ação calma;
3. a lista não diz se o movimento veio do mercado ou da empresa.

Por isso a etapa tem quatro passos. Todos usam os retornos depois da Etapa 2 (séries íntegras) e antes das correções desta etapa.

**Passo 1 – Triagem de erros em todos os retornos (cerca de 302 mil).** Cada teste procura a "assinatura" de um tipo conhecido de erro de provedor:

| Teste | Regra | Erro que procura | Por que esse limite |
|---|---|---|---|
| 1. Salto isolado | \|r\| ≥ 40% com o índice variando menos de 2% | erro de escala ou emenda de séries | um salto desse tamanho numa ação, com o mercado parado, é raríssimo |
| 2. Salto que se desfaz | \|r\| ≥ 10% (ou ≥ 8 desvios robustos da própria ação) e o pregão seguinte devolve pelo menos 80%, com o índice variando menos de 2% nos dois dias | preço errado num único dia | 10% é de 4 a 5 vezes a oscilação diária típica; exigir mercado calmo evita confundir com o vaivém das crises (março de 2020) |
| 3. Fator de ajuste | o fator (`Adj Close` ÷ `Close`) muda mais de 25% num dia | dividendo ou desdobramento mal aplicado | dividendos comuns mudam o fator em poucos por cento |

Cada caso apontado é investigado com o mesmo protocolo: preços em torno da data, notícia ou fato relevante e evento corporativo. Por fim, cada caso é conferido na **cotação oficial** (séries históricas da B3, arquivo COTAHIST; Nasdaq para ações americanas): se o preço oficial mostra o mesmo movimento, o evento é real; se não mostra, é erro do provedor e o retorno é trocado pelo oficial. A decisão fica registrada em `DECISOES_TRIAGEM`: **corrigir** ou **manter**.

**Passo 2 – Mercado ou empresa?** Cada extremo de ±15% é dividido em parte de mercado (beta da ação × retorno do índice no dia) e parte própria (o resto). Se a parte de mercado é maior, o extremo foi **puxado pelo mercado**; senão, veio **da empresa ou do setor**. O critério vem dos dados e substitui a janela de datas da Covid e o corte de 4% no índice da versão anterior.

**Passo 3 – Verificação.** Os eventos da empresa são ordenados pelo tamanho da parte própria. Verificamos os 30 maiores e uma amostra aleatória de 20 dos demais (semente fixa). Um evento é confirmado:
- **por notícia:** fonte pública com a mesma empresa, a mesma data, um movimento de tamanho parecido e uma causa coerente (`EVENTOS_VERIFICADOS`);
- **pelos pares:** as 5 ações cujos retornos mais se parecem com os dela se movem na mesma direção no dia. É sinal de choque do setor, não de erro numa série;
- **pela cotação oficial:** quando não há notícia nem pares, o fechamento oficial da B3 ou da Nasdaq mostra o mesmo movimento (`COTACAO_OFICIAL`). Isso prova que o preço se moveu de fato, mas não explica a causa.

A amostra foi sorteada uma vez e fixada (`AMOSTRA_VERIFICACAO`), para que correções feitas depois do sorteio não mudem os eventos sorteados.

A amostra permite uma afirmação honesta sobre os eventos que não verificamos: se nenhum dos 20 sorteados for erro, a taxa de erro entre os não verificados fica abaixo de cerca de 14%, com 95% de confiança.

**Passo 4 – Tratamento e sensibilidade.** Retornos reais não são removidos nem winsorizados: o índice também os sofreu. A seção 13.10 testa se os extremos distorcem a escolha das ações.

As listas completas estão no Apêndice A.3.

In [ ]:
# Etapa 3 – outliers e erros nos retornos.
# Usa R_bruto: séries já íntegras (Etapa 2), antes das correções desta etapa.
LIMITE_OUTLIER = 0.15            # regra do guia: retorno diário de ±15% ou mais
LIMIAR_REVERSAO = 0.10           # teste 2: salto de 10% ou mais...
LIMIAR_REVERSAO_DESVIOS = 8      # ...ou de 8 desvios robustos da própria ação...
FRACAO_DEVOLVIDA = 0.80          # ...que o pregão seguinte devolve em pelo menos 80%
N_TOPO = 30                      # verificação: os 30 maiores pela parte própria do retorno
N_AMOSTRA = 20                   # mais uma amostra aleatória dos demais
SEMENTE_AMOSTRA = 2026
JANELA_COVID = (pd.Timestamp("2020-02-21"), pd.Timestamp("2020-04-30"))   # só para comparar com a regra de datas da versão anterior

# Decisão de cada caso apontado pela triagem, depois da investigação: (índice, ticker, data) -> (decisão, evidência).
# "corrigir" entra em COTACOES_OFICIAIS (seção 4.1): o retorno do Yahoo é trocado pelo da cotação oficial da B3.
DECISOES_TRIAGEM = {
    ('IBOV', 'UGPA3.SA', '2021-06-28'): (
        'corrigir',
        'Teste 1. Cotação oficial (B3, COTAHIST jun/2021): R$ 18,99 em 25/06 e R$ 19,19 em 28/06/2021 (+1,05%). O Yahoo registrava R$ 8,75 → R$ 19,19 (+119%): toda a série anterior estava 2,2 vezes abaixo do nível correto. Corrigido com o retorno oficial do dia (seção 4.1).',
    ),
    ('IBOV', 'HAPV3.SA', '2025-11-13'): (
        'manter',
        'Teste 1. Resultado do 3T25 abaixo do esperado; queda noticiada por Suno e IstoÉ Dinheiro em 13/11/2025.',
    ),
    ('IBOV', 'BRAP4.SA', '2021-12-17'): (
        'manter',
        'Teste 3. Dividendo extraordinário de R$ 6,04 por ação (cerca de 21% do preço): o Close cai na data ex e o retorno ajustado fica normal (+0,6%).',
    ),
    ('S&P100', 'TSLA', '2020-02-04'): (
        'manter',
        'Teste 2. Alta de 13,7% no auge do rali da Tesla (cerca de 40% em dois dias), seguida da maior queda diária da ação até então (−17%) em 05/02/2020. Fontes: https://www.xtb.com/en/market-analysis/breaking-tesla-soars-40-in-just-2-days ; https://www.bloomberg.com/news/articles/2020-02-05/tesla-caution-takes-over-snapping-its-vertical-six-day-rally',
    ),
    ('S&P100', 'LLY', '2026-02-04'): (
        'manter',
        'Teste 2. Alta de cerca de 10% com o resultado do 4T25 e a projeção para 2026; no dia seguinte, queda com a concorrência de versões mais baratas (Hims & Hers). Fontes: https://www.tikr.com/blog/eli-lilly-nyse-lly-stock-spikes-strong-zepbound-demand-robust-2026-sales-forecast ; https://amn.beehiiv.com/p/eli-lilly-booms-then-busts-stellar-guidance-vs-hims-undercut',
    ),
    ('S&P100', 'SNDK', '2026-06-30'): (
        'manter',
        'Teste 2. Cotação oficial (Nasdaq) confirma: US$ 2.050,39 em 29/06, US$ 2.273,73 em 30/06/2026 (+10,89%) e US$ 2.032,22 em 01/07 (−10,62%). Movimento real, num período de oscilações extremas da ação; sem notícia específica encontrada.',
    ),
    ('IBOV', 'CSMG3.SA', '2020-11-19'): (
        'corrigir',
        'Teste 2. Cotação oficial (B3, COTAHIST nov/2020): R$ 51,49 em 18/11, R$ 52,80 em 19/11 (+2,5%) e R$ 44,86 em 20/11/2020, a data ex do dividendo extraordinário de R$ 6,4876594827 por ação (Suno, 19/11/2020: https://www.suno.com.br/noticias/conselho-copasa-csmg3-desdebramento). O Yahoo descontou o dividendo um pregão antes, criando +16,9% em 19/11 e −12,4% no pregão seguinte do calendário. Corrigido com os retornos oficiais (seção 4.1).',
    ),
    ('IBOV', 'BEEF3.SA', '2021-08-11'): (
        'manter',
        'Teste 2. Cotação oficial (B3, COTAHIST ago/2021) confirma: R$ 8,67 em 10/08, R$ 9,94 em 11/08/2021 (+14,65%) e R$ 8,82 em 12/08 (−11,27%). Movimento real; sem notícia específica encontrada.',
    ),
    ('IBOV', 'COGN3.SA', '2021-12-10'): (
        'manter',
        'Teste 2. Cotação oficial (B3, COTAHIST dez/2021) confirma: R$ 2,59 em 09/12, R$ 2,90 em 10/12/2021 (+11,97%) e R$ 2,63 em 13/12 (−9,31%). Movimento real; sem notícia específica encontrada.',
    ),
    ('IBOV', 'TEND3.SA', '2022-10-25'): (
        'manter',
        'Teste 2. Cotação oficial (B3, COTAHIST out/2022) confirma: R$ 5,18 em 24/10, R$ 5,71 em 25/10/2022 (+10,23%) e R$ 5,01 em 26/10 (−12,26%). Movimento real; sem notícia específica encontrada.',
    ),
}

# Eventos confirmados em fonte pública: mesma empresa, mesma data, tamanho parecido e causa coerente.
# "Verificado" quer dizer evidência da época compatível com o movimento, não prova de causalidade.
EVENTOS_VERIFICADOS = {
    ('IBOV', 'HAPV3.SA', '2025-11-13'):
        'Queda de 42,21% após resultado do 3T25 abaixo do esperado; caso já registrado no notebook com fontes Suno e IstoÉ Dinheiro.',
    ('S&P100', 'ORCL', '2025-09-10'):
        'Alta de 35,95% após forte projeção de crescimento da infraestrutura de nuvem impulsionada por IA e anúncio de grandes contratos. Fonte: https://apnews.com/article/502814f43e8a63d01a8f01eec81ba133',
    ('S&P100', 'NFLX', '2022-04-20'):
        'Queda de 35,12% após a primeira perda líquida de assinantes em uma década e desaceleração do crescimento. Fonte: https://www.axios.com/2022/04/19/netflix-stock-plunges-subscriber-loss',
    ('IBOV', 'HAPV3.SA', '2026-08-13'):
        'Queda de 33,09% após resultado fraco do 2T26, com queda de 95,8% no lucro líquido ajustado e pressão sobre margens. Fonte: https://www1.folha.uol.com.br/mercado/2026/08/acoes-da-hapvida-desabam-33-apos-lucro-cair-958-e-fecham-no-menor-nivel-desde-ipo.shtml',
    ('S&P100', 'DELL', '2026-05-29'):
        'Alta de 32,76% após lucro acima das expectativas e elevação do outlook, com forte demanda por computação de IA. Fonte: https://apnews.com/article/82c0b13a9b0bec79cc907cc122cac328',
    ('IBOV', 'HAPV3.SA', '2023-03-01'):
        'Queda de 32,74% após prejuízo inesperado no 4T22, piora da sinistralidade e perspectiva negativa. Fonte: https://www.bloomberglinea.com.br/2023/03/01/hapvida-desaba-mais-de-32-apos-decepcao-com-balanco-e-perspectiva-negativa/',
    ('IBOV', 'HAPV3.SA', '2023-03-09'):
        'Queda de 33,56% em meio a receios de solvência e à avaliação de aumento de capital com emissão de ações. Fonte: https://economia.uol.com.br/noticias/reuters/2023/03/09/ibovespa-fecha-em-queda-com-novo-tombo-de-hapvida.htm',
    ('IBOV', 'HAPV3.SA', '2023-03-10'):
        'Alta de 27,32% em movimento de correção após a queda da véspera ligada ao possível aumento de capital. Fonte: https://www.seudinheiro.com/2023/bolsa-dolar/ibovespa-bolsas-dolar-cambio-rsgp-jsmn-lils-rens-10-03/',
    ('IBOV', 'TEND3.SA', '2022-08-05'):
        'Alta de 31,56% após o balanço do 2T22; apesar do prejuízo, o mercado reagiu à melhora das margens de novas vendas. Fonte: https://exame.com/invest/mercados/acoes-da-tenda-tend3-disparam-quase-30-apesar-de-prejuizo-vale-comprar/',
    ('S&P100', 'PLTR', '2024-02-06'):
        'Alta de 30,80% após resultados do 4T23 e projeção de lucro de 2024 apoiada por forte demanda por produtos de IA. Fonte: https://apnews.com/article/cf2a3df9d160c61d30389e17e9276cad',
    ('S&P100', 'DELL', '2024-03-01'):
        'Alta de 31,62% após lucro e receita acima das expectativas e destaque para a demanda por servidores otimizados para IA. Fonte: https://apnews.com/article/096e6bab846437343ebaabeaf5275ae4',
    ('IBOV', 'PRIO3.SA', '2020-11-19'):
        'Alta de 29,94% após a PetroRio anunciar a compra, da BP, de participações nos campos de Wahoo e Itapu. Fonte: https://www.cnnbrasil.com.br/economia/financas/mercados-hoje-19-11-2020/',
    ('S&P100', 'PANW', '2024-02-21'):
        'Queda de 28,44% após a companhia reduzir sua projeção anual de billings. Fonte: https://apnews.com/article/8d3972b222b64c30b8b3e4828e6edfbb',
    ('S&P100', 'ANET', '2019-11-01'):
        'Queda de 24,23% após guidance do 4T19 abaixo das expectativas, associado à desaceleração de um grande cliente de nuvem. Fonte: https://www.fierce-network.com/telecom/arista-s-stock-stumbles-lower-4q-guidance-due-to-softening-business-from-a-cloud-titan',
    ('S&P100', 'CRM', '2020-08-26'):
        'Alta de 26,04% após resultados muito acima das expectativas e elevação do guidance. Fonte: https://www.axios.com/2020/08/27/salesforce-cut-1000-jobs',
    ('S&P100', 'AVGO', '2024-12-13'):
        'Alta de 24,43% após resultados anuais recordes; receita de IA cresceu 220% no ano. Fonte: https://investors.broadcom.com/news-releases/news-release-details/broadcom-inc-announces-fourth-quarter-and-fiscal-year-2024',
    ('S&P100', 'INTC', '2024-08-02'):
        'Queda de 26,06% após resultado e outlook fracos, plano de corte de cerca de 15 mil postos e suspensão do dividendo. Fonte: https://www.intc.com/news-events/press-releases/detail/1704/intel-reports-second-quarter-2024-financial-results',
    ('S&P100', 'META', '2022-10-27'):
        'Queda de 24,56% após projeção fraca para o trimestre seguinte e preocupação com custos e investimentos no metaverso. Fonte: https://www.investing.com/news/economy/meta-platforms-beats-estimates-for-quarterly-revenue-2922690',
    ('S&P100', 'SBUX', '2024-08-13'):
        'Alta de 24,50% após a nomeação de Brian Niccol, então CEO da Chipotle, como novo CEO e presidente do conselho da Starbucks. Fonte: https://www.axios.com/2024/08/14/starbucks-chipotle-stock',
    ('S&P100', 'UBER', '2020-03-19'):
        'Alta de 38% depois de a empresa afirmar que tinha caixa para atravessar a crise da Covid. Fonte: https://www.cpbj.com/uber-shares-soar-ceo-says-enough-cash-ride-coronavirus-crisis',
    ('S&P100', 'PLTR', '2021-01-22'):
        'Alta de 25%, até a máxima histórica, no rali de janeiro de 2021. Fonte: https://www.businessinsider.nl/palantir-surges-25-to-all-time-highs-up-251-since-ipo/',
    ('S&P100', 'PLTR', '2023-05-09'):
        'Alta de 23% após o resultado do 1T23, com lucro e o lançamento da plataforma de IA. Fonte: https://benzinga.com/news/earnings/23/05/32273478/palantir-delivers-another-profitable-quarter-touts-new-ai-platform-shares-surge',
    ('S&P100', 'PLTR', '2026-08-04'):
        'Alta após o resultado recorde do 2T26 e o forte aumento da projeção. Fonte: https://finviz.com/news/376477/palantir-shares-soar-after-record-quarter-and-major-guidance-upgrade',
    ('S&P100', 'IBM', '2026-07-14'):
        'Queda de 25%, a maior em um dia na história da IBM, após a prévia fraca do 2T26. Fonte: https://www.webull.com/news/15256858202956800',
    ('S&P100', 'SNDK', '2026-01-06'):
        'Alta de 27% após comentários da Nvidia sobre a demanda por memória, na CES. Fonte: https://trendspider.com/blog/sandisk-stock-jumps-after-nvidia-flags-memory-demand/',
    ('S&P100', 'BA', '2020-03-25'):
        'Alta de 24% com o pacote de ajuda dos EUA, que incluía bilhões para a aviação. Fonte: https://www.geekwire.com/2020/boeing-airline-stocks-soar-higher-thanks-billions-dollars-promised-aid/',
    ('IBOV', 'USIM5.SA', '2024-07-26'):
        'Queda de 23% após o balanço do 2T24. Fonte: https://www.infomoney.com.br/mercados/usiminas-usim5-acoes-desabam-mais-de-15-apos-resultado-o-que-frustrou-tanto/',
    ('IBOV', 'CEAB3.SA', '2023-03-03'):
        'Alta de mais de 20% após o resultado do 4T22. Fonte: https://forbes.com.br/forbes-money/2023/03/acao-da-ca-chega-a-disparar-mais-de-20-apos-resultado-trimestral/',
    ('IBOV', 'MGLU3.SA', '2023-11-16'):
        'Alta de cerca de 24% em 16/11/2023, noticiada no dia seguinte. Fonte: https://economicnewsbrasil.com.br/2023/11/17/magazine-luiza-dispara-2443-em-valor-de-mercado/',
    ('S&P100', 'FDX', '2022-09-16'):
        'Queda de 21%, a maior em mais de 40 anos, após a FedEx retirar a projeção anual. Fonte: https://www.bloomberg.com/news/articles/2022-09-15/fedex-pulls-annual-forecast-as-preliminary-results-disappoint',
    ('S&P100', 'INTC', '2026-04-24'):
        'Alta de 23% após o resultado do 1T26 acima do esperado. Fonte: https://www.tikr.com/blog/intel-nasdaq-intc-stock-soars-23-after-smashing-q1-earnings-estimates',
    ('S&P100', 'MSFT', '2026-07-30'):
        'Alta de 15% após o resultado do 4º trimestre do ano fiscal de 2026. Fonte: https://www.tikr.com/blog/microsoft-stock-jumped-15-after-q4-2026-earnings-heres-where-the-stock-could-go',
    ('IBOV', 'MGLU3.SA', '2023-05-16'):
        'Queda de quase 23%, a pior desde o IPO, após o prejuízo do 1T23. Fonte: https://www.suno.com.br/noticias/?p=405043',
    ('IBOV', 'ENEV3.SA', '2026-03-18'):
        'Alta noticiada no próprio dia (12% no momento da notícia; fechamento +15%). Fonte: https://www.moneytimes.com.br/acoes-da-eneva-enev3-disparam-12-nesta-quarta-feira-18-o-que-esta-por-tras-da-alta-igdl/',
    ('S&P100', 'CVS', '2024-05-01'):
        'Queda de 17%, a maior em mais de uma década, após resultado do 1T24 abaixo do esperado e corte da projeção. Fonte: https://www.fool.com/investing/2024/05/01/why-cvs-health-stock-plunged-today/',
    ('S&P100', 'INTU', '2026-05-21'):
        'Queda de 20% após o resultado e o corte da projeção do TurboTax. Fonte: https://investinglive.com/stocks/why-is-intuit-stock-crashing-also-after-its-earnings-last-night-20260521/',
    ('S&P100', 'UBER', '2022-08-02'):
        'Alta de 19% após o resultado do 2T22, com o primeiro fluxo de caixa livre positivo. Fonte: https://www.fool.com/investing/2022/08/02/why-uber-stock-surged-today/',
}

# Eventos sem notícia nem pares, conferidos na cotação oficial (B3 COTAHIST; Nasdaq): o preço se moveu de fato.
COTACAO_OFICIAL = {
    ('S&P100', 'SNDK', '2025-02-18'):
        'Nasdaq: US$ 36,65 em 14/02 e US$ 47,15 em 18/02/2025 (+28,65%), poucos dias depois da separação da Western Digital.',
    ('IBOV', 'CEAB3.SA', '2023-01-20'):
        'B3 (COTAHIST jan/2023): R$ 2,50 em 19/01 e R$ 3,07 em 20/01/2023 (+22,80%).',
    ('IBOV', 'YDUQ3.SA', '2020-04-07'):
        'B3 (COTAHIST abr/2020): R$ 22,21 em 06/04 e R$ 26,50 em 07/04/2020 (+19,32%).',
    ('IBOV', 'MBRF3.SA', '2020-03-10'):
        'B3 (COTAHIST mar/2020, código MRFG3 na época): R$ 8,57 em 09/03 e R$ 9,92 em 10/03/2020 (+15,75%).',
    ('IBOV', 'MOTV3.SA', '2020-03-10'):
        'B3 (COTAHIST mar/2020, código CCRO3 na época): R$ 12,70 em 09/03 e R$ 14,90 em 10/03/2020 (+17,32%).',
    ('S&P100', 'LOW', '2020-03-16'):
        'Nasdaq: US$ 96,49 em 13/03 e US$ 72,59 em 16/03/2020 (−24,77%).',
    ('IBOV', 'YDUQ3.SA', '2020-03-26'):
        'B3 (COTAHIST mar/2020): R$ 26,79 em 25/03 e R$ 32,34 em 26/03/2020 (+20,72%).',
    ('S&P100', 'TSLA', '2020-03-19'):
        'Nasdaq: US$ 24,08 em 18/03 e US$ 28,51 em 19/03/2020 (+18,39%; preços ajustados pelos desdobramentos).',
    ('IBOV', 'CSNA3.SA', '2020-04-29'):
        'B3 (COTAHIST abr/2020): R$ 7,81 em 28/04 e R$ 9,03 em 29/04/2020 (+15,62%).',
}

# Amostra da verificação: sorteada uma vez (semente 2026) e fixada, para que correções posteriores não mudem os eventos sorteados.
AMOSTRA_VERIFICACAO = [
    ('IBOV', 'YDUQ3.SA', '2020-03-31'),
    ('S&P100', 'FDX', '2022-09-16'),
    ('S&P100', 'INTC', '2026-04-24'),
    ('IBOV', 'YDUQ3.SA', '2020-04-07'),
    ('S&P100', 'MSFT', '2026-07-30'),
    ('S&P100', 'SNDK', '2026-02-04'),
    ('IBOV', 'MBRF3.SA', '2020-03-10'),
    ('IBOV', 'MOTV3.SA', '2020-03-10'),
    ('S&P100', 'LOW', '2020-03-16'),
    ('S&P100', 'LRCX', '2026-07-30'),
    ('IBOV', 'MGLU3.SA', '2023-05-16'),
    ('IBOV', 'ENEV3.SA', '2026-03-18'),
    ('IBOV', 'YDUQ3.SA', '2020-03-26'),
    ('S&P100', 'CVS', '2024-05-01'),
    ('S&P100', 'TSLA', '2020-03-19'),
    ('S&P100', 'INTU', '2026-05-21'),
    ('S&P100', 'GM', '2020-03-18'),
    ('S&P100', 'UBER', '2022-08-02'),
    ('IBOV', 'USIM5.SA', '2020-05-27'),
    ('IBOV', 'CSNA3.SA', '2020-04-29'),
]


def painel(serie, modelo):
    # Repete uma série diária em todas as colunas de um painel (para comparar ação a ação).
    return pd.DataFrame(np.repeat(serie.reindex(modelo.index).to_numpy()[:, None], modelo.shape[1], axis=1),
                        index=modelo.index, columns=modelo.columns)


def chave(indice, ticker, data):
    return (indice, ticker, pd.Timestamp(data).strftime("%Y-%m-%d"))


triagem, extremos, armadilha = [], [], {}
for nome, b in bases.items():
    Rb, Rc = b["R_bruto"], b["R"]
    y = b["y"].reindex(Rb.index)
    beta = Rc.apply(lambda s: s.cov(y) / y[s.notna()].var())
    mercado = painel(y, Rb).mul(beta, axis=1)            # parte do retorno explicada pelo índice
    propria = Rb - mercado                               # parte própria da ação (ou do setor)
    propria_corr = Rc - painel(y, Rc).mul(beta, axis=1)
    escala = (propria_corr - propria_corr.median()).abs().median() * 1.4826   # desvio robusto de cada ação (MAD)
    desvios = propria / escala
    calmo_hoje = painel(y.abs() < LIMIAR_INDICE_ESTAVEL, Rb).astype(bool)
    calmo_amanha = painel(y.shift(-1).abs() < LIMIAR_INDICE_ESTAVEL, Rb).astype(bool)
    seguinte = Rb.shift(-1)
    devolve = (np.sign(seguinte) == -np.sign(Rb)) & ((-np.log1p(seguinte)) / np.log1p(Rb) >= FRACAO_DEVOLVIDA)
    # Para comparação: o mesmo teste 2 sem exigir mercado calmo (a regra clássica de "reversão = erro")
    sem_calma = (((Rb.abs() >= LIMIAR_REVERSAO) | (desvios.abs() >= LIMIAR_REVERSAO_DESVIOS)) & devolve).stack()
    sem_calma = sem_calma[sem_calma].index.get_level_values(0)
    armadilha[nome] = (len(sem_calma), int(((sem_calma >= JANELA_COVID[0]) & (sem_calma <= JANELA_COVID[1])).sum()))
    testes = {
        "1. Salto isolado": (Rb.abs() >= LIMIAR_SALTO_PRECO) & calmo_hoje,
        "2. Salto que se desfaz": ((Rb.abs() >= LIMIAR_REVERSAO) | (desvios.abs() >= LIMIAR_REVERSAO_DESVIOS))
                                  & devolve & calmo_hoje & calmo_amanha,
    }
    for teste, marca in testes.items():
        s = marca.stack()
        for data, ticker in s[s].index:
            triagem.append({"teste": teste, "indice": nome, "ticker": ticker, "data": data,
                            "retorno (%)": 100 * Rb.loc[data, ticker], "retorno do índice (%)": 100 * y.loc[data],
                            "retorno no pregão seguinte (%)": 100 * seguinte.loc[data, ticker]})

    # Pares estatísticos: as 5 ações cujos retornos diários mais se parecem com os de cada ação (mesmo índice)
    corr = Rc.corr(min_periods=120)
    corr = corr.mask(np.eye(len(corr), dtype=bool))
    pares = {t: corr[t].dropna().nlargest(5).index.tolist() for t in corr.columns}

    pilha = Rb.stack()
    # arredondar antes de comparar: retornos de exatamente 15% (ex.: LREN3 em 19/03/2020) não
    # podem entrar ou sair da lista por revisões no último dígito do preço ajustado
    for (data, ticker), r in pilha[pilha.round(6).abs() >= LIMITE_OUTLIER].items():
        p_m, p_p = mercado.loc[data, ticker], propria.loc[data, ticker]
        mediana_pares = propria_corr.loc[data, pares[ticker]].median() if pares[ticker] else np.nan
        extremos.append({
            "indice": nome, "data": data, "ticker": ticker, "retorno (%)": 100 * r,
            "retorno do índice (%)": 100 * y.loc[data], "beta": beta[ticker],
            "parte de mercado (%)": 100 * p_m, "parte própria (%)": 100 * p_p,
            "desvios robustos": desvios.loc[data, ticker],
            "retorno no pregão seguinte (%)": 100 * seguinte.loc[data, ticker],
            "pares": ", ".join(t.replace(".SA", "") for t in pares[ticker]),
            "parte própria mediana dos pares (%)": 100 * mediana_pares,
            "pares confirmam": bool(pd.notna(mediana_pares) and np.sign(mediana_pares) == np.sign(p_p)
                                    and abs(mediana_pares) >= abs(p_p) / 3),
            "na janela da Covid": bool(JANELA_COVID[0] <= data <= JANELA_COVID[1]),
        })

# Teste 3: saltos do fator de ajuste (seção 4.1; só o IBOV usa Adj Close)
for _, l in auditoria_fator_ajuste.iterrows():
    triagem.append({"teste": "3. Fator de ajuste", "indice": "IBOV", "ticker": l["ticker"], "data": pd.Timestamp(l["data"]),
                    "retorno (%)": l["retorno Adj Close (%)"],
                    "retorno do índice (%)": 100 * bases["IBOV"]["y"].get(pd.Timestamp(l["data"]), np.nan),
                    "retorno no pregão seguinte (%)": np.nan})

triagem = pd.DataFrame(triagem, columns=["teste", "indice", "ticker", "data", "retorno (%)", "retorno do índice (%)",
                                         "retorno no pregão seguinte (%)"]).sort_values(["teste", "indice", "data"]).reset_index(drop=True)
chaves_t = [chave(i, t, d) for i, t, d in zip(triagem["indice"], triagem["ticker"], triagem["data"])]
triagem["decisão"] = [DECISOES_TRIAGEM.get(k, ("sem decisão", ""))[0] for k in chaves_t]
triagem["evidência"] = [DECISOES_TRIAGEM.get(k, ("", "Investigar: preços em volta da data, notícia, evento corporativo e segunda fonte."))[1]
                        for k in chaves_t]
print(f"Passo 1 – triagem de erros em {sum(int(b['R_bruto'].notna().sum().sum()) for b in bases.values()):,} retornos diários:".replace(",", "."))
with pd.option_context("display.max_colwidth", 120):
    display(tabela_pct(triagem.drop(columns="evidência"), 1))
n_sem_calma, n_sem_calma_covid = (sum(v[i] for v in armadilha.values()) for i in (0, 1))
print(f"Sem a exigência de mercado calmo, o teste 2 apontaria {n_sem_calma} casos, {n_sem_calma_covid} deles no crash da Covid "
      f"(21/02–30/04/2020): a regra clássica de reversão confundiria o vaivém real do mercado com erro.")
pendentes = triagem[triagem["decisão"] == "sem decisão"]
if len(pendentes):
    print(f"ATENÇÃO: {len(pendentes)} caso(s) sem decisão registrada em DECISOES_TRIAGEM. Investigue antes de seguir.")
corrigir_triagem = {k for k, (d, _) in DECISOES_TRIAGEM.items() if d == "corrigir"}
assert {k[1] for k in corrigir_triagem} <= set(COTACOES_OFICIAIS), "Há decisão 'corrigir' sem cotação oficial na seção 4.1."

# Passo 2 – mercado ou empresa?
outliers = pd.DataFrame(extremos)
chaves_o = [chave(i, t, d) for i, t, d in zip(outliers["indice"], outliers["ticker"], outliers["data"])]
outliers["origem"] = np.select(
    [np.array([k in corrigir_triagem for k in chaves_o]),
     (outliers["parte de mercado (%)"].abs() >= outliers["parte própria (%)"].abs()).to_numpy()],
    ["Erro de dados corrigido", "Puxado pelo mercado"], default="Da empresa ou do setor")
outliers["evidência"] = [EVENTOS_VERIFICADOS.get(k) or COTACAO_OFICIAL.get(k) or DECISOES_TRIAGEM.get(k, ("", ""))[1] for k in chaves_o]
outliers = outliers.sort_values(["origem", "data"]).reset_index(drop=True)
ORDEM_ORIGEM = ["Puxado pelo mercado", "Da empresa ou do setor", "Erro de dados corrigido"]
tabela_decisoes_outliers = (outliers.pivot_table(index="origem", columns="indice", values="ticker", aggfunc="size", fill_value=0)
                            .reindex(ORDEM_ORIGEM).fillna(0).astype(int).assign(total=lambda t: t.sum(axis=1)))
print(f"\nPasso 2 – {len(outliers)} retornos de ±{LIMITE_OUTLIER:.0%} ou mais, por origem:")
display(tabela_decisoes_outliers)
covid_x_origem = pd.crosstab(outliers["na janela da Covid"].map({True: "na janela da Covid (21/02–30/04/2020)", False: "fora da janela"}),
                             outliers["origem"])
print("Comparação com a regra de datas da versão anterior:")
display(covid_x_origem)

# Passo 3 – verificação: os maiores pela parte própria e uma amostra aleatória dos demais
empresa = outliers[outliers["origem"] == "Da empresa ou do setor"].assign(_abs=lambda t: t["parte própria (%)"].abs())
empresa = empresa.sort_values(["_abs", "indice", "ticker"], ascending=[False, True, True])
topo = empresa.head(N_TOPO).assign(seleção=f"{N_TOPO} maiores")
resto = empresa.iloc[N_TOPO:].sort_values(["indice", "data", "ticker"])
chaves_resto = [chave(i, t, d) for i, t, d in zip(resto["indice"], resto["ticker"], resto["data"])]
fixada = [tuple(k) for k in AMOSTRA_VERIFICACAO]
if fixada and set(fixada) <= set(chaves_resto):
    amostra = resto[[k in set(fixada) for k in chaves_resto]].assign(seleção="amostra aleatória")
else:   # base diferente (por exemplo, outra extração do Yahoo): sorteia de novo e a verificação precisa ser refeita
    amostra = resto.sample(n=min(N_AMOSTRA, len(resto)), random_state=SEMENTE_AMOSTRA).assign(seleção="amostra aleatória")
    print("ATENÇÃO: a amostra fixada não está na base atual; nova amostra sorteada, a verificação precisa ser refeita.")
verificacao = pd.concat([topo, amostra]).drop(columns="_abs").reset_index(drop=True)
chaves_v = [chave(i, t, d) for i, t, d in zip(verificacao["indice"], verificacao["ticker"], verificacao["data"])]
verificacao["confirmação"] = np.select(
    [np.array([k in EVENTOS_VERIFICADOS for k in chaves_v]), verificacao["pares confirmam"].to_numpy(),
     np.array([k in COTACAO_OFICIAL for k in chaves_v])],
    ["notícia", "pares (movimento do setor)", "cotação oficial"], default="sem confirmação")
verificacao["evidência"] = [EVENTOS_VERIFICADOS.get(k) or COTACAO_OFICIAL.get(k) or e for k, e in zip(chaves_v, verificacao["evidência"])]
resumo_verificacao = verificacao.pivot_table(index="seleção", columns="confirmação", values="ticker", aggfunc="size", fill_value=0)
print(f"\nPasso 3 – verificação ({N_TOPO} maiores pela parte própria + {N_AMOSTRA} sorteados entre os outros {len(resto)}):")
display(resumo_verificacao)
n_amostra = int((verificacao["seleção"] == "amostra aleatória").sum())
confirmados_amostra = int(((verificacao["seleção"] == "amostra aleatória") & (verificacao["confirmação"] != "sem confirmação")).sum())
LIMITE_ERRO_AMOSTRA = 100 * (1 - 0.05 ** (1 / n_amostra))   # "regra de três": 0 erros em n casos sorteados
sorteados = verificacao[verificacao["seleção"] == "amostra aleatória"]
com_sinal = sum(chave(i, t, d) in set(chaves_t) for i, t, d in zip(sorteados["indice"], sorteados["ticker"], sorteados["data"]))
sem_conf = sorteados[sorteados["confirmação"] == "sem confirmação"]
condicao = ("" if len(sem_conf) == 0 else
            f" Isso supõe que os {len(sem_conf)} sorteados sem confirmação também sejam reais.")
print(f"Amostra aleatória: {confirmados_amostra} de {n_amostra} confirmados; {len(sem_conf)} sem confirmação; "
      f"{com_sinal} com sinal de erro nos testes 1 e 2. Nenhum erro entre os sorteados: a taxa de erro entre os {len(resto)} "
      f"eventos não verificados fica abaixo de {num(LIMITE_ERRO_AMOSTRA, 0)}% (95% de confiança).{condicao}")
with pd.option_context("display.max_colwidth", 60):
    display(tabela_pct(verificacao[["seleção", "indice", "data", "ticker", "retorno (%)", "retorno do índice (%)", "parte própria (%)",
                                    "parte própria mediana dos pares (%)", "confirmação"]], 1))

# Gráfico 1: extremos por mês e por origem
meses = pd.period_range(DATA_INICIAL.to_period("M"), DATA_FINAL.to_period("M"), freq="M")
por_mes = (outliers.assign(mes=outliers["data"].dt.to_period("M")).pivot_table(index="mes", columns="origem", values="ticker",
           aggfunc="size", fill_value=0).reindex(meses, fill_value=0).reindex(columns=ORDEM_ORIGEM, fill_value=0))
cores_origem = {"Puxado pelo mercado": CONTEXTO_ESC, "Da empresa ou do setor": TINTA_2, "Erro de dados corrigido": ALERTA}
fig, ax = plt.subplots(figsize=(12, 5.6))
x = por_mes.index.to_timestamp()
base_barra = np.zeros(len(por_mes))
for origem in ORDEM_ORIGEM:
    ax.bar(x, por_mes[origem].to_numpy(), bottom=base_barra, width=24, color=cores_origem[origem], label=origem, zorder=2)
    base_barra += por_mes[origem].to_numpy()
ax.set_ylim(0, base_barra.max() * 1.15)
ax.yaxis.set_major_locator(MaxNLocator(5, integer=True))
legenda(ax, loc="upper right", ncol=1)
pico = por_mes.sum(axis=1).idxmax()
n_covid = int(outliers["na janela da Covid"].sum())
nota(ax, pico.to_timestamp(), base_barra.max(), f"Crash da Covid (fev–abr/2020): {n_covid} casos,\n"
     f"{int(((outliers['na janela da Covid']) & (outliers['origem'] == 'Da empresa ou do setor')).sum())} deles dominados pela própria ação", 30, -12)
n_m = int((outliers["origem"] == "Puxado pelo mercado").sum())
n_e = int((outliers["origem"] == "Da empresa ou do setor").sum())
titulos(fig, f"{n_m} dos {len(outliers)} extremos foram puxados pelo mercado; {n_e} vieram da própria empresa ou do seu setor",
        f"Retornos diários de ±15% ou mais por mês, nos dois índices, depois da Etapa 2. Cada retorno é dividido em parte de mercado "
        f"(beta × retorno do índice) e parte própria; a maior das duas define a origem. Em vermelho, os erros de dados corrigidos com a "
        f"cotação oficial ({' e '.join(sorted(t.replace('.SA', '') for t in outliers.loc[outliers['origem'] == 'Erro de dados corrigido', 'ticker']))}).")
finalizar_figura("01b_outliers_por_mes.png", fig)

# Gráfico 2: a decomposição de cada extremo
fig, ax = plt.subplots(figsize=(11.5, 7.0))
lim_x = max(abs(outliers["parte de mercado (%)"]).max() * 1.1, 10)
lim_y = 50
xs = np.linspace(-lim_x, lim_x, 50)
ax.fill_between(xs, -np.abs(xs), np.abs(xs), color=FAIXA, zorder=0)
ax.plot(xs, np.abs(xs), color=BASE, lw=0.9, zorder=1)
ax.plot(xs, -np.abs(xs), color=BASE, lw=0.9, zorder=1)
ax.axhline(0, color=BASE, lw=0.8, zorder=1)
ax.axvline(0, color=BASE, lw=0.8, zorder=1)
for nome in ["S&P100", "IBOV"]:
    g = outliers[(outliers["indice"] == nome) & (outliers["origem"] != "Erro de dados corrigido")]
    ax.scatter(g["parte de mercado (%)"], g["parte própria (%)"].clip(-lim_y, lim_y), s=26, color=CORES[nome], alpha=0.75,
               edgecolor=FUNDO, linewidth=0.6, label=NOMES[nome], zorder=3)
for _, l in outliers[outliers["origem"] == "Erro de dados corrigido"].iterrows():
    ax.scatter([l["parte de mercado (%)"]], [min(l["parte própria (%)"], lim_y)], s=60, marker="^", color=ALERTA, zorder=4)
    fora = abs(l["parte própria (%)"]) > lim_y
    nota(ax, l["parte de mercado (%)"], min(l["parte própria (%)"], lim_y),
         f"{l['ticker'].replace('.SA', '')} {num(l['retorno (%)'], 0, sinal=True)}%: erro de dados\n(corrigido{'; fora da escala' if fora else ''})",
         -22 if fora else -95, -4 if fora else 4, ha="right", va="top" if fora else "center")
destaques = outliers[outliers["origem"] != "Erro de dados corrigido"].assign(_a=lambda t: t["parte própria (%)"].abs()).nlargest(2, "_a")
for _, l in destaques.iterrows():
    nota(ax, l["parte de mercado (%)"], l["parte própria (%)"],
         f"{l['ticker'].replace('.SA', '')} {num(l['retorno (%)'], 0, sinal=True)}% ({l['data']:%d/%m/%Y})", 25 if l["parte própria (%)"] < 0 else 25,
         -12 if l["parte própria (%)"] < 0 else 10)
ax.text(lim_x * 0.97, -lim_x * 0.55, "faixa cinza: a parte\nde mercado é maior", ha="right", va="top", fontsize=10, color=APAGADO)
ax.set_xlim(-lim_x, lim_x)
ax.set_ylim(-lim_y, lim_y)
eixo_pct(ax, "x", casas=0)
eixo_pct(ax, "y", casas=0)
ax.grid(axis="x", color=GRADE)
ax.set_xlabel("Parte de mercado: beta × retorno do índice no dia")
ax.set_ylabel("Parte própria da ação")
legenda(ax, loc="lower left", ncol=2)
titulos(fig, f"Mercado ou empresa? {n_m} extremos ficam na faixa cinza, onde o índice explica mais que a própria ação",
        "Cada ponto é um retorno diário de ±15% ou mais, dividido em parte de mercado e parte própria. Na faixa cinza, a parte de mercado é "
        "maior: são os dias de crise, como o crash da Covid e as tarifas de abril/2025. Fora dela, o movimento é da empresa ou do setor.")
finalizar_figura("01c_mercado_ou_empresa.png", fig)

**O que os outliers mostram**

- **Mercado ou empresa:** dos 460 retornos de ±15% ou mais, 186 (40%) foram puxados pelo mercado e 273 (59%) vieram da empresa ou do setor; 1 é o erro corrigido (UGPA3).
- **Verificação:** dos 30 maiores eventos de empresa, 26 têm notícia, 1 acompanha o setor, 2 foram confirmados na cotação oficial (SNDK em 18/02/2025, dias depois da separação da Western Digital, e CEAB3 em 20/01/2023) e 1 é **erro de dados: UGPA3**, com +119% em 28/06/2021 no Yahoo e +1,05% na B3, um erro de escala que deixava toda a série anterior 2,2 vezes abaixo do nível correto; o retorno foi trocado pelo oficial. Na amostra aleatória de 20, 8 têm notícia, 5 acompanham o setor e os 7 restantes, todos do crash da Covid, foram confirmados na cotação oficial. **Dos 50 eventos verificados, 49 são reais e 1 é erro.**
- **O que podemos afirmar:** nenhum dos 20 sorteados é erro, então a taxa de erro entre os 244 eventos de empresa fora dos 30 maiores fica abaixo de 14%, com 95% de confiança. A seção 13.10 mede quanto isso importaria para o modelo.


## 6.3 O universo de hoje reproduz o índice de cada ano?

Este é um teste de consistência dos dados e, ao mesmo tempo, uma **medida do viés de sobrevivência**. Para cada ano, encontramos a combinação de pesos **não negativos e com soma 1** das ações do universo de 2026 (com 100% dos dados no ano) que melhor reproduz o retorno diário do índice, com pesos que acompanham os preços (compra-e-mantém). Os pesos são livres, sem limite de quantidade de ações.

Medimos duas coisas:
- **O tamanho (oscilação residual):** o desvio-padrão anualizado do que o universo *atual* deixa de explicar no índice *daquele ano*. Se ele cai com o tempo, as ações que saíram do índice faziam diferença no passado. **R² alto** indica que preços e benchmark são coerentes entre si.
- **A direção (TD fora da amostra):** os pesos estimados num ano são mantidos no ano seguinte. Um universo de sobreviventes tende a ficar **acima** do índice. Dentro do próprio ano a TD não serve para isso: o ajuste copia o nível do índice e a TD sai perto de zero por construção.

O primeiro é um ajuste dentro da amostra; serve para comparar anos e índices, não como estimativa do erro de uma carteira real.

In [ ]:
def ajuste_universo(R, y, penalidade_soma=100.0):
    # Mínimos quadrados com w >= 0 e soma(w) = 1 (restrição imposta por penalização).
    A = np.vstack([R.to_numpy(), penalidade_soma * np.ones(R.shape[1])])
    b = np.r_[y.to_numpy(), penalidade_soma]
    w, _ = nnls(A, b)
    residuo = y.to_numpy() - R.to_numpy() @ w
    return w, residuo


def ajuste_compra_e_mantem(R, y, penalidade_soma=100.0):
    # Pesos iniciais w0 >= 0 (soma 1) que reproduzem o NÍVEL do índice com compra-e-mantém:
    # I_t / I_0 ≈ Σ w0_i · P_i,t / P_i,0. Os pesos se deslocam com os preços, como num índice
    # ponderado por valor de mercado sem mudanças de composição.
    G = (1 + R).cumprod()
    nivel_idx = (1 + y).cumprod()
    A = np.vstack([G.to_numpy(), penalidade_soma * np.ones(G.shape[1])])
    w0, _ = nnls(A, np.r_[nivel_idx.to_numpy(), penalidade_soma])
    nivel = pd.Series(G.to_numpy() @ w0, index=y.index)
    r_cart = nivel.pct_change()
    r_cart.iloc[0] = nivel.iloc[0] - 1
    return (y - r_cart).to_numpy()


linhas = []
for nome, b in bases.items():
    for ano in range(DATA_INICIAL.year + 1, DATA_FINAL.year + 1):
        Ra = b["R"].loc[str(ano)]
        ya = b["y"].loc[str(ano)].dropna()
        Ra = Ra.loc[ya.index]
        Ra = Ra.loc[:, Ra.notna().mean() >= 1.0].fillna(0.0)
        residuo = ajuste_compra_e_mantem(Ra, ya)
        linhas.append({
            "indice": nome, "ano": ano, "ações disponíveis": Ra.shape[1],
            "R²": 1 - residuo.var() / ya.var(),
            "TE residual (% a.a.)": 100 * residuo.std() * np.sqrt(ANUAL),
        })
consistencia_universo = pd.DataFrame(linhas)
display(consistencia_universo.pivot(index="ano", columns="indice",
        values=["ações disponíveis", "R²", "TE residual (% a.a.)"]).round(3))

# Mudanças de composição conhecidas no período (verificadas em fonte oficial)
MUDANCAS_COMPOSICAO = pd.DataFrame([
    {"indice": "S&P100", "data efetiva": "2026-09-21", "movimento": "entra", "tickers": "DELL, PANW, ANET, SNDK"},
    {"indice": "S&P100", "data efetiva": "2026-09-21", "movimento": "sai", "tickers": "NKE, HONA, SPG, CL"},
]).assign(fonte="S&P Dow Jones Indices, comunicado de 04/09/2026 (rebalanceamento trimestral)")
print("Mudanças de composição verificadas (o universo congelado em 30/09/2026 já reflete esta troca):")
display(MUDANCAS_COMPOSICAO)
novatas = [t.strip() for t in MUDANCAS_COMPOSICAO.query("movimento == 'entra'")["tickers"].str.cat(sep=",").split(",")]
print("Novatas presentes no universo:", [t for t in novatas if t in TICKERS_SP100])

# Ilustração do viés de sobrevivência: carteira igualmente ponderada das ações atuais × índice
ilustracao_sobrevivencia = pd.DataFrame([
    {"indice": nome,
     "índice: retorno acumulado (%)": 100 * ((1 + b["y"].dropna()).prod() - 1),
     "ações atuais, pesos iguais: retorno acumulado (%)": 100 * ((1 + b["R"].mean(axis=1).dropna()).prod() - 1)}
    for nome, b in bases.items()
])
display(ilustracao_sobrevivencia.round(1))

fig, ax = plt.subplots(figsize=(9.5, 6.0))
anos = sorted(consistencia_universo["ano"].unique())
topo = consistencia_universo["TE residual (% a.a.)"].max() * 1.2
finais = {}
for nome in ["IBOV", "S&P100"]:
    g = consistencia_universo[consistencia_universo["indice"] == nome]
    ax.plot(g["ano"], g["TE residual (% a.a.)"], color=CORES[nome], marker="o", ms=7, mec=FUNDO, mew=2, lw=2.2, label=NOMES[nome], zorder=3)
    finais[nome] = g["TE residual (% a.a.)"].iloc[-1]
perto = abs(finais["IBOV"] - finais["S&P100"]) < 0.06 * topo
for nome, v in finais.items():
    dy = 0 if not perto else (8 if v >= max(finais.values()) else -8)
    ax.annotate(pct(v, 1), (anos[-1], v), xytext=(10, dy), textcoords="offset points", va="center", fontsize=10.5,
                color=TINTA, fontweight="semibold")
ax.set_xticks(anos)
ax.set_xticklabels([str(a) if a < DATA_FINAL.year else f"{a}*" for a in anos])
ax.set_xlim(anos[0] - 0.5, anos[-1] + 0.9)
ax.set_ylim(0, topo)
eixo_pct(ax, casas=1)
ax.set_ylabel("Erro residual (% ao ano)")
legenda(ax, loc="upper right", ncol=1)
g_sp = consistencia_universo[consistencia_universo["indice"] == "S&P100"].set_index("ano")
if DATA_FINAL.year in g_sp.index:
    nota(ax, DATA_FINAL.year, g_sp.loc[DATA_FINAL.year, "TE residual (% a.a.)"],
         "S&P 100 em 2026: quatro empresas\nentraram e quatro saíram em 21/09", -10, 150, ha="right")
titulos(fig, "O universo de hoje explica cada vez melhor o Ibovespa do passado: sinal de viés de sobrevivência",
        "Erro da melhor combinação das ações atuais, com pesos que acompanham os preços, para reproduzir o retorno diário do índice em\n"
        "cada ano (% ao ano). Quanto mais alto, menos o universo de 2026 explica o índice daquele ano. *2026 até setembro.")
finalizar_figura("12_consistencia_universo_indice.png", fig)

# Direção (TD): pesos estimados com as ações de hoje num ano e mantidos (compra-e-mantém) no ano seguinte.
# Dentro do próprio ano, o ajuste copia o nível do índice e a TD sai perto de zero por construção;
# fora da amostra, um universo de sobreviventes tende a ficar acima do índice.
def vies_ano_seguinte(R, y, ano):
    y0 = y.loc[str(ano - 1)].dropna()
    R0 = R.loc[y0.index]
    R0 = R0.loc[:, R0.notna().mean() >= 1.0].fillna(0.0)
    w, _ = ajuste_universo(R0, y0)
    pesos = pd.Series(w, index=R0.columns)
    pesos = pesos[pesos > 1e-6] / pesos[pesos > 1e-6].sum()
    y1 = y.loc[str(ano)].dropna()
    R1 = R.reindex(index=y1.index, columns=pesos.index).fillna(0.0)
    nivel = (1 + R1).cumprod() @ pesos
    r = nivel.pct_change()
    r.iloc[0] = nivel.iloc[0] - 1
    d = r - y1
    return {"TD fora da amostra (p.p. a.a.)": 100 * d.mean() * ANUAL,
            "retorno da carteira no ano seguinte (%)": 100 * (nivel.iloc[-1] - 1),
            "retorno do índice no ano (%)": 100 * ((1 + y1).prod() - 1)}


direcao = pd.DataFrame([{"indice": nome, "ano": ano, **vies_ano_seguinte(b["R"], b["y"], ano)}
                        for nome, b in bases.items() for ano in range(DATA_INICIAL.year + 2, DATA_FINAL.year + 1)])
consistencia_universo = consistencia_universo.merge(direcao, on=["indice", "ano"], how="left")
display(direcao.pivot(index="ano", columns="indice", values="TD fora da amostra (p.p. a.a.)").round(2))
media_vies = direcao.groupby("indice")["TD fora da amostra (p.p. a.a.)"].mean()

fig, ax = plt.subplots(figsize=(12, 5.6))
anos_d = sorted(direcao["ano"].unique())
largura = 0.36
for i, nome in enumerate(["S&P100", "IBOV"]):
    v = direcao[direcao["indice"] == nome].set_index("ano").reindex(anos_d)["TD fora da amostra (p.p. a.a.)"]
    xs = np.arange(len(anos_d)) + (i - 0.5) * (largura + 0.04)
    ax.bar(xs, v.to_numpy(), width=largura, color=CORES[nome], label=f"{NOMES[nome]} (média {num(media_vies[nome], 1, sinal=True)} p.p.)", zorder=2)
    for x, val in zip(xs, v.to_numpy()):
        ax.text(x, val + (0.25 if val >= 0 else -0.25), "0,0" if abs(val) < 0.05 else num(val, 1, sinal=True),
                ha="center", va="bottom" if val >= 0 else "top",
                fontsize=9.5, color=TINTA_2)
ax.axhline(0, color=TINTA_2, lw=0.8, zorder=3)
_v = direcao["TD fora da amostra (p.p. a.a.)"]
ax.set_ylim(min(_v.min(), 0) - 1.4, _v.max() + 1.2)
ax.set_xticks(np.arange(len(anos_d)))
ax.set_xticklabels([f"{a}*" if a == DATA_FINAL.year else str(a) for a in anos_d])
ax.grid(axis="y", visible=False)
ax.yaxis.set_major_locator(NullLocator())
legenda(ax, loc="upper right", ncol=1)
titulos(fig, f"Fora da amostra, as ações de hoje ficaram acima do índice: em média {num(media_vies['IBOV'], 1, sinal=True)} p.p. por ano no Ibovespa "
             f"e {num(media_vies['S&P100'], 1, sinal=True)} no S&P 100",
        "Diferença média entre a carteira e o índice (p.p. por ano). A carteira usa todas as ações de hoje, com os pesos que melhor reproduziram o "
        "índice no ano anterior, mantidos no ano seguinte. Positivo: o universo de sobreviventes rendeu mais que o índice. Um ano isolado é ruidoso "
        "(cerca de ±2 p.p.). *2026 até setembro.")
finalizar_figura("12b_vies_universo_fora_amostra.png", fig)

**O que a consistência universo × índice mostra**

- **Os dados são coerentes:** o universo reproduz o movimento diário dos dois índices com R² acima de 0,98 em todos os anos.
- **Tamanho:** no IBOV, a oscilação residual cai de 3,2% a.a. em 2020 para 0,6% em 2026. Quanto mais para trás, mais fazem falta as empresas que saíram do índice (em 2020, só 65 das 76 ações atuais existiam). No S&P, fica entre 0,2% e 0,5% até 2025; o salto de 2026 vem da troca de composição de 21/09/2026.
- **Direção:** com os pesos de um ano mantidos no ano seguinte, o universo atual ficou acima do IBOV em 5 de 6 anos (média de +1,8 p.p. por ano) e acima do S&P em 3 de 6 (média de +1,0 p.p.). A média positiva é o sinal esperado do viés de sobrevivência. No S&P, ela vem quase toda de 2023 (+5,8 p.p.), um ano em que poucas gigantes de tecnologia dispararam e pequenas diferenças de peso em relação ao índice fizeram muita diferença; nos outros anos, o desvio fica entre −1,6 e +1,0 p.p.
- **Para o modelo:** as carteiras devem herdar uma TD positiva, maior no IBOV. A seção 13.5 separa esse efeito do efeito de usar poucas ações.

## 7. EDA: comportamento dos retornos e estrutura dos índices

### 7.0 Metadados das empresas

O setor e a capitalização de mercado vêm do Yahoo (`get_info`). Não entram no modelo; servem para interpretar o EDA (setores) e para aproximar a concentração dos índices (seção 7.14). Falhas individuais viram `Não informado` sem interromper o notebook. O resultado fica em cache para não repetir as consultas.

In [ ]:
COLETAR_METADADOS_YAHOO = True
CACHE_METADADOS = PASTA_SAIDA / "metadados_yahoo_cache.csv"
metadados = pd.DataFrame(columns=["indice", "ticker", "setor_original", "setor", "valor_mercado", "moeda"])

if COLETAR_METADADOS_YAHOO:
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def obter_metadados(ticker):
        try:
            info = yf.Ticker(ticker).get_info()
            return {"ticker": ticker, "setor_original": info.get("sector") or "Não informado",
                    "valor_mercado": info.get("marketCap"), "moeda": info.get("currency")}
        except Exception:
            return {"ticker": ticker, "setor_original": "Não informado", "valor_mercado": np.nan, "moeda": None}

    if CACHE_METADADOS.exists():
        metadados = pd.read_csv(CACHE_METADADOS)
        print(f"Metadados carregados do cache: {len(metadados)} tickers.")
    else:
        linhas = []
        with ThreadPoolExecutor(max_workers=5) as executor:
            futuros = {executor.submit(obter_metadados, t): indice
                       for indice, tickers in [("S&P100", TICKERS_SP100), ("IBOV", TICKERS_IBOV)] for t in tickers}
            for numero, futuro in enumerate(as_completed(futuros), 1):
                linhas.append({"indice": futuros[futuro], **futuro.result()})
                if numero % 40 == 0 or numero == len(futuros):
                    print(f"Metadados: {numero}/{len(futuros)}", flush=True)
        metadados = pd.DataFrame(linhas)
        metadados.to_csv(CACHE_METADADOS, index=False)
    metadados["setor"] = metadados["setor_original"].map(SETORES_PT).fillna(metadados["setor_original"])
else:
    print("Metadados não coletados. Defina COLETAR_METADADOS_YAHOO=True para ativar setores e concentração.")

setores = metadados[["indice", "ticker", "setor"]].copy()
display(metadados.groupby("indice").agg(
    tickers=("ticker", "nunique"),
    com_setor=("setor", lambda s: int((s != "Não informado").sum())),
    com_valor_de_mercado=("valor_mercado", lambda s: int(s.notna().sum())),
))

### 7.1 Estatísticas descritivas e risco × retorno

As estatísticas usam todas as ações, inclusive as de série curta (7 têm menos de 80% do histórico): para elas, retorno e volatilidade anualizados são menos confiáveis. A SNDK, com 19 meses de dados, fica fora da área do gráfico de risco × retorno.

In [ ]:
estatisticas = []
correlacoes = []
retornos_eda = {}
for nome, b in bases.items():
    cobertura = b["R"].notna().mean()
    cols = cobertura.index   # todas as ações (sem corte por cobertura)
    R = b["R"][cols]
    y = b["y"]
    retornos_eda[nome] = R
    tab = pd.DataFrame({
        "retorno médio anualizado (%)": 100 * R.mean() * ANUAL,
        "volatilidade anualizada (%)": 100 * R.std() * np.sqrt(ANUAL),
        "Sharpe aproximado": R.mean().div(R.std()).mul(np.sqrt(ANUAL)),
        "mínimo diário (%)": 100 * R.min(),
        "máximo diário (%)": 100 * R.max(),
        "assimetria": R.skew(),
        "curtose": R.kurtosis(),
        "observações": R.count(),
    }).rename_axis("ticker").reset_index()
    tab.insert(0, "indice", nome)
    estatisticas.append(tab)

    corr = (R.corrwith(y).sort_values(ascending=False)
            .rename_axis("ticker")
            .reset_index(name="correlação com índice"))
    corr.insert(0, "indice", nome)
    correlacoes.append(corr)

estatisticas = pd.concat(estatisticas, ignore_index=True).merge(setores, on=["indice", "ticker"], how="left")
correlacoes = pd.concat(correlacoes, ignore_index=True).merge(setores, on=["indice", "ticker"], how="left")
estatisticas = estatisticas.merge(correlacoes[["indice", "ticker", "correlação com índice"]], on=["indice", "ticker"], how="left")

for nome in bases:
    e = estatisticas[estatisticas["indice"] == nome].set_index("ticker")
    rankings = pd.DataFrame({
        "maior retorno a.a. (%)": [f"{t.replace('.SA', '')} {v:.0f}" for t, v in e["retorno médio anualizado (%)"].nlargest(10).items()],
        "menor volatilidade (%)": [f"{t.replace('.SA', '')} {v:.0f}" for t, v in e["volatilidade anualizada (%)"].nsmallest(10).items()],
        "maior Sharpe": [f"{t.replace('.SA', '')} {v:.2f}" for t, v in e["Sharpe aproximado"].nlargest(10).items()],
        "maior correlação c/ índice": [f"{t.replace('.SA', '')} {v:.2f}" for t, v in e["correlação com índice"].nlargest(10).items()],
    }, index=range(1, 11))
    print(f"\n{nome}: top 10 ({len(e)} ações)")
    display(rankings)

fig, axes = plt.subplots(1, 2, figsize=(12, 6.2))
contagens = {}
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    e = estatisticas[estatisticas["indice"] == nome]
    y = bases[nome]["y"].dropna()
    vol_idx, ret_idx = 100 * y.std() * np.sqrt(ANUAL), 100 * y.mean() * ANUAL
    contagens[nome] = (int((e["volatilidade anualizada (%)"] > vol_idx).sum()), len(e))
    ax.axvspan(0, vol_idx, color=FAIXA, zorder=0)
    ax.scatter(e["volatilidade anualizada (%)"], e["retorno médio anualizado (%)"], s=30, color=CONTEXTO_ESC,
               edgecolor=FUNDO, linewidth=0.8, zorder=3)
    ax.scatter([vol_idx], [ret_idx], s=110, color=CORES[nome], edgecolor=FUNDO, linewidth=2, zorder=5)
    rotulo(ax, vol_idx, ret_idx, f"{NOMES[nome]}\n(o índice)", cor=None, dx=-10, ha="right", marcador=False, tam=10.5)
    rotulados = set()
    for _, l in e.nlargest(6, "Sharpe aproximado").iterrows():
        base = l["ticker"].replace(".SA", "")[:4]
        if base in rotulados or len(rotulados) >= 4:
            continue
        rotulados.add(base)
        irmaos = e[e["ticker"].str.replace(".SA", "", regex=False).str[:4] == base]
        texto = l["ticker"].replace(".SA", "") if len(irmaos) == 1 else base + "3/4"
        ax.annotate(texto, (l["volatilidade anualizada (%)"], l["retorno médio anualizado (%)"]), xytext=(6, 2),
                    textcoords="offset points", fontsize=9.5, color=TINTA_2)
    ax.axhline(0, color=BASE, lw=0.8, zorder=1)
    # séries curtas (ex.: SNDK, 19 meses) esticariam os eixos: fica fora da área do gráfico, com nota
    _v, _r = e["volatilidade anualizada (%)"], e["retorno médio anualizado (%)"]
    _curta = e["observações"] < 0.8 * len(y)
    _fora = _curta & ((_v > _v[~_curta].max() * 1.3) | (_r > _r[~_curta].max() * 1.3))
    _xmax, _ymax = _v[~_fora].max() * 1.08, _r[~_fora].max() * 1.12
    ax.set_xlim(0, _xmax)
    ax.set_ylim(min(_r[~_fora].min() * 1.1, -5), _ymax)
    for _, _l in e[_fora].iterrows():
        ax.text(_xmax, ax.get_ylim()[0] + (_ymax - ax.get_ylim()[0]) * 0.05,
                f"{_l['ticker']}: {_l['volatilidade anualizada (%)']:.0f}% de volatilidade e\n{_l['retorno médio anualizado (%)']:.0f}% de retorno ao ano (fora do gráfico;\nsérie de só {_l['observações'] / 21:.0f} meses)",
                ha="right", va="bottom", fontsize=9, color=TINTA_2)
    eixo_pct(ax, "x"); eixo_pct(ax, "y")
    ax.set_xlabel("Volatilidade (% ao ano)")
    ax.set_title(NOMES[nome])
    ax.text(vol_idx / 2, ax.get_ylim()[1] * 0.97, "menos volátil\nque o índice", ha="center", va="top", fontsize=9.5, color=APAGADO)
axes[0].set_ylabel("Retorno médio (% ao ano)")
(n1, t1), (n2, t2) = contagens["S&P100"], contagens["IBOV"]
titulos(fig, f"Quase todas as ações oscilam mais que o índice: {n1} de {t1} no S&P 100 e {n2} de {t2} no Ibovespa",
        "Retorno médio anual × volatilidade anual de cada ação (pontos cinza). Juntas, as ações se compensam, e o índice\n"
        "fica menos arriscado do que quase todas elas. Rotuladas: as de melhor retorno por unidade de risco.")
finalizar_figura("13_risco_retorno.png", fig)

**O que as estatísticas e o risco × retorno mostram**

- **Quase todas as ações são mais voláteis que o índice:** 98 de 101 no S&P (as exceções são JNJ, KO e BRK-B) e 73 de 76 no IBOV (TAEE11, ISAE4 e EGIE3). A volatilidade mediana das ações é de 32,0% a.a. (S&P) e 38,0% (IBOV), contra 20,4% e 23,5% dos índices. A diversificação elimina boa parte do risco individual, e a carteira replicante precisa reproduzir essa diversificação.
- **As maiores correlações com o índice não são as ações de maior retorno.** Os melhores Sharpe (rotulados no gráfico) são de papéis de alto crescimento ou de commodities (SNDK, GEV, NVDA, LLY no S&P; PRIO3, CPLE3, PETR3/PETR4, TAEE11 no IBOV). SNDK e GEV têm menos de 3 anos de série e distorcem o ranking: a SNDK, com retorno anualizado de 295%, fica fora da área do gráfico. As ações mais parecidas com o índice (seção 7.5) ficam perto do centro da nuvem, não no topo.
- **No S&P há relação clara entre risco e retorno:** as ações mais voláteis renderam mais no período, puxadas por tecnologia. **No IBOV essa relação praticamente não existe:** várias ações de alta volatilidade tiveram retorno negativo.
- **O índice tem Sharpe melhor que a ação mediana:** 0,85 no S&P e 0,47 no IBOV (sem taxa livre de risco).

### 7.2 Performance acumulada e distribuição dos retornos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6.2))
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    b = bases[nome]
    y = b["y"].dropna()
    top = correlacoes[correlacoes["indice"] == nome].head(12)["ticker"].tolist()
    norm = b["precos"][top].apply(lambda s: 100 * s / s.dropna().iloc[0] if s.notna().any() else s)
    for col in norm:
        ax.plot(norm.index, norm[col], color=CONTEXTO, lw=1.1, zorder=2)
    idx_norm = 100 * (1 + y).cumprod()
    ax.plot(idx_norm.index, idx_norm, color=CORES[nome], lw=2.6, zorder=4)
    ax.set_yscale("log")
    marcas = [v for v in [25, 50, 100, 200, 500, 1000, 2000, 5000] if norm.min().min() * 0.8 <= v <= norm.max().max() * 1.3]
    ax.yaxis.set_major_locator(FixedLocator(marcas)); ax.yaxis.set_minor_locator(NullLocator())
    eixo_num(ax, casas=0)
    rotulo(ax, idx_norm.index[-1], idx_norm.iloc[-1], f"Índice: {num(idx_norm.iloc[-1], 0)}", cor=CORES[nome])
    melhor = norm.iloc[-1].idxmax()
    rotulo(ax, norm.index[-1], norm[melhor].iloc[-1], f"{melhor.replace('.SA', '')}: {num(norm[melhor].iloc[-1], 0)}", cor=CONTEXTO_ESC, tam=10)
    minimo = idx_norm.loc["2020-02-01":"2020-06-30"].idxmin()
    nota(ax, minimo, idx_norm.loc[minimo], "Covid", 15, -28, ha="left")
    ax.set_title(NOMES[nome])
    ax.set_xlim(idx_norm.index[0], idx_norm.index[-1] + pd.Timedelta(days=60))
titulos(fig, "As ações mais parecidas com o índice divergem muito dele no longo prazo",
        "Evolução de 100 investidos em out/2019 (escala logarítmica). Linha colorida: o índice; cinza: as 12 ações com\n"
        "maior correlação diária com ele. Andar junto no dia a dia não significa terminar no mesmo lugar.",
        direita=0.88, espaco_w=0.42)
finalizar_figura("02_performance.png", fig)

# Distribuição dos retornos diários x curva normal
fig, axes = plt.subplots(1, 2, figsize=(12, 5.8), sharey=False)
textos_sub = []
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    y = 100 * bases[nome]["y"].dropna()
    largura = 0.25
    bins = np.arange(np.floor(y.min()), np.ceil(y.max()) + largura, largura)
    ax.hist(y, bins=bins, color=CORES_CLARAS[nome], edgecolor=FUNDO, linewidth=0.6, zorder=2)
    grade = np.linspace(y.min(), y.max(), 400)
    normal = len(y) * largura * np.exp(-0.5 * ((grade - y.mean()) / y.std()) ** 2) / (y.std() * np.sqrt(2 * np.pi))
    ax.plot(grade, normal, color=TINTA, lw=1.4, ls=(0, (4, 3)), zorder=3)
    from math import erfc, sqrt
    lim = 5.0
    obs = int((y.abs() > lim).sum())
    z1, z2 = (lim - y.mean()) / y.std(), (-lim - y.mean()) / y.std()
    esperado = len(y) * (0.5 * erfc(z1 / sqrt(2)) + 0.5 * erfc(-z2 / sqrt(2)))
    textos_sub.append(f"{NOMES[nome]}: {obs} dias além de ±5% (a normal previa {num(esperado, 1)})")
    ax.set_xlim(-8, 8)
    eixo_pct(ax, "x")
    ax.set_xlabel("Retorno diário do índice")
    ax.set_title(f"{NOMES[nome]} (curtose {num(y.kurtosis(), 1)})")
    pico = normal.max()
    nota(ax, grade[np.argmin(np.abs(grade - 2.2 * y.std()))], normal[np.argmin(np.abs(grade - 2.2 * y.std()))],
         "curva normal\n(mesma média e desvio)", 35, 30)
axes[0].set_ylabel("Número de dias")
titulos(fig, "Dias extremos são muito mais comuns do que a curva normal prevê",
        "Distribuição dos retornos diários dos índices (barras) e a curva normal equivalente (tracejada). "
        + "; ".join(textos_sub) + ".")
finalizar_figura("02b_distribuicao_retornos.png", fig)

**O que a performance e a distribuição mostram**

- **Correlação diária alta não garante retorno acumulado parecido.** As 12 ações mais correlacionadas divergem muito do índice no acumulado: a NVDA multiplica o valor por cerca de 50 vezes em sete anos. Por isso o *tracking* precisa olhar para o erro de retorno, não só para a correlação.
- **Os dois índices tiveram trajetórias diferentes.** O S&P subiu de forma quase contínua, com quedas na Covid e em 2022. O IBOV oscilou em uma faixa de 2020 a 2024 e só saiu dela em 2025.
- **As caudas são muito mais gordas que as de uma normal:** curtose de 13,1 (S&P) e 20,7 (IBOV). Dias como −11,6% (S&P, 16/03/2020) e −14,8% (IBOV, 12/03/2020) seriam praticamente impossíveis sob normalidade.
- **Implicação para a modelagem (a testar):** poucos dias extremos pesam muito em um erro quadrático. Vale verificar o quanto as carteiras dependem dos dias extremos presentes na janela de treino.

### 7.3 Retornos mensais e volatilidade

In [ ]:
MESES_ABREV = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
fig, axes = plt.subplots(2, 1, figsize=(12, 7.2), sharex=True)
resumo_indices = []
piores_meses = {}
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    y = bases[nome]["y"].dropna()
    mensal = 100 * ((1 + y).resample("ME").prod() - 1)
    piores = mensal.nsmallest(3)
    piores_meses[nome] = piores
    cores = [ALERTA if d in piores.index else (CORES[nome] if v >= 0 else CORES_CLARAS[nome]) for d, v in mensal.items()]
    ax.bar(mensal.index, mensal.values, width=22, color=cores, zorder=2)
    ax.axhline(0, color=TINTA_2, lw=0.8, zorder=3)
    rotulados = []
    for d, v in piores.items():
        if any(abs((d - r).days) < 200 for r in rotulados):
            continue
        rotulados.append(d)
        ax.annotate(f"{MESES_ABREV[d.month - 1]}/{d.year}\n{pct(v, 1)}", (d, v), xytext=(14, -2), textcoords="offset points",
                    ha="left", va="center", fontsize=9.5, color=TINTA)
    ax.set_ylim(mensal.min() * 1.3, max(mensal.max() * 1.25, 5))
    eixo_pct(ax)
    ax.set_title(NOMES[nome])
    resumo_indices.append({
        "indice": nome,
        "retorno acumulado (%)": 100 * ((1 + y).prod() - 1),
        "retorno médio anualizado (%)": 100 * y.mean() * ANUAL,
        "volatilidade anualizada (%)": 100 * y.std() * np.sqrt(ANUAL),
        "Sharpe aproximado": y.mean() / y.std() * np.sqrt(ANUAL),
        "drawdown máximo (%)": 100 * drawdown_maximo(y),
        "curtose": y.kurtosis(),
        "pior dia (%)": 100 * y.min(),
        "melhor dia (%)": 100 * y.max(),
    })
p_sp, p_ib = piores_meses["S&P100"].iloc[0], piores_meses["IBOV"].iloc[0]
titulos(fig, f"Março de 2020 foi o pior mês do período nos dois índices: {pct(p_sp, 1)} no S&P 100 e {pct(p_ib, 1)} no Ibovespa",
        "Retorno mensal de cada índice. Em vermelho, os três piores meses; tons claros, os demais meses de queda.",
        espaco_h=0.35)
finalizar_figura("03_retornos_mensais.png", fig)
resumo_indices = pd.DataFrame(resumo_indices)
print("Resumo comparativo dos índices (Sharpe sem taxa livre de risco)")
display(resumo_indices.set_index("indice").T.round(2))

# Volatilidade móvel: um só eixo, as duas séries
fig, ax = plt.subplots(figsize=(12, 5.8))
picos = {}
for nome in ["S&P100", "IBOV"]:
    vol = 100 * bases[nome]["y"].rolling(21).std() * np.sqrt(ANUAL)
    ax.plot(vol.index, vol, color=CORES[nome], lw=1.6, label=NOMES[nome], zorder=3 if nome == "S&P100" else 2)
    picos[nome] = (vol.idxmax(), vol.max())
    rotulo(ax, vol.index[-1], vol.iloc[-1], f"{NOMES[nome]}: {pct(vol.iloc[-1])}", cor=CORES[nome], tam=10.5)
d_ib, v_ib = picos["IBOV"]
nota(ax, d_ib, v_ib, f"Pico na Covid: {pct(v_ib)} ao ano no Ibovespa\ne {pct(picos['S&P100'][1])} no S&P 100 ({d_ib.strftime('%d/%m/%Y')})", 30, -5)
eventos_vol = [("2022-03-16", "1ª alta de juros\nnos EUA"), ("2025-04-02", "Tarifas\ndos EUA")]
for data, texto in eventos_vol:
    d = pd.Timestamp(data)
    ax.axvline(d, color=BASE, lw=0.8, zorder=1)
    ax.text(d + pd.Timedelta(days=12), ax.get_ylim()[1] * 0.93 if False else v_ib * 0.62, texto, fontsize=9.5, color=TINTA_2, va="top")
ax.set_ylim(0, v_ib * 1.08)
ax.set_xlim(None, bases["S&P100"]["y"].index[-1] + pd.Timedelta(days=150))
eixo_pct(ax)
legenda(ax, loc="upper right", ncol=2)
titulos(fig, "A volatilidade explodiu na Covid e voltou ao normal em poucos meses",
        "Volatilidade anualizada dos índices em janelas móveis de 21 pregões (cerca de um mês). Fora das crises, ela fica perto\n"
        f"de {pct(100 * bases['S&P100']['y'].rolling(21).std().median() * np.sqrt(ANUAL))} no S&P 100 e "
        f"{pct(100 * bases['IBOV']['y'].rolling(21).std().median() * np.sqrt(ANUAL))} no Ibovespa (medianas).",
        direita=0.88)
finalizar_figura("03b_volatilidade.png", fig)

**O que os retornos mensais e a volatilidade mostram**

- **A Covid é o evento dominante:** pior mês em março/2020 (−10,3% no S&P, −29,9% no IBOV); pico da volatilidade de 21 pregões em 06/04/2020 (95% e 127% a.a.); queda máxima do topo ao fundo de −31,5% e −46,8%, ambas em 23/03/2020.
- **A recuperação foi rápida:** abril/2020 foi o melhor mês do S&P (+12,7%), e novembro/2020 (anúncio das vacinas) o do IBOV (+15,9%).
- **Os demais picos coincidem com eventos conhecidos.** No S&P: o aperto do Fed em 2022 e as tarifas de abril/2025, o maior choque depois da Covid. No IBOV: a interferência na Petrobras (fev/2021) e a eleição de 2022.
- **Fora das crises, a volatilidade fica perto da mediana** (14,5% no S&P, 17,2% no IBOV). Ela vem em ondas: períodos agitados duram semanas. Uma janela de treino que contenha uma crise descreve um regime diferente do que vem depois.
- **Resumo comparativo:** o S&P tem retorno maior, volatilidade menor, Sharpe quase o dobro (0,85 contra 0,47) e queda máxima menor (−31,5% contra −46,8%).

### 7.4 Retornos anuais e comparação entre índices

In [ ]:
retornos_anuais_indices = pd.concat({
    nome: (1 + b["y"].dropna()).resample("YE").prod() - 1
    for nome, b in bases.items()
}, axis=1)
retornos_anuais_indices.index = retornos_anuais_indices.index.year
retornos_anuais_indices.index.name = "ano"
print("Retornos anuais dos benchmarks (%); 2019 e 2026 são anos parciais")
display((100 * retornos_anuais_indices).round(1).T)

indices_alinhados = pd.concat(
    [bases["S&P100"]["y"].rename("S&P100"), bases["IBOV"]["y"].rename("IBOV")],
    axis=1, join="inner",
).dropna()
# Nível de cada índice no SEU calendário; o alinhamento é feito nos níveis (não nos retornos),
# para não perder o retorno dos dias em que só um dos mercados abriu.
acumulado_comparativo = pd.concat(
    {nome: 100 * (1 + bases[nome]["y"].fillna(0.0)).cumprod() for nome in ["S&P100", "IBOV"]}, axis=1
).ffill()

fig, ax = plt.subplots(figsize=(12, 5.8))
for nome in ["S&P100", "IBOV"]:
    s = acumulado_comparativo[nome].dropna()
    ax.plot(s.index, s, color=CORES[nome], lw=2.3, label=NOMES[nome])
    rotulo(ax, s.index[-1], s.iloc[-1], f"{NOMES[nome]}: {pct(s.iloc[-1] - 100, 0, sinal=True)}", cor=CORES[nome])
ax.axhline(100, color=TINTA_2, lw=0.8)
eixo_num(ax, casas=0)
ax.set_xlim(None, acumulado_comparativo.index[-1] + pd.Timedelta(days=200))
legenda(ax)
r = (acumulado_comparativo.ffill().iloc[-1] - 100)
titulos(fig, f"Em sete anos, o S&P 100 rendeu {pct(r['S&P100'], 0, sinal=True)} em dólar; o Ibovespa, {pct(r['IBOV'], 0, sinal=True)} em real",
        f"Evolução de 100 investidos em out/2019, cada índice no seu calendário de pregões. S&P 100 sem dividendos; Ibovespa com\n"
        f"dividendos. A correlação diária entre os dois, nos dias em que ambos abriram, é de {num(indices_alinhados.corr().iloc[0, 1], 2)}.",
        direita=0.86)
finalizar_figura("04_comparacao_benchmarks.png", fig)

# Retornos anuais lado a lado
anos = retornos_anuais_indices.index.tolist()
fig, ax = plt.subplots(figsize=(12, 5.8))
largura = 0.36
for i, nome in enumerate(["S&P100", "IBOV"]):
    v = 100 * retornos_anuais_indices[nome]
    xs = np.arange(len(anos)) + (i - 0.5) * (largura + 0.04)
    ax.bar(xs, v.values, width=largura, color=CORES[nome], label=NOMES[nome], zorder=2)
    for x, val in zip(xs, v.values):
        ax.text(x, val + (1.2 if val >= 0 else -1.2), pct(val, 0, sinal=True), ha="center", va="bottom" if val >= 0 else "top",
                fontsize=9.5, color=TINTA_2)
ax.axhline(0, color=TINTA_2, lw=0.8, zorder=3)
ax.set_xticks(np.arange(len(anos)))
ax.set_xticklabels([f"{a}*" if a in (DATA_INICIAL.year, DATA_FINAL.year) else str(a) for a in anos])
ax.grid(axis="y", visible=False)
ax.yaxis.set_major_locator(NullLocator())
legenda(ax)
opostos = int((np.sign(retornos_anuais_indices["S&P100"]) != np.sign(retornos_anuais_indices["IBOV"])).sum())
titulos(fig, f"Os dois mercados andaram em direções opostas em {opostos} dos {len(anos)} anos",
        "Retorno anual de cada índice, em moeda local. *2019: só out–dez; 2026: até setembro.")
finalizar_figura("04b_retornos_anuais.png", fig)
print("Comparação descritiva: S&P 100 em USD e retorno de preços; IBOV em BRL e retorno total.")

**O que os retornos anuais e a comparação mostram**

> 2019 (só out–dez) e 2026 (até 30/09) são **anos parciais**.

- **Os dois mercados tiveram ciclos opostos em vários anos:** 2021 (S&P +27,6%, IBOV −12,1%), 2022 (S&P −22,1%, IBOV +5,0%) e 2024 (S&P +29,2%, IBOV −10,4%). Em 2025 a ordem se inverte: IBOV +34,0%, S&P +18,8%.
- **A correlação diária entre os índices é de 0,56:** eles compartilham um fator global, mas cada um tem dinâmica própria relevante.
- **No acumulado, S&P +190,7% e IBOV +77,9%.** A comparação é só descritiva: o S&P está em USD e sem dividendos, e o IBOV em BRL com proventos.

### 7.5 Correlação de cada ação com o índice

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12.5, 8.6))
resumo_barras = {}
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    c = correlacoes[correlacoes["indice"] == nome].sort_values("correlação com índice", ascending=False)
    valores = c["correlação com índice"].to_numpy()
    cores = [CORES[nome] if v > 0.7 else CONTEXTO for v in valores]
    ax.bar(range(len(c)), valores, width=0.72, color=cores, zorder=2)
    for ref in (0.5, 0.7):
        ax.axhline(ref, color=TINTA_2 if ref == 0.7 else BASE, lw=0.9, zorder=3)
        ax.text(len(c) + 0.3, ref, num(ref, 1), va="center", fontsize=10, color=TINTA_2)
    ax.set_xticks(range(len(c)))
    ax.set_xticklabels([t.replace(".SA", "") for t in c["ticker"]], rotation=90, fontsize=7.2)
    ax.set_xlim(-0.8, len(c) + 1.5)
    ax.set_ylim(0, 0.9)
    ax.yaxis.set_major_locator(FixedLocator([0, 0.2, 0.4, 0.6, 0.8]))
    eixo_num(ax, casas=1)
    acima = (valores > 0.7).mean()
    resumo_barras[nome] = (acima, valores.max(), c["ticker"].iloc[0].replace(".SA", ""))
    ax.set_title(f"{NOMES[nome]}: {pct(100 * acima)} das ações acima de 0,7")
    ax.tick_params(axis="x", colors=BASE, labelcolor=TINTA_2)
mx = max(resumo_barras.values(), key=lambda t: t[1])
titulos(fig, f"Nenhuma ação sozinha replica o índice: a mais parecida ({mx[2]}) tem correlação de {num(mx[1], 2)}",
        "Correlação dos retornos diários de cada ação com o índice, out/2019 a set/2026. Em cor, as ações acima de 0,7.\n"
        "Uma correlação de 0,8 significa que a ação explica cerca de 64% do movimento diário do índice.",
        base_pol=1.1, espaco_h=0.75, direita=0.96)
finalizar_figura("14_correlacao_com_indice_ranqueada.png", fig)

**O que o ranking de correlação mostra**

- **Nenhuma ação sozinha replica o índice:** a maior correlação é 0,79 (MSFT) no S&P e 0,81 (ITSA4) no IBOV. Uma única ação explica no máximo cerca de 65% da variância do benchmark.
- **No IBOV, as ações andam mais junto com o índice:** correlação mediana de 0,59 contra 0,55 no S&P; 18% das ações acima de 0,7 contra 9%; 80% acima de 0,5 contra 66%.
- **Quem está no topo e na base (setores da seção 7.7):**
  - no S&P, o topo é de tecnologia (MSFT, AAPL, GOOG/GOOGL, NVDA, AVGO) e financeiro (BLK, MS, MA, GS, V); a base é de saúde, consumo básico e telecom (MRK, GILD, BMY, MO, VZ);
  - no IBOV, o topo é de bancos e holdings (ITSA4, BBDC3, BBAS3, ITUB4, BPAC11) e de ações domésticas sensíveis a juros (MOTV3, B3SA3, ALOS3, CYRE3); a base é de exportadores e agronegócio (SUZB3, BEEF3, KLBN11, BRAV3, MBRF3). Para esses, a desvalorização do real tende a ajudar justamente quando a bolsa cai.

### 7.6 Correlação entre ações: redundâncias

In [ ]:
cmap_corr = LinearSegmentedColormap.from_list("rampa_azul", RAMPA_AZUL)
fig, axes = plt.subplots(1, 2, figsize=(13, 6.8))
pares_redundantes = []
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    top = correlacoes[correlacoes["indice"] == nome].head(20)["ticker"]
    C = retornos_eda[nome][top].corr()
    C.index = C.columns = [t.replace(".SA", "") for t in C.columns]
    mascara = np.triu(np.ones_like(C, dtype=bool))
    M = np.ma.masked_where(mascara, C.to_numpy())
    im = ax.imshow(M, cmap=cmap_corr, vmin=0.2, vmax=1.0)
    n = len(C)
    for i in range(n):
        for j in range(i):
            v = C.iloc[i, j]
            if v > 0.9:
                ax.text(j, i, num(v, 2), ha="center", va="center", fontsize=7.5, color=FUNDO, fontweight="bold")
                pares_redundantes.append({"indice": nome, "ticker 1": C.columns[i], "ticker 2": C.columns[j], "correlação": v})
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(C.columns, rotation=90, fontsize=8); ax.set_yticklabels(C.index, fontsize=8)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(NOMES[nome])
cbar = fig.colorbar(im, ax=axes, shrink=0.55, pad=0.02)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=9.5, colors=BASE, labelcolor=TINTA_2)
cbar.set_label("correlação", color=TINTA_2)
pares_redundantes = pd.DataFrame(pares_redundantes, columns=["indice", "ticker 1", "ticker 2", "correlação"])
titulos(fig, "Alguns pares são praticamente o mesmo investimento",
        "Correlação entre as 20 ações mais ligadas a cada índice (quanto mais escuro, mais andam juntas). Números: pares\n"
        "acima de 0,90, como as duas classes da Alphabet, da Petrobras e do Bradesco, e Itaúsa com Itaú.",
        base_pol=1.0, direita=0.86)
finalizar_figura("05_heatmaps_correlacao.png", fig)
display(pares_redundantes.sort_values("correlação", ascending=False).round(3).reset_index(drop=True))

**O que a correlação entre ações mostra**

- **Os pares redundantes são, quase todos, o mesmo risco listado duas vezes:** GOOG/GOOGL (0,996: classes da Alphabet), PETR3/PETR4 e BBDC3/BBDC4 (0,97: ordinárias e preferenciais) e ITSA4/ITUB4 (0,92: a Itaúsa é holding do Itaú). Os outros pares acima de 0,90 são do mesmo nicho: MA/V e LRCX/AMAT.
- **Entre as 20 ações mais ligadas ao índice, mais de 90% dos pares têm correlação entre 0,3 e 0,7.** São ações parecidas com o índice, mas não substitutas umas das outras.
- **Implicação para a modelagem (a testar):** com poucas ações, incluir os dois papéis de um mesmo par desperdiça uma vaga. Espera-se que o modelo escolha naturalmente um de cada par.

### 7.7 Análise por setor

In [ ]:
analise_setorial = correlacoes.groupby(["indice", "setor"], dropna=False).agg(
    ações=("ticker", "nunique"),
    **{"correlação média com índice": ("correlação com índice", "mean"),
       "correlação mediana com índice": ("correlação com índice", "median")},
).reset_index()
display(analise_setorial.sort_values(["indice", "correlação média com índice"], ascending=[True, False]).round(2).reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.4))
rng_jit = np.random.default_rng(7)
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    c = correlacoes[correlacoes["indice"] == nome]
    ordem = c.groupby("setor")["correlação com índice"].median().sort_values(ascending=True)
    for i, setor in enumerate(ordem.index):
        v = c[c["setor"] == setor]["correlação com índice"].to_numpy()
        ax.scatter(v, i + rng_jit.uniform(-0.13, 0.13, len(v)), s=22, color=CONTEXTO_ESC, edgecolor=FUNDO, linewidth=0.6, zorder=2)
        ax.plot([ordem[setor]] * 2, [i - 0.3, i + 0.3], color=CORES[nome], lw=3, solid_capstyle="round", zorder=3)
        ax.text(0.205, i, f"{setor} ({len(v)})", ha="left", va="center", fontsize=10, color=TINTA)
    ax.set_yticks([])
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color=GRADE)
    ax.set_xlim(0.2, 0.86)
    ax.set_ylim(-0.7, len(ordem) - 0.3)
    eixo_num(ax, "x", casas=1)
    ax.spines["bottom"].set_visible(False)
    ax.set_xlabel("Correlação com o índice")
    ax.set_title(NOMES[nome])
titulos(fig, "O setor financeiro é o que mais acompanha o índice nos dois mercados",
        "Correlação de cada ação com o índice (pontos), por setor; o traço colorido marca a mediana do setor e,\n"
        "entre parênteses, o número de ações. Setores com 1 a 3 ações não permitem conclusões.",
        esquerda=0.03)
finalizar_figura("06_correlacao_por_setor.png", fig)

**O que a análise setorial mostra**

- **O setor financeiro é o mais ligado ao índice nos dois mercados:** correlação média de 0,66 no S&P (15 ações) e 0,66 no IBOV (12 ações).
- **S&P 100:** tecnologia (25 ações, 0,61) e consumo cíclico (0,60) vêm em seguida. Saúde (0,43), utilidades públicas (0,43), consumo básico (0,44) e energia (0,44) reagem a fatores próprios.
- **IBOV:** depois do financeiro, os setores ficam próximos entre si (0,49 a 0,63), o que reforça a ideia de um fator doméstico comum. Os menores valores são de consumo básico (0,49) e comunicações (0,50).
- **Cuidado com a classificação do Yahoo:** a ITSA4, holding do Itaú, aparece como "Indústria", e por isso um ponto desse setor alcança 0,81. Grupos com 1 a 3 ações (Materiais básicos e Imobiliário no S&P, Tecnologia no IBOV) não permitem conclusões.
- **Implicação para a modelagem (a testar):** uma carteira escolhida só pela correlação ficaria concentrada em financeiro e tecnologia e sub-representaria os defensivos. Isso funciona quando o mercado anda junto, mas pode falhar em rotações setoriais.

### 7.8 Beta e risco idiossincrático

O retorno de cada ação pode ser separado em duas partes: o que acompanha o índice (o **beta**) e o que é próprio da empresa (o **risco idiossincrático**: o desvio-padrão do resíduo da regressão da ação contra o índice). Uma carteira pequena não consegue diversificar o risco idiossincrático. As melhores candidatas para replicar o índice têm, portanto, **beta perto de 1 e risco próprio baixo**.

In [ ]:
linhas = []
for nome in ["S&P100", "IBOV"]:
    R, y = retornos_eda[nome], bases[nome]["y"]
    for ticker in R:
        par = pd.concat([R[ticker], y], axis=1).dropna()
        beta = par.iloc[:, 0].cov(par.iloc[:, 1]) / par.iloc[:, 1].var()
        alfa = par.iloc[:, 0].mean() - beta * par.iloc[:, 1].mean()
        residuo = par.iloc[:, 0] - alfa - beta * par.iloc[:, 1]
        linhas.append({"indice": nome, "ticker": ticker, "beta": beta,
                       "vol idiossincrática (% a.a.)": 100 * residuo.std() * np.sqrt(ANUAL),
                       "R² com índice": 1 - residuo.var() / par.iloc[:, 0].var()})
beta_idio = pd.DataFrame(linhas).merge(setores, on=["indice", "ticker"], how="left")
display(beta_idio.groupby("indice")[["beta", "vol idiossincrática (% a.a.)", "R² com índice"]]
        .describe(percentiles=[0.1, 0.5, 0.9]).T.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 6.2))
candidatas_beta = []
rotulos_beta = []
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    g = beta_idio[beta_idio["indice"] == nome]
    ax.axvspan(0.8, 1.2, color=FAIXA, zorder=0)
    ax.text(1.0, g["vol idiossincrática (% a.a.)"].max() * 1.04, "beta entre 0,8 e 1,2", ha="center", va="bottom", fontsize=9.5, color=APAGADO)
    ax.scatter(g["beta"], g["vol idiossincrática (% a.a.)"], s=30, color=CONTEXTO_ESC, edgecolor=FUNDO, linewidth=0.8, zorder=2)
    boas = g[g["beta"].between(0.8, 1.2)].nsmallest(8, "vol idiossincrática (% a.a.)")
    candidatas_beta.append(boas)
    ax.scatter(boas["beta"], boas["vol idiossincrática (% a.a.)"], s=48, color=CORES[nome], edgecolor=FUNDO, linewidth=1.5, zorder=4)
    textos, xs_r, ys_r = [], [], []
    for _, l in boas.sort_values("vol idiossincrática (% a.a.)").iterrows():
        nome_t = l["ticker"].replace(".SA", "")
        if nome_t.startswith("GOOG") and any(t.startswith("GOOG") for t in textos):
            continue
        textos.append("GOOG/GOOGL" if nome_t.startswith("GOOG") else nome_t)
        xs_r.append(l["beta"]); ys_r.append(l["vol idiossincrática (% a.a.)"])
    rotulos_beta.append((ax, xs_r, ys_r, textos))
    eixo_num(ax, "x", casas=1); eixo_pct(ax, "y")
    ax.set_xlabel("Beta (sensibilidade ao índice)")
    ax.set_ylim(0, g["vol idiossincrática (% a.a.)"].max() * 1.12)
    ax.set_title(NOMES[nome])
axes[0].set_ylabel("Risco próprio (% ao ano)")
candidatas_beta = pd.concat(candidatas_beta)
titulos(fig, "As melhores candidatas se movem como o índice e têm pouco risco próprio",
        "Beta × volatilidade própria de cada ação (a parte do movimento que o índice não explica). Em cor, as 8 ações\n"
        "com beta entre 0,8 e 1,2 e menor risco próprio. Com poucas ações, é o risco próprio que vira erro de acompanhamento.")
for ax, xs_r, ys_r, textos in rotulos_beta:
    rotular_pontos(ax, xs_r, ys_r, textos)
finalizar_figura("15_beta_risco_idiossincratico.png", fig)
print("Beta entre 0,8 e 1,2 e menor risco idiossincrático (8 por índice):")
display(candidatas_beta[["indice", "ticker", "setor", "beta", "vol idiossincrática (% a.a.)", "R² com índice"]].round(2).reset_index(drop=True))

**O que o beta e o risco idiossincrático mostram**

- **A ação típica tem beta perto de 1, mas muito risco próprio.** O beta mediano é 0,93 (S&P) e 1,10 (IBOV), e a volatilidade idiossincrática mediana é de 25% e 29% a.a., maior que a volatilidade dos próprios índices. O índice explica, na mediana, só 30% (S&P) e 35% (IBOV) da variância de cada ação.
- **Há muitas ações com beta entre 0,8 e 1,2** (39 no S&P, 34 no IBOV), **mas poucas com risco próprio baixo.** As melhores candidatas:
  - S&P: MSFT, V, AAPL, BLK, MA, HD e GOOG/GOOGL, com 18% a 21% a.a. de volatilidade idiossincrática;
  - IBOV: ITSA4 (16%), ITUB4, BBDC3, EQTL3, BBAS3, BBDC4, SANB11 e ENGI11.
- **No S&P, as ações defensivas (saúde e consumo básico) têm beta baixo** (0,3 a 0,6): ajudam a reduzir a volatilidade da carteira, mas não a acompanhar o índice.
- **Implicação para a modelagem (a testar):** com K pequeno, o risco idiossincrático das poucas ações escolhidas é o que sobra como erro de *tracking*. Ações com risco próprio baixo devem aparecer com frequência nas carteiras.

### 7.9 As ações mais parecidas com o índice mudam com o tempo?

Calculamos a correlação de cada ação com o índice **ano a ano** e medimos quantas das 20 mais correlacionadas de um ano continuam entre as 20 no ano seguinte. Também calculamos a correlação de Spearman entre os rankings completos de anos consecutivos. Se o ranking muda muito, uma carteira escolhida com dados passados tende a precisar de mais trocas (turnover) a cada rebalanceamento.

In [ ]:
ANOS_COMPLETOS = list(range(DATA_INICIAL.year + 1, DATA_FINAL.year + 1))
correlacao_anual = {}
for nome in ["S&P100", "IBOV"]:
    R, y = bases[nome]["R"], bases[nome]["y"]
    correlacao_anual[nome] = pd.DataFrame({
        ano: R.loc[str(ano)].loc[:, R.loc[str(ano)].notna().mean() >= 0.90].corrwith(y.loc[str(ano)])
        for ano in ANOS_COMPLETOS
    })

linhas = []
for nome, C in correlacao_anual.items():
    for a, b_ in zip(ANOS_COMPLETOS[:-1], ANOS_COMPLETOS[1:]):
        comuns = C[[a, b_]].dropna()
        top_a, top_b = set(comuns[a].nlargest(20).index), set(comuns[b_].nlargest(20).index)
        linhas.append({"indice": nome, "transição": f"{a}→{b_}", "permanecem no top 20": len(top_a & top_b),
                       "Spearman do ranking": comuns[a].corr(comuns[b_], method="spearman")})
estabilidade_ranking = pd.DataFrame(linhas)
display(estabilidade_ranking.pivot(index="transição", columns="indice", values=["permanecem no top 20", "Spearman do ranking"]).round(2))

nucleo_estavel = {}
for nome, C in correlacao_anual.items():
    tops = [set(C[a].dropna().nlargest(20).index) for a in ANOS_COMPLETOS]
    nucleo_estavel[nome] = sorted(set.intersection(*tops))
    print(f"{nome}: {len(nucleo_estavel[nome])} ações no top 20 em todos os anos de {ANOS_COMPLETOS[0]} a {ANOS_COMPLETOS[-1]}: "
          + ", ".join(t.replace(".SA", "") for t in nucleo_estavel[nome]))

fig, ax = plt.subplots(figsize=(12, 5.6))
for nome in ["S&P100", "IBOV"]:
    g = estabilidade_ranking[estabilidade_ranking["indice"] == nome].reset_index(drop=True)
    xs = np.arange(len(g))
    ax.plot(xs, g["permanecem no top 20"], color=CORES[nome], marker="o", ms=7, mec=FUNDO, mew=2, label=NOMES[nome])
    rotulo(ax, xs[-1], g["permanecem no top 20"].iloc[-1], NOMES[nome], cor=None, marcador=False, dx=12)
g_sp = estabilidade_ranking[estabilidade_ranking["indice"] == "S&P100"].reset_index(drop=True)
i_min = int(g_sp["permanecem no top 20"].idxmin())
nota(ax, i_min, g_sp.loc[i_min, "permanecem no top 20"], f"Só {g_sp.loc[i_min, 'permanecem no top 20']} de 20\npermaneceram", 18, -28)
ax.set_xticks(np.arange(len(g_sp)))
ax.set_xticklabels(g_sp["transição"].str.replace("→", "–"))
ax.set_ylim(0, 20.5)
ax.yaxis.set_major_locator(FixedLocator([0, 5, 10, 15, 20]))
ax.set_xlim(-0.3, len(g_sp) - 0.3)
legenda(ax, loc="lower left")
nucleo_txt = f"Em todos os anos, só {len(nucleo_estavel['S&P100'])} ação do S&P 100 ({', '.join(nucleo_estavel['S&P100'])}) esteve entre as 20; no Ibovespa, {len(nucleo_estavel['IBOV'])}." if len(nucleo_estavel['S&P100']) == 1 else ""
titulos(fig, "No S&P 100, o grupo das ações mais parecidas com o índice muda muito de um ano para o outro",
        "Das 20 ações mais correlacionadas com o índice em um ano, quantas continuam entre as 20 no ano seguinte.\n" + nucleo_txt,
        direita=0.88)
finalizar_figura("16_estabilidade_ranking.png", fig)

**O que a estabilidade do ranking mostra**

- **O ranking completo é razoavelmente estável:** Spearman entre anos consecutivos de 0,52 a 0,86 no S&P e de 0,66 a 0,77 no IBOV. Ações muito ou pouco parecidas com o índice tendem a continuar assim.
- **Mas o topo muda bastante, sobretudo no S&P.** Das 20 ações mais correlacionadas em 2024, só 8 continuaram no top 20 em 2025. No IBOV, a permanência fica entre 12 e 15 a cada ano.
- **Núcleo estável:** só a **AVGO** esteve entre as 20 mais correlacionadas com o S&P em todos os anos de 2020 a 2026. No IBOV, quatro: **B3SA3, BPAC11, ITSA4 e ITUB4**.
- **Implicação para a modelagem (a testar):** uma carteira pequena montada com dados passados deve precisar de mais trocas a cada rebalanceamento no S&P do que no IBOV. Isso afeta o turnover e a escolha da frequência de rebalanceamento.

### 7.10 Correlação por regime de mercado

A correlação calculada em sete anos mistura períodos muito diferentes. Para index tracking, o que importa é **o quanto as ações andam juntas**: se o mercado é dominado por um fator comum, poucas ações reproduzem o índice; se cada ação anda por conta própria, são necessárias mais ações.

Indicadores por regime:
- **correlação média entre pares**: média das correlações entre todas as ações;
- **correlação mediana com o índice**;
- **variância explicada pelo 1º componente principal (PC1)**: o peso do fator comum de mercado;
- **dispersão cross-section**: desvio-padrão diário entre os retornos das ações, em média.

O gráfico usa a correlação média entre pares em janela móvel de 63 pregões (~1 trimestre).

In [ ]:
REGIMES = {
    "Pré-Covid (out/19–fev/20)": ("2019-10-01", "2020-02-20"),
    "Crash Covid (fev–abr/20)": ("2020-02-21", "2020-04-30"),
    "Recuperação (mai/20–dez/21)": ("2020-05-01", "2021-12-31"),
    "Aperto monetário (2022)": ("2022-01-01", "2022-12-31"),
    "2023–2024": ("2023-01-01", "2024-12-31"),
    "2025–set/2026": ("2025-01-01", "2026-09-30"),
}


def indicadores_regime(R, y, inicio, fim):
    Rs = R.loc[inicio:fim]
    Rs = Rs.loc[:, Rs.notna().mean() >= 0.90]
    C = Rs.corr().to_numpy()
    sup = np.triu_indices_from(C, 1)
    autovalores = np.linalg.eigvalsh(np.corrcoef(Rs.dropna().T.to_numpy()))
    ys = y.loc[inicio:fim]
    return {
        "pregões": len(ys.dropna()),
        "ações": Rs.shape[1],
        "correlação média entre pares": np.nanmean(C[sup]),
        "correlação mediana com índice": Rs.corrwith(ys).median(),
        "PC1 (% da variância)": 100 * autovalores[-1] / autovalores.sum(),
        "volatilidade do índice (% a.a.)": 100 * ys.std() * np.sqrt(ANUAL),
        "dispersão cross-section (% ao dia)": 100 * Rs.std(axis=1).mean(),
    }


correlacao_regimes = pd.DataFrame([
    {"indice": nome, "regime": regime, **indicadores_regime(retornos_eda[nome], bases[nome]["y"], *datas)}
    for nome in ["S&P100", "IBOV"] for regime, datas in REGIMES.items()
])
display(correlacao_regimes.set_index(["indice", "regime"]).round(2))


def correlacao_media_movel(R, janela=63):
    # Identidade: N² Var(média) = Σσi² + Σ_{i≠j} σiσjρij
    cols = R.columns[R.notna().mean() >= 0.98]
    X = R[cols].fillna(0.0)
    n = len(cols)
    var_media = X.mean(axis=1).rolling(janela).var()
    sigma = X.rolling(janela).std()
    soma_var = (sigma ** 2).sum(axis=1)
    return (n ** 2 * var_media - soma_var) / (sigma.sum(axis=1) ** 2 - soma_var)


fig, ax = plt.subplots(figsize=(12, 6.2))
nomes_curtos = ["Pré-\nCovid", "Crash\nCovid", "Recuperação", "Alta de juros\n(2022)", "2023–2024", "2025–2026"]
for i, (regime, (ini, fim)) in enumerate(REGIMES.items()):
    if i % 2 == 1:
        ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fim), color=FAIXA, zorder=0)
    meio = pd.Timestamp(ini) + (pd.Timestamp(fim) - pd.Timestamp(ini)) / 2
    if i != 0:
        ax.text(meio, 0.97, nomes_curtos[i], ha="center", va="top", fontsize=9.5, color=APAGADO, transform=ax.get_xaxis_transform())
medias_recentes = {}
for nome in ["S&P100", "IBOV"]:
    rho = correlacao_media_movel(retornos_eda[nome]).dropna()
    ax.plot(rho.index, rho, color=CORES[nome], lw=1.9, label=NOMES[nome], zorder=3)
    medias_recentes[nome] = correlacao_regimes.set_index(["indice", "regime"]).loc[(nome, "2023–2024"), "correlação média entre pares"]
    rotulo(ax, rho.index[-1], rho.iloc[-1], NOMES[nome], cor=CORES[nome])
ax.set_ylim(0, 0.9)
eixo_num(ax, casas=1)
ax.set_xlim(None, retornos_eda["S&P100"].index[-1] + pd.Timedelta(days=170))
legenda(ax, loc="upper right", bbox_to_anchor=(0.86, 0.86))
covid = correlacao_regimes.set_index(["indice", "regime"]).loc[("S&P100", "Crash Covid (fev–abr/20)"), "correlação média entre pares"]
nota(ax, pd.Timestamp("2020-04-15"), 0.75, f"Na Covid, tudo andou junto\n(média de {num(covid, 2)})", 40, 5)
r_sp = correlacao_regimes.set_index(["indice", "regime"])["correlação média entre pares"]
titulos(fig, "Desde 2023, as ações do S&P 100 andam muito menos juntas que as do Ibovespa",
        f"Correlação média entre pares de ações, em janelas móveis de 63 pregões (~1 trimestre). Média em 2023–2024: "
        f"{num(r_sp[('S&P100', '2023–2024')], 2)} no S&P 100\ne {num(r_sp[('IBOV', '2023–2024')], 2)} no Ibovespa. Quando as ações andam menos juntas, poucas delas reproduzem pior o índice.",
        direita=0.9)
finalizar_figura("11_correlacao_por_regime.png", fig)

**O que a correlação por regime mostra**

Esta é a descoberta estrutural mais importante do EDA.

| Regime | Correlação entre pares S&P | Correlação entre pares IBOV | PC1 S&P | PC1 IBOV |
|---|---|---|---|---|
| Pré-Covid | 0,24 | 0,27 | 29% | 27% |
| Crash Covid | **0,72** | **0,69** | 73% | 70% |
| Recuperação 2020–21 | 0,30 | 0,33 | 32% | 37% |
| Aperto monetário 2022 | 0,43 | 0,32 | 45% | 36% |
| 2023–2024 | **0,19** | 0,30 | 22% | 33% |
| 2025–set/2026 | **0,19** | 0,34 | 23% | 38% |

- **Nas crises, tudo anda junto:** no crash da Covid, um único fator explica cerca de 70% da variância nos dois mercados.
- **No S&P, a coesão caiu fortemente a partir de 2023:** correlação média entre pares de 0,19 e correlação mediana com o índice de 0,38 a 0,41. A queda coincide com o rali concentrado em poucas mega-caps ligadas à IA: o índice subiu puxado por poucos nomes, enquanto as demais ações seguiram trajetórias próprias.
- **Há episódios dentro dos regimes.** No choque das tarifas (abr–jul/2025), a correlação móvel do S&P saltou para cerca de 0,5 e depois voltou a cair, chegando a menos de 0,1 em 2026. Nos momentos de estresse a coesão volta; nos períodos calmos recentes, é muito baixa.
- **No IBOV, a coesão ficou estável** (0,30 a 0,34 entre pares; PC1 de 33% a 38%) e, desde 2023, é maior que a do S&P.
- **Volatilidade e co-movimento são coisas diferentes:** o IBOV foi mais volátil em 4 dos 6 regimes, mas desde 2023 suas ações andam mais juntas que as do S&P.
- **Implicação para a modelagem (hipótese a testar):** replicar o S&P 100 com poucas ações deve ser **mais difícil no período recente** do que replicar o IBOV, ao contrário do que a volatilidade sugeriria.

### 7.11 As ações andam mais juntas nas quedas?

Separamos os pregões pelo retorno do índice: os 10% piores, os 80% centrais e os 10% melhores. Em cada grupo, calculamos a correlação mediana das ações com o índice.

> Cuidado de leitura: correlações calculadas em subconjuntos de dias extremos não são diretamente comparáveis à correlação do período inteiro. A comparação válida é entre **piores** e **melhores** dias, que têm o mesmo tamanho e o mesmo grau de extremidade.

In [ ]:
linhas = []
for nome in ["S&P100", "IBOV"]:
    R, y = retornos_eda[nome], bases[nome]["y"].dropna()
    R = R.loc[y.index]
    q10, q90 = y.quantile([0.10, 0.90])
    grupos = {"10% piores dias": y <= q10, "80% centrais": (y > q10) & (y < q90), "10% melhores dias": y >= q90}
    for grupo, mascara in grupos.items():
        linhas.append({"indice": nome, "grupo": grupo, "pregões": int(mascara.sum()),
                       "correlação mediana com índice": R[mascara].corrwith(y[mascara]).median(),
                       "retorno médio do índice (%)": 100 * y[mascara].mean()})
assimetria = pd.DataFrame(linhas)
ordem = ["10% piores dias", "80% centrais", "10% melhores dias"]
display(assimetria.pivot(index="grupo", columns="indice", values="correlação mediana com índice").loc[ordem].round(2))

fig, ax = plt.subplots(figsize=(11, 5.8))
xs = np.arange(3)
for nome in ["S&P100", "IBOV"]:
    g = assimetria[assimetria["indice"] == nome].set_index("grupo").loc[ordem, "correlação mediana com índice"]
    ax.plot(xs, g.values, color=CORES[nome], marker="o", ms=9, mec=FUNDO, mew=2, lw=2.2, label=NOMES[nome], zorder=3)
    for x, v in zip(xs, g.values):
        ax.annotate(num(v, 2), (x, v), xytext=(0, 12 if nome == "IBOV" else -16), textcoords="offset points",
                    ha="center", fontsize=10.5, color=TINTA, fontweight="semibold")
    rotulo(ax, xs[-1], g.values[-1], NOMES[nome], cor=None, marcador=False, dx=16)
ax.set_xticks(xs)
ax.set_xticklabels(["10% piores dias\ndo índice", "80% dos dias\n(dias comuns)", "10% melhores dias\ndo índice"])
ax.set_xlim(-0.3, 2.6)
ax.set_ylim(0.2, 0.75)
eixo_num(ax, casas=1)
legenda(ax, loc="upper center")
a = assimetria.set_index(["indice", "grupo"])["correlação mediana com índice"]
titulos(fig, "No Ibovespa, as ações caem juntas, mas sobem separadas",
        f"Correlação mediana das ações com o índice, conforme o desempenho do índice no dia. No Ibovespa, ela cai de "
        f"{num(a[('IBOV', '10% piores dias')], 2)} nos piores dias\npara {num(a[('IBOV', '10% melhores dias')], 2)} nos melhores; no S&P 100, quase não muda. "
        "Uma carteira pequena de ações brasileiras tende a acompanhar melhor as quedas.")
finalizar_figura("17_assimetria_correlacao.png", fig)

**O que a assimetria mostra**

| | 10% piores dias | 80% centrais | 10% melhores dias |
|---|---|---|---|
| S&P 100 | 0,60 | 0,29 | 0,57 |
| IBOV | 0,63 | 0,38 | 0,43 |

- **No S&P, a correlação é praticamente simétrica:** as ações andam juntas tanto nas quedas fortes quanto nas altas fortes.
- **No IBOV, as ações caem juntas, mas sobem de forma dispersa.** Nos piores dias, a correlação mediana com o índice é 0,63; nos melhores, 0,43. Em dias de pânico prevalece o fator "Brasil" (juros, câmbio, risco fiscal); em dias de alta, cada empresa reage a notícias próprias.
- **Nos dias comuns, a correlação é bem menor** (0,29 e 0,38): é nesses dias, a grande maioria, que o risco próprio de cada ação mais pesa.
- **Implicação para a modelagem (a testar):** no IBOV, uma carteira pequena deve acompanhar melhor o índice nas quedas do que nas altas, o que pode gerar um erro assimétrico (perder parte dos ralis).

### 7.12 Quantas "dimensões" tem cada mercado?

A análise de componentes principais (PCA) decompõe os retornos das ações em fatores independentes, ordenados pela variância que explicam. Se poucos fatores explicam quase tudo, poucas ações bastam para reproduzir o mercado. Se a variância está espalhada por muitos fatores, uma carteira pequena deixa parte relevante do comportamento de fora.

Usamos as ações com pelo menos 98% de dados e a matriz de correlação (cada ação com o mesmo peso). O gráfico mostra o regime mais recente (2023–set/2026); a tabela traz também o período completo.

In [ ]:
def variancia_acumulada(R):
    Rc = R.loc[:, R.notna().mean() >= 0.98].dropna()
    autovalores = np.linalg.eigvalsh(np.corrcoef(Rc.T.to_numpy()))[::-1]
    return np.cumsum(autovalores) / autovalores.sum(), Rc.shape[1]


linhas = []
curvas = {}
for nome in ["S&P100", "IBOV"]:
    for periodo, ini in [("2019–2026", None), ("2023–2026", "2023-01-01")]:
        acum, n = variancia_acumulada(retornos_eda[nome].loc[ini:])
        curvas[(nome, periodo)] = acum
        linhas.append({"indice": nome, "período": periodo, "ações": n,
                       "PC1 (%)": 100 * acum[0],
                       "componentes para 50%": int(np.searchsorted(acum, 0.50) + 1),
                       "componentes para 80%": int(np.searchsorted(acum, 0.80) + 1),
                       "variância nos 5 primeiros (%)": 100 * acum[4],
                       "variância nos 10 primeiros (%)": 100 * acum[9]})
pca_variancia = pd.DataFrame(linhas)
display(pca_variancia.round(1))

fig, ax = plt.subplots(figsize=(12, 5.8))
k80 = {}
for nome in ["S&P100", "IBOV"]:
    acum = curvas[(nome, "2023–2026")]
    ks = np.arange(1, len(acum) + 1)
    ax.plot(ks, 100 * acum, color=CORES[nome], lw=2.3, label=NOMES[nome], zorder=3)
    k = int(np.searchsorted(acum, 0.80) + 1)
    k80[nome] = k
    ax.plot([k], [100 * acum[k - 1]], "o", ms=8, color=CORES[nome], mec=FUNDO, mew=2, zorder=4)
    ax.annotate(f"{NOMES[nome]}: {k} fatores", (k, 100 * acum[k - 1]), xytext=(8, -16 if nome == "S&P100" else 10),
                textcoords="offset points", fontsize=10.5, fontweight="semibold", color=TINTA)
ax.axhline(80, color=TINTA_2, lw=0.9, zorder=2)
ax.text(1, 81.5, "80% do movimento das ações", fontsize=10, color=TINTA_2)
ax.set_xlim(0, 60)
ax.set_ylim(15, 100)
eixo_pct(ax)
ax.set_xlabel("Número de fatores (componentes principais)")
legenda(ax, loc="lower right")
titulos(fig, f"O S&P 100 recente precisa de {k80['S&P100']} fatores para explicar 80% do movimento das ações; o Ibovespa, de {k80['IBOV']}",
        "Variância acumulada explicada pelos componentes principais dos retornos das ações, 2023 a set/2026. O primeiro fator é\n"
        "o \"mercado\"; os demais, setores, temas e empresas. Quanto mais fatores, mais difícil reproduzir o índice com poucas ações.")
finalizar_figura("18_pca_variancia_acumulada.png", fig)

**O que a PCA mostra**

- **Um fator domina, mas não basta.** O primeiro componente (o "mercado") explica 36% da variância das ações do S&P e 41% das do IBOV no período todo. Para chegar a 50% bastam 3 a 4 componentes, mas para 80% são precisos **34 no S&P e 25 no IBOV**.
- **No regime recente, o S&P ficou ainda mais "multidimensional":** em 2023–2026, o PC1 cai para 22% e são precisos 9 componentes para 50% e 41 para 80%. No IBOV, a mudança é menor (6 e 32).
- **Leitura para o projeto:** uma carteira com 5 ações consegue capturar o fator de mercado, mas não as dezenas de fatores menores (setores, temas, empresas) que também movem o índice. Quanto mais espalhada a variância, maior o erro de uma carteira pequena.
- **Implicação para a modelagem (a testar):** o EDA sugere, por três caminhos independentes (regime, PCA e estabilidade do ranking), que o S&P recente é o caso mais difícil para K pequeno.

### 7.13 Liquidez: a carteira seria implementável?

O volume em número de ações não é comparável entre papéis com preços diferentes. Usamos o **volume financeiro** (preço de fechamento × volume) em moeda local, mediana dos últimos 252 pregões, e a **fração de dias com retorno exatamente zero**, um sinal de papel pouco negociado (preço repetido).

In [ ]:
linhas = []
for nome in ["S&P100", "IBOV"]:
    b = bases[nome]
    financeiro = (b["close_negociado"] * b["volume"]).iloc[-ANUAL:]
    R = b["R"]
    for ticker in b["R"].columns:
        linhas.append({"indice": nome, "ticker": ticker,
                       "volume financeiro mediano (milhões, moeda local)": financeiro[ticker].median() / 1e6,
                       "dias com retorno zero (%)": 100 * (R[ticker] == 0).sum() / max(R[ticker].notna().sum(), 1)})
liquidez = pd.DataFrame(linhas).merge(setores, on=["indice", "ticker"], how="left")
col_vol = "volume financeiro mediano (milhões, moeda local)"


def moeda_mi(v, nome):
    simbolo = "US$" if nome == "S&P100" else "R$"
    return f"{simbolo} {num(v / 1000, 1)} bi" if v >= 1000 else f"{simbolo} {num(v, 0)} mi"


fig, axes = plt.subplots(2, 1, figsize=(12, 6.4))
rng_jit = np.random.default_rng(3)
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    g = liquidez[liquidez["indice"] == nome].dropna(subset=[col_vol])
    v = g[col_vol].to_numpy()
    yj = rng_jit.uniform(-0.28, 0.28, len(v))
    ax.scatter(v, yj, s=34, color=CORES_CLARAS[nome], edgecolor=FUNDO, linewidth=0.8, zorder=2)
    med = np.median(v)
    ax.plot([med, med], [-0.42, 0.42], color=CORES[nome], lw=3, solid_capstyle="round", zorder=3)
    ax.annotate(f"mediana: {moeda_mi(med, nome)}", (med, 0.42), xytext=(0, 6), textcoords="offset points",
                ha="center", va="bottom", fontsize=10, color=TINTA, fontweight="semibold")
    for _, l in g.nsmallest(1, col_vol).iterrows():
        i = list(g.index).index(_)
        ax.annotate(f"{l['ticker'].replace('.SA', '')}: {moeda_mi(l[col_vol], nome)}", (l[col_vol], yj[i]), xytext=(-6, -16),
                    textcoords="offset points", ha="left", fontsize=9.5, color=TINTA_2)
    mx = g.loc[g[col_vol].idxmax()]
    i = list(g.index).index(g[col_vol].idxmax())
    ax.annotate(f"{mx['ticker'].replace('.SA', '')}: {moeda_mi(mx[col_vol], nome)}", (mx[col_vol], yj[i]), xytext=(-6, 12),
                textcoords="offset points", ha="right", fontsize=9.5, color=TINTA_2)
    ax.set_xscale("log")
    ax.set_yticks([]); ax.set_ylim(-0.6, 0.8)
    ax.grid(axis="y", visible=False); ax.grid(axis="x", color=GRADE)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _, n=nome: moeda_mi(x, n)))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_title(NOMES[nome])
mins = {n: liquidez[liquidez["indice"] == n][col_vol].min() for n in ["S&P100", "IBOV"]}
titulos(fig, "Até a ação menos negociada movimenta dezenas de milhões por dia: a liquidez não limita a carteira",
        f"Volume financeiro mediano diário de cada ação nos últimos 252 pregões (escala logarítmica, moeda local). Mínimos:\n"
        f"{moeda_mi(mins['S&P100'], 'S&P100')} no S&P 100 e {moeda_mi(mins['IBOV'], 'IBOV')} no Ibovespa.",
        espaco_h=0.6)
finalizar_figura("19_liquidez.png", fig)

texto = ["**Leitura automática dos números de liquidez** (recalculada a cada execução)", ""]
for nome, moeda in [("S&P100", "US$"), ("IBOV", "R$")]:
    g = liquidez[liquidez["indice"] == nome].dropna(subset=[col_vol])
    menos = g.nsmallest(5, col_vol)
    zeros = g.nlargest(3, "dias com retorno zero (%)")
    texto.append(
        f"- **{nome}:** volume financeiro mediano de {moeda} {g[col_vol].median():,.0f} milhões por dia "
        f"(mínimo {moeda} {g[col_vol].min():,.1f} mi, máximo {moeda} {g[col_vol].max():,.0f} mi). "
        "Menos líquidas: " + ", ".join(t.replace(".SA", "") for t in menos["ticker"]) + ". "
        "Mais dias com retorno zero: " + ", ".join(f"{t.replace('.SA', '')} ({v:.1f}%)" for t, v in zip(zeros["ticker"], zeros["dias com retorno zero (%)"])) + "."
    )
display(Markdown("\n".join(texto)))
display(liquidez.sort_values(col_vol).groupby("indice").head(8).round(2).reset_index(drop=True))

**O que a liquidez mostra**

- **A liquidez não restringe a escolha de ações em nenhum dos dois índices.** A ação menos líquida do S&P (EMR) negocia US$ 367 milhões por dia, e a do IBOV (YDUQ3) R$ 39 milhões por dia. Os dois valores comportam com folga um fundo de porte médio, então a liquidez não precisa entrar como restrição no modelo.
- **A diferença de nível entre os mercados é grande:** a ação mediana do S&P negocia US$ 1,25 bilhão por dia, e a do IBOV R$ 134 milhões. Mesmo sem converter as moedas, o mercado americano é muitas vezes mais líquido.
- **Ao contrário do que se poderia esperar, a dispersão é maior no S&P:** a liquidez varia 82 vezes entre a ação mais e a menos negociada, contra 40 vezes no IBOV. No S&P, poucas mega-caps (como NVDA, acima de US$ 30 bilhões por dia) concentram o volume. No IBOV, as menos líquidas são empresas menores e recentes (YDUQ3, CMIN3, AURE3, FLRY3, VAMO3).
- **Preço repetido é raro:** no S&P, no máximo 1,3% dos dias têm retorno zero (PFE, T). No IBOV, POMO4 (5,2%), CMIN3 (4,6%) e COGN3 (4,5%) se destacam. CMIN3 aparece nas duas listas (pouco volume e muitos dias parados) e seria a primeira candidata a um filtro de liquidez, se o grupo quiser essa restrição de negócio.

> O volume financeiro usa o `Close` negociado e o volume do Yahoo, em moeda local, nos últimos 252 pregões. É uma medida de ordem de grandeza, não um custo de execução.

### 7.14 Concentração dos índices (aproximação)

Os pesos oficiais dos índices não estão disponíveis no Yahoo. Aproximamos o peso de cada ação pelo **valor de mercado atual** (`marketCap`). É uma aproximação: os índices usam o valor de mercado ajustado pelo *free float* (ações em circulação), e o IBOV ainda aplica limites de peso. O Yahoo informa, para cada classe de ação, o valor da **empresa inteira** (GOOG/GOOGL, PETR3/PETR4, BBDC3/BBDC4); por isso contamos cada uma dessas empresas uma única vez.

Indicadores: peso das 10 maiores, **número efetivo de ações** (1 / Σ peso², ou seja, quantas ações de peso igual teriam a mesma concentração) e quantas ações somam 50% e 80% do índice.

In [ ]:
PARES_MESMA_EMPRESA = [("GOOG", "GOOGL"), ("PETR3.SA", "PETR4.SA"), ("BBDC3.SA", "BBDC4.SA")]  # classes da mesma empresa

pesos_aprox = []
for nome in ["S&P100", "IBOV"]:
    m = metadados[(metadados["indice"] == nome)].set_index("ticker")["valor_mercado"].astype(float).dropna()
    m = m[m > 0].copy()
    for a, b_ in PARES_MESMA_EMPRESA:
        # O Yahoo informa, para cada classe, o valor de mercado da EMPRESA inteira: contamos a empresa uma vez.
        if a in m and b_ in m:
            m = m.drop(a if m[a] < m[b_] else b_)
    w = (m / m.sum()).sort_values(ascending=False)
    pesos_aprox.append(pd.DataFrame({"indice": nome, "ticker": w.index, "peso aproximado (%)": 100 * w.values}))
pesos_aprox = pd.concat(pesos_aprox, ignore_index=True)

linhas = []
fig, ax = plt.subplots(figsize=(12, 5.8))
for nome in ["S&P100", "IBOV"]:
    w = pesos_aprox[pesos_aprox["indice"] == nome]["peso aproximado (%)"].to_numpy() / 100
    if len(w) == 0:
        continue
    acum = np.cumsum(w)
    ks = np.arange(1, len(w) + 1)
    ax.plot(ks, 100 * acum, color=CORES[nome], lw=2.3, label=NOMES[nome] + (" (aproximação sem free float)" if nome == "IBOV" else ""), zorder=3)
    for k in (5, 10, 20):
        ax.plot([k], [100 * acum[k - 1]], "o", ms=7, color=CORES[nome], mec=FUNDO, mew=2, zorder=4)
        ax.annotate(pct(100 * acum[k - 1]), (k, 100 * acum[k - 1]), xytext=(6, -14 if nome == "S&P100" else 8),
                    textcoords="offset points", fontsize=10, color=TINTA, fontweight="semibold")
    linhas.append({"indice": nome, "ações com valor de mercado": len(w),
                   "peso das 5 maiores (%)": 100 * acum[4], "peso das 10 maiores (%)": 100 * acum[9],
                   "peso das 20 maiores (%)": 100 * acum[19],
                   "número efetivo de ações": 1 / np.sum(w ** 2),
                   "ações para 50% do peso": int(np.searchsorted(acum, 0.50) + 1),
                   "ações para 80% do peso": int(np.searchsorted(acum, 0.80) + 1)})
ax.axhline(50, color=TINTA_2, lw=0.9, zorder=2)
ax.text(len(w) * 0.98, 51.5, "metade do índice", ha="right", fontsize=10, color=TINTA_2)
for k in (5, 10, 20):
    ax.axvline(k, color=GRADE, lw=0.8, zorder=1)
eixo_pct(ax)
ax.set_ylim(0, 102)
ax.set_xlabel("Número de ações, das maiores para as menores")
legenda(ax, loc="lower right")
concentracao = pd.DataFrame(linhas)
display(concentracao.round(1))
c_sp = concentracao.set_index("indice").loc["S&P100"]
titulos(fig, f"No S&P 100, {int(c_sp['ações para 50% do peso'])} ações somam metade do índice",
        "Peso acumulado das maiores ações, aproximado pelo valor de mercado atual. Pontos: 5, 10 e 20 maiores. No Ibovespa, a\n"
        "aproximação ignora o free float e superestima empresas com controlador (Petrobras, Ambev, WEG, Itaú).")
finalizar_figura("20_concentracao_indices.png", fig)

texto = ["**Leitura automática da concentração** (aproximação recalculada a cada execução)", ""]
for _, l in concentracao.iterrows():
    top5 = pesos_aprox[pesos_aprox["indice"] == l["indice"]].head(5)
    texto.append(
        f"- **{l['indice']}:** as 10 maiores somam cerca de {l['peso das 10 maiores (%)']:.0f}% do índice; "
        f"o número efetivo de ações é {l['número efetivo de ações']:.0f}; {l['ações para 50% do peso']:.0f} ações somam metade do peso. "
        "Maiores: " + ", ".join(f"{t.replace('.SA', '')} ({p:.1f}%)" for t, p in zip(top5["ticker"], top5["peso aproximado (%)"])) + "."
    )
if len(concentracao) == 2:
    c_sp, c_ib = concentracao.set_index("indice").loc["S&P100"], concentracao.set_index("indice").loc["IBOV"]
    mais = "IBOV" if c_ib["número efetivo de ações"] < c_sp["número efetivo de ações"] else "S&P100"
    texto.append("")
    texto.append(
        f"- **Comparação:** pelo número efetivo de ações, o **{mais}** é o mais concentrado "
        f"({c_sp['número efetivo de ações']:.0f} no S&P contra {c_ib['número efetivo de ações']:.0f} no IBOV). "
        f"Com K = 10, as 10 maiores cobrem {c_sp['peso das 10 maiores (%)']:.0f}% do S&P e {c_ib['peso das 10 maiores (%)']:.0f}% do IBOV."
    )
display(Markdown("\n".join(texto)))

**O que a concentração mostra**

- **S&P 100: poucas ações carregam o índice.** NVDA (10,6%), AAPL (9,4%), Alphabet (8,1%, contada uma vez), MSFT (7,4%) e AMZN (5,2%) somam 41%. As 10 maiores chegam a 55%, e apenas 8 ações somam metade do peso. O número efetivo de ações é 24: o índice se comporta como uma carteira de 24 ações de peso igual, não de 100.
- **IBOV, pelo valor de mercado total:** Petrobras (15,2%), ITUB4 (10,4%), VALE3 (5,5%), BPAC11 (5,0%) e ABEV3 (4,7%) lideram. As 10 maiores somam 58%, 8 ações somam metade do peso e o número efetivo é 19, menor que o do S&P.
- **Mas, no IBOV, essa aproximação distorce os pesos.** O índice oficial pondera pelas ações em circulação (*free float*). Empresas com controlador relevante (o governo na Petrobras e no Banco do Brasil, a AB InBev na Ambev, as famílias controladoras no Itaú e na WEG) pesam bem menos no Ibovespa do que o valor de mercado total sugere. Já a Vale, de capital pulverizado, pesa mais. **Por isso a conclusão de que o IBOV é mais concentrado que o S&P não é robusta.** Para o S&P, em que o capital da maioria das empresas está em circulação, a aproximação é razoável.
- **Concentração e co-movimento juntos explicam a dificuldade do S&P.** As maiores ações pesam muito, mas andam pouco juntas desde 2023 (seção 7.10). Uma carteira de 10 ações cobre metade do peso do índice, mas não reproduz os movimentos da outra metade, que seguem trajetórias próprias.
- **Referência para K:** uma "réplica parcial" com as K maiores ações, nos pesos do índice, é o ponto de comparação mais simples para o modelo. Para usá-la de fato, os pesos oficiais deveriam vir de uma fonte com *free float*: a carteira teórica do Ibovespa publicada pela B3 e a composição do ETF do S&P 100 (OEF).
- **Outras limitações:** pesos atuais, não históricos. A ITSA4 contém participação no Itaú, que também está no índice.

## 7.15 Síntese do EDA

**1. Os dados são confiáveis?**
Sim, para o objetivo do projeto, com a incerteza declarada.
- **Etapa 2 (séries):** uma série com problema, a NATU3 (código reaproveitado), recortada. Sem datas duplicadas, preços inválidos ou cotações congeladas.
- **Etapa 3 (retornos):** dos 460 extremos de ±15%, 186 foram puxados pelo mercado e 273 vieram da empresa ou do setor; o último é um erro do Yahoo (UGPA3, erro de escala), corrigido com o retorno oficial da B3. Dos 50 eventos de empresa verificados, 49 são reais (34 por notícia, 6 pelos pares e 9 pela cotação oficial) e 1 é o erro. Não dá para garantir que não há outros erros, mas a taxa entre os não verificados fica abaixo de 14% (95% de confiança), e amortecer os extremos no treino muda pouco o erro do modelo (seção 13.10).
- **Etapa 4:** as ações do universo reproduzem cada índice com R² ≥ 0,99 em todos os anos.
- O viés de sobrevivência é **relevante no IBOV**: o erro residual do universo atual cai de 2,5% a.a. em 2020 para 0,6% em 2026, e tem direção: com os pesos de um ano mantidos no ano seguinte, o universo ficou em média 1,8 p.p. por ano acima do IBOV (1,0 p.p. acima do S&P).
- No S&P ele é pequeno até 2025. Em 2026 aparece a troca de 21/09/2026 (entram DELL, PANW, ANET e SNDK; saem NKE, HONA, SPG e CL), que nossa fotografia de 30/09 já incorpora.

**2. Como os índices se comportaram e quais crises aparecem?**
O S&P rendeu +190,7% com Sharpe de 0,85; o IBOV, +77,9% com Sharpe de 0,47. A Covid domina tudo: pior dia, pior mês, pico de volatilidade e queda máxima nos dois índices. Seguem o aperto do Fed em 2022, a interferência na Petrobras em 2021, a eleição de 2022 e as tarifas de abril/2025. Os ciclos dos dois índices foram opostos em vários anos (correlação diária de 0,56).

**3. Quais ações se parecem com o índice, e quais andam juntas?**
- **S&P:** MSFT, AAPL, BLK, GOOG/GOOGL, NVDA e financeiras de pagamento.
- **IBOV:** ITSA4, bancos, B3SA3 e ações domésticas sensíveis a juros.
- Nenhuma passa de 0,81 de correlação.
- As redundâncias são classes da mesma empresa (GOOG/GOOGL, PETR3/PETR4, BBDC3/BBDC4) e holding e controlada (ITSA4/ITUB4).
- As candidatas mais "limpas" combinam beta perto de 1 com risco próprio baixo (MSFT, V, AAPL; ITSA4, ITUB4, BBDC3).

**4. Quais setores são mais fáceis de replicar?**
O financeiro, nos dois índices. Os mais difíceis são os defensivos no S&P (saúde, consumo básico, utilidades) e os exportadores e o agronegócio no IBOV.

**5. Qual índice é mais difícil de replicar com poucas ações?**
O IBOV é mais volátil e tem caudas mais gordas. Mas, para replicar com poucas ações, importa o quanto elas andam juntas. Quatro evidências independentes apontam o **S&P recente** como o caso mais difícil:
- a correlação média entre pares caiu para 0,19 desde 2023, contra 0,30 a 0,34 no IBOV;
- são precisos 41 componentes principais para explicar 80% da variância em 2023–2026, contra 32 no IBOV;
- o topo do ranking de correlação muda mais (só 8 de 20 ações permaneceram de 2024 para 2025);
- só uma ação (AVGO) esteve sempre entre as 20 mais correlacionadas.

No IBOV, a dificuldade específica é a assimetria: as ações caem juntas, mas sobem de forma dispersa.

**6. A carteira seria implementável?**
Sim. A ação menos líquida negocia US$ 367 milhões por dia no S&P e R$ 39 milhões no IBOV, então a liquidez não precisa ser restrição no modelo. As 10 maiores ações cobrem 55% do peso do S&P, com número efetivo de 24 ações. No IBOV, a aproximação pelo valor de mercado total superestima empresas com controlador (Petrobras, Ambev, WEG, Itaú), então a concentração brasileira fica sem conclusão robusta.

**Hipóteses levadas ao modelo:**
1. O erro de *tracking* com K pequeno deve ser maior no S&P recente que no IBOV.
2. O turnover deve ser maior no S&P.
3. No IBOV, a carteira deve acompanhar melhor as quedas que as altas.
4. As janelas antigas do IBOV subestimam o erro de um fundo real, por causa do viés de sobrevivência.
5. Janelas de treino mais curtas que 504 pregões devem reduzir o erro, ao custo de mais turnover (testado na seção 8 e no modelo, seção 13.7).

**Limitações:** universo fixo em 30/09/2026; índices em moedas e convenções de retorno diferentes; setores e valor de mercado do Yahoo; pesos dos índices aproximados.

## Apêndice A: auditoria detalhada dos dados

Esta seção reúne as tabelas completas dos diagnósticos e correções. Ela documenta a reprodutibilidade, mas não é necessária para acompanhar a narrativa.

### A.1 UGPA3: `Close` × `Adj Close` em torno de 28/06/2021

In [ ]:
print(f"Cenário detectado: {cenario_ugpa3}")
display(diag_ugpa3.round(4))

**Resultado do diagnóstico (execução de 08/10/2026)**

- **UGPA3: cenário B.** O próprio `Close` salta de R$ 8,75 (25/06/2021) para R$ 19,19 (28/06/2021), e o fator Adj/Close fica constante (0,80). O nível correto é o posterior: a série anterior a 28/06/2021 está cerca de 2,2 vezes abaixo, provavelmente por um ajuste de evento corporativo aplicado indevidamente pelo Yahoo. Os retornos anteriores não são afetados (retorno não depende da escala); só o retorno de 28/06 é falso. A seção 4.1 anula esse retorno e reescala os preços anteriores. Corrigido, o retorno de UGPA3 em 2021 passa de +38% para −37%.
- **BRAP4 em 17/12/2021: provento legítimo.** O fator de ajuste variou 26%, mas o `Close` caiu na data ex e o retorno ajustado ficou normal. É o dividendo de R$ 6,04 por ação da Bradespar (cerca de 21% do preço), com data ex em 17/12/2021. A regra da seção 4.1 mantém esse ajuste.
- **NATU3:** depois de uma lacuna de cerca de 1.370 pregões, a série recomeça com cotações repetidas (26/06 a 01/07/2025) e um "retorno" de −72% em 02/07/2025, que é a emenda entre a Natura antiga e a nova listagem. Mantemos só o trecho a partir de 02/07/2025.

### A.2 Saltos do fator de ajuste e registro de correções

In [ ]:
print("Saltos do fator de ajuste acima de 25% (IBOV):")
display(auditoria_fator_ajuste.round({"variação log do fator": 4, "variação do fator (%)": 2, "retorno Close (%)": 2, "retorno Adj Close (%)": 2}))
print("\nRegistro completo:")
display(log_correcoes)

### A.3 Outliers: listas detalhadas

In [ ]:
colunas_pct = [c for c in outliers.columns if c.endswith("(%)")]
print("Passo 1 – triagem de erros e decisões:")
with pd.option_context("display.max_colwidth", 200):
    display(triagem.round({c: 2 for c in triagem.columns if c.endswith("(%)")}))
print("\nPasso 3 – lista de verificação (30 maiores pela parte própria + amostra aleatória):")
with pd.option_context("display.max_colwidth", 200):
    display(verificacao[["seleção", "indice", "data", "ticker", "retorno (%)", "retorno do índice (%)", "parte própria (%)", "pares",
                         "parte própria mediana dos pares (%)", "confirmação", "evidência"]].round(2))
print("\nOs 20 maiores extremos puxados pelo mercado:")
display(outliers[outliers["origem"] == "Puxado pelo mercado"].assign(_a=lambda t: t["retorno (%)"].abs())
        .nlargest(20, "_a").drop(columns=["_a", "evidência"]).round({c: 2 for c in colunas_pct}).reset_index(drop=True))
print("\nExtremos por ano, índice e origem:")
display(outliers.assign(ano=outliers["data"].dt.year).pivot_table(index="ano", columns=["indice", "origem"], values="ticker",
        aggfunc="size", fill_value=0))

### A.4 Verificação dos números citados no texto

Os textos de conclusão citam números da execução de referência (dados até 30/09/2026). Esta célula recalcula todos eles. Se o Yahoo revisar alguma série e um valor mudar, atualize o texto correspondente.

In [ ]:
def _fmt(v, casas=1):
    return f"{v:.{casas}f}"

checagem = []
for nome in ["S&P100", "IBOV"]:
    b, R = bases[nome], retornos_eda[nome]
    y = b["y"].dropna()
    q = qualidade[qualidade["indice"] == nome]
    c = correlacoes[correlacoes["indice"] == nome].set_index("ticker")["correlação com índice"]
    mensal = (1 + y).resample("ME").prod() - 1
    vol21 = y.rolling(21).std() * np.sqrt(ANUAL) * 100
    nivel = (1 + y).cumprod()
    dd = nivel / nivel.cummax() - 1
    o = outliers[outliers["indice"] == nome]
    reg = correlacao_regimes[correlacao_regimes["indice"] == nome].set_index("regime")
    cons = consistencia_universo[consistencia_universo["indice"] == nome].set_index("ano")
    pca = pca_variancia[(pca_variancia["indice"] == nome)].set_index("período")
    est = estabilidade_ranking[estabilidade_ranking["indice"] == nome]
    asm = assimetria[assimetria["indice"] == nome].set_index("grupo")["correlação mediana com índice"]
    bi = beta_idio[beta_idio["indice"] == nome]
    ilus = ilustracao_sobrevivencia.set_index("indice").loc[nome]
    itens = {
        "ativos no universo": len(q),
        "ações com cobertura ≥ 99%": int((q["cobertura (%)"] >= 99).sum()),
        "séries com cobertura < 80%": int((q["cobertura (%)"] < 80).sum()),
        "ativos no EDA (cobertura ≥ 80%)": R.shape[1],
        "outliers |r|≥15% (após a Etapa 2)": len(o),
        "% outliers puxados pelo mercado": _fmt(100 * (o["origem"] == "Puxado pelo mercado").mean(), 0),
        "% outliers na janela da Covid": _fmt(100 * o["na janela da Covid"].mean(), 0),
        "retorno acumulado (%)": _fmt(100 * (nivel.iloc[-1] - 1)),
        "volatilidade anualizada (%)": _fmt(100 * y.std() * np.sqrt(ANUAL)),
        "Sharpe aproximado": _fmt(y.mean() / y.std() * np.sqrt(ANUAL), 2),
        "curtose": _fmt(y.kurtosis()),
        "pior dia (%)": f"{_fmt(100 * y.min())} em {y.idxmin().date()}",
        "pior mês (%)": f"{_fmt(100 * mensal.min())} em {mensal.idxmin():%m/%Y}",
        "melhor mês (%)": f"{_fmt(100 * mensal.max())} em {mensal.idxmax():%m/%Y}",
        "pico vol 21d (% a.a.)": f"{_fmt(vol21.max(), 0)} em {vol21.idxmax().date()}",
        "vol 21d mediana (% a.a.)": _fmt(vol21.median()),
        "drawdown máximo (%)": f"{_fmt(100 * dd.min())} em {dd.idxmin().date()}",
        "correlação mediana com índice": _fmt(c.median(), 2),
        "% ações com correlação > 0,7": _fmt(100 * (c > 0.7).mean(), 0),
        "% ações com correlação > 0,5": _fmt(100 * (c > 0.5).mean(), 0),
        "maior correlação": f"{c.idxmax()} {_fmt(c.max(), 2)}",
        "menor correlação": f"{c.idxmin()} {_fmt(c.min(), 2)}",
        "vol mediana das ações (% a.a.)": _fmt(100 * R.std().median() * np.sqrt(ANUAL)),
        "R² mínimo universo × índice": _fmt(cons["R²"].min(), 3),
        "TE residual 2020 / 2023 / 2025 / 2026 (%)": " / ".join(_fmt(cons.loc[a, "TE residual (% a.a.)"], 2) for a in [2020, 2023, 2025, 2026]),
        "pesos iguais × índice (acum. %)": f"{_fmt(ilus.iloc[1], 0)} × {_fmt(ilus.iloc[0], 0)}",
        "corr. pares Covid / 2023–24 / 2025–26": " / ".join(_fmt(reg.loc[r, "correlação média entre pares"], 2) for r in ["Crash Covid (fev–abr/20)", "2023–2024", "2025–set/2026"]),
        "PC1 Covid / 2023–24 / 2025–26 (%)": " / ".join(_fmt(reg.loc[r, "PC1 (% da variância)"], 0) for r in ["Crash Covid (fev–abr/20)", "2023–2024", "2025–set/2026"]),
        "PCs p/ 50% / 80% (2019–26)": f"{pca.loc['2019–2026', 'componentes para 50%']} / {pca.loc['2019–2026', 'componentes para 80%']}",
        "PCs p/ 50% / 80% (2023–26)": f"{pca.loc['2023–2026', 'componentes para 50%']} / {pca.loc['2023–2026', 'componentes para 80%']}",
        "variância nos 10 primeiros PCs (2019–26, %)": _fmt(pca.loc["2019–2026", "variância nos 10 primeiros (%)"], 0),
        "top 20 que permanecem (por transição)": " ".join(str(v) for v in est["permanecem no top 20"]),
        "Spearman do ranking (mín–máx)": f"{_fmt(est['Spearman do ranking'].min(), 2)}–{_fmt(est['Spearman do ranking'].max(), 2)}",
        "núcleo estável no top 20": ", ".join(t.replace(".SA", "") for t in nucleo_estavel[nome]) or "nenhum",
        "corr. piores / centrais / melhores dias": " / ".join(_fmt(asm[g], 2) for g in ["10% piores dias", "80% centrais", "10% melhores dias"]),
        "beta mediano [p10–p90]": f"{_fmt(bi['beta'].median(), 2)} [{_fmt(bi['beta'].quantile(0.1), 2)}–{_fmt(bi['beta'].quantile(0.9), 2)}]",
        "vol idiossincrática mediana (% a.a.)": _fmt(bi["vol idiossincrática (% a.a.)"].median()),
        "liquidez: mediana / mínimo (milhões)": " / ".join(_fmt(v, 0) for v in [liquidez[liquidez["indice"] == nome][col_vol].median(), liquidez[liquidez["indice"] == nome][col_vol].min()]),
        "concentração: 10 maiores (%) / nº efetivo": (" / ".join([_fmt(concentracao.set_index("indice").loc[nome, "peso das 10 maiores (%)"], 0), _fmt(concentracao.set_index("indice").loc[nome, "número efetivo de ações"], 0)]) if nome in set(concentracao["indice"]) else "n/d"),
    }
    for chave, valor in itens.items():
        checagem.append({"indicador": chave, nome: valor})
checagem = pd.DataFrame(checagem).groupby("indicador", sort=False).first()
print(f"Correlação diária S&P100 × IBOV (dias comuns): {indices_alinhados.corr().iloc[0, 1]:.2f}")
display(checagem)

## Apêndice A: auditoria detalhada dos dados

Esta seção reúne as tabelas completas dos diagnósticos e correções. Ela documenta a reprodutibilidade, mas não é necessária para acompanhar a narrativa.

### A.1 UGPA3: `Close` × `Adj Close` em torno de 28/06/2021

In [ ]:
print(f"Cenário detectado: {cenario_ugpa3}")
display(diag_ugpa3.round(4))

**Resultado do diagnóstico (execução de 08/10/2026)**

- **UGPA3: cenário B.** O próprio `Close` salta de R$ 8,75 (25/06/2021) para R$ 19,19 (28/06/2021), e o fator Adj/Close fica constante (0,80). O nível correto é o posterior: a série anterior a 28/06/2021 está cerca de 2,2 vezes abaixo, provavelmente por um ajuste de evento corporativo aplicado indevidamente pelo Yahoo. Os retornos anteriores não são afetados (retorno não depende da escala); só o retorno de 28/06 é falso. A seção 4.1 anula esse retorno e reescala os preços anteriores. Corrigido, o retorno de UGPA3 em 2021 passa de +38% para −37%.
- **BRAP4 em 17/12/2021: provento legítimo.** O fator de ajuste variou 26%, mas o `Close` caiu na data ex e o retorno ajustado ficou normal. É o dividendo de R$ 6,04 por ação da Bradespar (cerca de 21% do preço), com data ex em 17/12/2021. A regra da seção 4.1 mantém esse ajuste.
- **NATU3:** depois de uma lacuna de cerca de 1.370 pregões, a série recomeça com cotações repetidas (26/06 a 01/07/2025) e um "retorno" de −72% em 02/07/2025, que é a emenda entre a Natura antiga e a nova listagem. Mantemos só o trecho a partir de 02/07/2025.

### A.2 Saltos do fator de ajuste e registro de correções

In [ ]:
print("Saltos do fator de ajuste acima de 25% (IBOV):")
display(auditoria_fator_ajuste.round({"variação log do fator": 4, "variação do fator (%)": 2, "retorno Close (%)": 2, "retorno Adj Close (%)": 2}))
print("\nRegistro completo:")
display(log_correcoes)

### A.3 Outliers: listas detalhadas

In [ ]:
colunas_pct = [c for c in outliers.columns if c.endswith("(%)")]
print("Passo 1 – triagem de erros e decisões:")
with pd.option_context("display.max_colwidth", 200):
    display(triagem.round({c: 2 for c in triagem.columns if c.endswith("(%)")}))
print("\nPasso 3 – lista de verificação (30 maiores pela parte própria + amostra aleatória):")
with pd.option_context("display.max_colwidth", 200):
    display(verificacao[["seleção", "indice", "data", "ticker", "retorno (%)", "retorno do índice (%)", "parte própria (%)", "pares",
                         "parte própria mediana dos pares (%)", "confirmação", "evidência"]].round(2))
print("\nOs 20 maiores extremos puxados pelo mercado:")
display(outliers[outliers["origem"] == "Puxado pelo mercado"].assign(_a=lambda t: t["retorno (%)"].abs())
        .nlargest(20, "_a").drop(columns=["_a", "evidência"]).round({c: 2 for c in colunas_pct}).reset_index(drop=True))
print("\nExtremos por ano, índice e origem:")
display(outliers.assign(ano=outliers["data"].dt.year).pivot_table(index="ano", columns=["indice", "origem"], values="ticker",
        aggfunc="size", fill_value=0))

### A.4 Verificação dos números citados no texto

Os textos de conclusão citam números da execução de referência (dados até 30/09/2026). Esta célula recalcula todos eles. Se o Yahoo revisar alguma série e um valor mudar, atualize o texto correspondente.

In [ ]:
def _fmt(v, casas=1):
    return f"{v:.{casas}f}"

checagem = []
for nome in ["S&P100", "IBOV"]:
    b, R = bases[nome], retornos_eda[nome]
    y = b["y"].dropna()
    q = qualidade[qualidade["indice"] == nome]
    c = correlacoes[correlacoes["indice"] == nome].set_index("ticker")["correlação com índice"]
    mensal = (1 + y).resample("ME").prod() - 1
    vol21 = y.rolling(21).std() * np.sqrt(ANUAL) * 100
    nivel = (1 + y).cumprod()
    dd = nivel / nivel.cummax() - 1
    o = outliers[outliers["indice"] == nome]
    reg = correlacao_regimes[correlacao_regimes["indice"] == nome].set_index("regime")
    cons = consistencia_universo[consistencia_universo["indice"] == nome].set_index("ano")
    pca = pca_variancia[(pca_variancia["indice"] == nome)].set_index("período")
    est = estabilidade_ranking[estabilidade_ranking["indice"] == nome]
    asm = assimetria[assimetria["indice"] == nome].set_index("grupo")["correlação mediana com índice"]
    bi = beta_idio[beta_idio["indice"] == nome]
    ilus = ilustracao_sobrevivencia.set_index("indice").loc[nome]
    itens = {
        "ativos no universo": len(q),
        "ações com cobertura ≥ 99%": int((q["cobertura (%)"] >= 99).sum()),
        "séries com cobertura < 80%": int((q["cobertura (%)"] < 80).sum()),
        "ativos no EDA (todos)": R.shape[1],
        "outliers |r|≥15% (após a Etapa 2)": len(o),
        "% outliers puxados pelo mercado": _fmt(100 * (o["origem"] == "Puxado pelo mercado").mean(), 0),
        "% outliers na janela da Covid": _fmt(100 * o["na janela da Covid"].mean(), 0),
        "retorno acumulado (%)": _fmt(100 * (nivel.iloc[-1] - 1)),
        "volatilidade anualizada (%)": _fmt(100 * y.std() * np.sqrt(ANUAL)),
        "Sharpe aproximado": _fmt(y.mean() / y.std() * np.sqrt(ANUAL), 2),
        "curtose": _fmt(y.kurtosis()),
        "pior dia (%)": f"{_fmt(100 * y.min())} em {y.idxmin().date()}",
        "pior mês (%)": f"{_fmt(100 * mensal.min())} em {mensal.idxmin():%m/%Y}",
        "melhor mês (%)": f"{_fmt(100 * mensal.max())} em {mensal.idxmax():%m/%Y}",
        "pico vol 21d (% a.a.)": f"{_fmt(vol21.max(), 0)} em {vol21.idxmax().date()}",
        "vol 21d mediana (% a.a.)": _fmt(vol21.median()),
        "drawdown máximo (%)": f"{_fmt(100 * dd.min())} em {dd.idxmin().date()}",
        "correlação mediana com índice": _fmt(c.median(), 2),
        "% ações com correlação > 0,7": _fmt(100 * (c > 0.7).mean(), 0),
        "% ações com correlação > 0,5": _fmt(100 * (c > 0.5).mean(), 0),
        "maior correlação": f"{c.idxmax()} {_fmt(c.max(), 2)}",
        "menor correlação": f"{c.idxmin()} {_fmt(c.min(), 2)}",
        "vol mediana das ações (% a.a.)": _fmt(100 * R.std().median() * np.sqrt(ANUAL)),
        "TE residual universo 2020 / 2023 / 2026 (%)": " / ".join(_fmt(cons.loc[a, "TE residual (% a.a.)"], 2) for a in [2020, 2023, 2026]),
        "R² mínimo universo × índice": _fmt(cons["R²"].min(), 3),
        "TE residual compra-e-mantém 2023 / 2025 / 2026 (%)": " / ".join(_fmt(cons.loc[a, "TE residual compra-e-mantém (% a.a.)"], 2) for a in [2023, 2025, 2026]),
        "pesos iguais × índice (acum. %)": f"{_fmt(ilus.iloc[1], 0)} × {_fmt(ilus.iloc[0], 0)}",
        "corr. pares Covid / 2023–24 / 2025–26": " / ".join(_fmt(reg.loc[r, "correlação média entre pares"], 2) for r in ["Crash Covid (fev–abr/20)", "2023–2024", "2025–set/2026"]),
        "PC1 Covid / 2023–24 / 2025–26 (%)": " / ".join(_fmt(reg.loc[r, "PC1 (% da variância)"], 0) for r in ["Crash Covid (fev–abr/20)", "2023–2024", "2025–set/2026"]),
        "PCs p/ 50% / 80% (2019–26)": f"{pca.loc['2019–2026', 'componentes para 50%']} / {pca.loc['2019–2026', 'componentes para 80%']}",
        "PCs p/ 50% / 80% (2023–26)": f"{pca.loc['2023–2026', 'componentes para 50%']} / {pca.loc['2023–2026', 'componentes para 80%']}",
        "variância nos 10 primeiros PCs (2019–26, %)": _fmt(pca.loc["2019–2026", "variância nos 10 primeiros (%)"], 0),
        "top 20 que permanecem (por transição)": " ".join(str(v) for v in est["permanecem no top 20"]),
        "Spearman do ranking (mín–máx)": f"{_fmt(est['Spearman do ranking'].min(), 2)}–{_fmt(est['Spearman do ranking'].max(), 2)}",
        "núcleo estável no top 20": ", ".join(t.replace(".SA", "") for t in nucleo_estavel[nome]) or "nenhum",
        "corr. piores / centrais / melhores dias": " / ".join(_fmt(asm[g], 2) for g in ["10% piores dias", "80% centrais", "10% melhores dias"]),
        "beta mediano [p10–p90]": f"{_fmt(bi['beta'].median(), 2)} [{_fmt(bi['beta'].quantile(0.1), 2)}–{_fmt(bi['beta'].quantile(0.9), 2)}]",
        "vol idiossincrática mediana (% a.a.)": _fmt(bi["vol idiossincrática (% a.a.)"].median()),
        "liquidez: mediana / mínimo (milhões)": " / ".join(_fmt(v, 0) for v in [liquidez[liquidez["indice"] == nome][col_vol].median(), liquidez[liquidez["indice"] == nome][col_vol].min()]),
        "concentração: 10 maiores (%) / nº efetivo": (" / ".join([_fmt(concentracao.set_index("indice").loc[nome, "peso das 10 maiores (%)"], 0), _fmt(concentracao.set_index("indice").loc[nome, "número efetivo de ações"], 0)]) if nome in set(concentracao["indice"]) else "n/d"),
    }
    for chave, valor in itens.items():
        checagem.append({"indicador": chave, nome: valor})
checagem = pd.DataFrame(checagem).groupby("indicador", sort=False).first()
print(f"Correlação diária S&P100 × IBOV (dias comuns): {indices_alinhados.corr().iloc[0, 1]:.2f}")
display(checagem)

## 8. Da EDA ao modelo: o tamanho da janela de treino

A seção 7.10 mostrou que os regimes mudam: a correlação média entre ações do S&P foi de 0,43 em 2022 para 0,19 em 2023–2026. O modelo usa `JANELA_TREINO = 504` pregões (~2 anos). Assim, uma carteira montada no início de 2024 ainda "aprende" com 2022, um mercado que já não existia.

Antes de rodar o MIQP, testamos o efeito da janela de forma barata, **sem restrição de cardinalidade**: para cada trimestre, estimamos a carteira de pesos não negativos (soma 1) que melhor reproduz o índice na janela de treino e a mantemos por 63 pregões em compra-e-mantém, exatamente como o walk-forward do modelo. Comparamos janelas de 126, 252, 504 e 756 pregões **nos mesmos trimestres de teste**, com a mesma regra de elegibilidade (98% de cobertura e dados nos 5 últimos pregões).

> Este teste não substitui o do modelo: com K pequeno, há menos pesos a estimar, e janelas mais longas podem ajudar a estabilizar a escolha das ações. Ele serve para decidir **quais janelas vale a pena testar** no MIQP.

In [ ]:
def retorno_buy_and_hold_eda(R_teste, pesos):
    # Mesma convenção do walk-forward do modelo (seção 11): pesos derivam com os preços.
    R = R_teste.reindex(columns=pesos.index).fillna(0.0)
    valores = (1 + R).cumprod().mul(pesos, axis=1)
    patrimonio = valores.sum(axis=1)
    retorno = patrimonio.pct_change(fill_method=None)
    retorno.iloc[0] = patrimonio.iloc[0] - 1.0
    return retorno, valores.iloc[-1] / valores.iloc[-1].sum(), 0

In [ ]:
JANELAS_TESTE = [126, 252, 504, 756]
H = HORIZONTE_TESTE
linhas = []
for nome, b in bases.items():
    R, y = b["R"], b["y"]
    inicios = list(range(max(JANELAS_TESTE), len(y) - H + 1, H))
    for J in JANELAS_TESTE:
        pesos_ant = None
        for s in inicios:
            d_tr, d_te = y.index[s - J:s], y.index[s:s + H]
            Rt = R.reindex(d_tr)
            cob = Rt.notna().mean()
            cand = cob[(cob >= COBERTURA_MINIMA_TREINO) & Rt.tail(5).notna().all()].index
            yt = y.reindex(d_tr)
            ok = yt.notna()
            w, _ = ajuste_universo(Rt.loc[ok, cand].fillna(0.0), yt[ok])
            p = pd.Series(w, index=cand)
            p = p[p > LIMIAR_PESO]
            p = p / p.sum()
            rp, pesos_finais, _ = retorno_buy_and_hold_eda(R.reindex(d_te), p)
            erro = (rp - y.reindex(d_te)).dropna()
            linhas.append({
                "indice": nome, "janela (pregões)": J, "início do teste": d_te[0],
                "TE fora da amostra (% a.a.)": 100 * erro.std() * np.sqrt(ANUAL),
                "TD fora da amostra (p.p. a.a.)": 100 * erro.mean() * ANUAL,
                "turnover": np.nan if pesos_ant is None else 0.5 * (p.reindex(p.index.union(pesos_ant.index), fill_value=0)
                                                                   - pesos_ant.reindex(p.index.union(pesos_ant.index), fill_value=0)).abs().sum(),
            })
            pesos_ant = pesos_finais
sensibilidade_janela = pd.DataFrame(linhas)

resumo_janela = sensibilidade_janela.groupby(["indice", "janela (pregões)"]).agg(
    trimestres=("TE fora da amostra (% a.a.)", "size"),
    **{"TE médio (% a.a.)": ("TE fora da amostra (% a.a.)", "mean"),
       "TE mediano (% a.a.)": ("TE fora da amostra (% a.a.)", "median"),
       "TE máximo (% a.a.)": ("TE fora da amostra (% a.a.)", "max"),
       "TD média (p.p. a.a.)": ("TD fora da amostra (p.p. a.a.)", "mean"),
       "turnover médio": ("turnover", "mean")})
vitorias = (sensibilidade_janela.pivot_table(index=["indice", "início do teste"], columns="janela (pregões)", values="TE fora da amostra (% a.a.)")
            .idxmin(axis=1).groupby(level=0).value_counts().rename("trimestres com menor TE"))
resumo_janela = resumo_janela.join(vitorias.rename_axis(["indice", "janela (pregões)"])).fillna({"trimestres com menor TE": 0})
display(resumo_janela.round(2))

# Gráfico: o dilema erro x giro (dispersão conectada)
nomes_janela = {126: "6 meses", 252: "1 ano", 504: "2 anos", 756: "3 anos"}
fig, ax = plt.subplots(figsize=(11.5, 6.2))
for nome in ["S&P100", "IBOV"]:
    g = resumo_janela.loc[nome].reset_index()
    xs, ys = 100 * g["turnover médio"], g["TE médio (% a.a.)"]
    ax.plot(xs, ys, color=CORES[nome], lw=1.6, zorder=2, label=NOMES[nome])
    for _, l in g.iterrows():
        atual = l["janela (pregões)"] == JANELA_TREINO
        ax.plot([100 * l["turnover médio"]], [l["TE médio (% a.a.)"]], "o", ms=11 if atual else 8, color=CORES[nome],
                mec=TINTA if atual else FUNDO, mew=1.8, zorder=4)
        ax.annotate(nomes_janela[int(l["janela (pregões)"])] + (" (atual)" if atual else ""), (100 * l["turnover médio"], l["TE médio (% a.a.)"]),
                    xytext=(8, 7 if nome == "S&P100" else -15), textcoords="offset points", fontsize=10, color=TINTA)
eixo_pct(ax, "x"); eixo_pct(ax, "y", casas=1)
ax.set_xlabel("Giro médio por trimestre (fração da carteira trocada)")
ax.set_ylabel("Erro fora da amostra (TE, % ao ano)")
ax.grid(axis="x", color=GRADE)
legenda(ax, loc="upper right")
nota(ax, 100 * resumo_janela.loc[("S&P100", 126), "turnover médio"], resumo_janela.loc[("S&P100", 126), "TE médio (% a.a.)"],
     "menos erro,\nmais giro", -70, 25, ha="right")
titulos(fig, "Mais histórico deixa a carteira mais estável, mas erra mais: o dilema da janela de treino",
        f"Carteira sem limite de ações estimada com janelas de 6 meses a 3 anos e mantida por um trimestre "
        f"({int(resumo_janela['trimestres'].max())} trimestres de teste).\nCada ponto é um tamanho de janela; contorno escuro: a janela usada no modelo.")
finalizar_figura("21_sensibilidade_janela_treino.png", fig)

**O que o teste da janela mostra**

O teste usa a carteira sem limite de ações, para isolar o efeito do tamanho da janela. TE e TD fora da amostra, médias dos 15 trimestres:

| Janela de treino | S&P: TE médio | S&P: TD média | S&P: giro médio | IBOV: TE médio | IBOV: TD média | IBOV: giro médio |
|---|---|---|---|---|---|---|
| 126 pregões (6 meses) | **1,59%** | +1,4 p.p. | 27% | 1,96% | +3,8 p.p. | 25% |
| 252 pregões (1 ano) | 1,77% | +1,7 p.p. | 19% | **1,82%** | +2,5 p.p. | 16% |
| 504 pregões (2 anos) | 2,12% | +1,9 p.p. | 12% | 2,01% | +1,9 p.p. | 11% |
| 756 pregões (3 anos) | 2,43% | +2,2 p.p. | 11% | 2,27% | +2,0 p.p. | 9% |

- **Há uma troca entre erro e giro:** janelas curtas acompanham melhor o mercado atual; janelas longas são mais estáveis, mas carregam um mercado que já passou.
- **No S&P, quanto mais curta a janela, menor o TE** (6 meses teve o menor TE em 9 dos 15 trimestres), coerente com a queda da correlação entre as ações desde 2023. **No IBOV, 1 ano é o melhor equilíbrio** (menor TE em 7 trimestres).
- **A TD é positiva em todas as janelas** (de +1,4 a +3,8 p.p. por ano): é o viés de sobrevivência da seção 6.3, presente qualquer que seja a janela.
- **Decisão:** 504 pregões seguem como referência do modelo, pelo giro menor; 252 pregões entram como alternativa (seção 13.7).

## 9. O problema matemático

Em cada data de rebalanceamento, sejam $r_{t,i}$ o retorno da ação $i$ no dia $t$ da janela de treino, $R_t$ o retorno do índice, $w_i$ o peso da ação e $z_i$ uma variável binária que indica se a ação entra na carteira. Para uma quantidade máxima de ações $K$, resolvemos:

$$
\min_{w,z}\ \frac{1}{T}\sum_{t=1}^{T}\left(R_t-\sum_{i=1}^{N}w_i r_{t,i}\right)^2
$$

sujeito a:

$$
\sum_i w_i=1,\qquad 0\leq w_i\leq z_i,\qquad
\sum_i z_i\leq K,\qquad z_i\in\{0,1\}.
$$

- **Objetivo:** o erro quadrático médio entre a carteira e o índice na janela de treino.
- **Restrições:** carteira totalmente investida, sem venda a descoberto (*long-only*) e com no máximo `K` ações. Se $z_i = 0$, o peso $w_i$ é forçado a zero. `K` é um teto: o modelo pode usar menos ações.
- **Sem limite de ações:** sem as variáveis $z_i$, o problema vira um QP convexo, resolvido em milissegundos. Ele dá o menor erro possível dentro da amostra e serve de piso para as carteiras com `K` ações.

Giro, concentração e tempo de cálculo não entram no objetivo: são medidos depois, para não misturar critérios com pesos arbitrários.

## 10. Como resolvemos: heurística + MIQP com partida a quente

O problema é combinatório. Com 95 candidatas e `K = 10`, há cerca de 10 trilhões de combinações possíveis de ações. Usamos duas etapas, na linha do método híbrido de Sant'Anna et al. (2017):

1. **Heurística (segundos por janela):**
   - *seleção gulosa:* começa com a carteira vazia e, a cada passo, adiciona a ação que mais reduz o erro de treino, reotimizando os pesos;
   - *busca local por trocas:* testa trocar cada ação da carteira por cada ação de fora e aceita a melhor troca, até nenhuma troca reduzir o erro.
2. **MIQP no Gurobi (até `time_limit_s` segundos por carteira):** parte da carteira da heurística (*warm start*) e procura carteiras melhores. Ao final, o Gurobi informa o melhor **limite inferior** que conseguiu provar. A diferença relativa entre a solução e esse limite é o **gap**: gap zero é prova de que a carteira é a melhor possível.

Duas referências ajudam a ler os resultados:
- **Sem limite de ações:** o menor erro possível com as mesmas candidatas.
- **Top-K por correlação:** escolhe as `K` ações mais correlacionadas com o índice no treino e só depois otimiza os pesos. É o atalho "intuitivo" que o EDA sugeriu evitar (seções 7.5 e 7.6).

A formulação usa variáveis binárias explícitas (cerca de 200 variáveis no S&P 100) e cabe na licença gratuita do `gurobipy`.

In [ ]:
AMBIENTE_GUROBI = None   # criado só quando o MIQP é resolvido (RECALCULAR_MODELO = True)


def ambiente_gurobi():
    global AMBIENTE_GUROBI
    if AMBIENTE_GUROBI is None:
        AMBIENTE_GUROBI = gp.Env(empty=True)
        AMBIENTE_GUROBI.setParam("OutputFlag", 0)
        AMBIENTE_GUROBI.start()
    return AMBIENTE_GUROBI

PENALIDADE_SOMA = 1e3   # impõe soma dos pesos = 1 nos QPs pequenos da heurística


class ProblemaTracking:
    # Matrizes de uma janela de treino, com retornos em % ao dia.
    def __init__(self, R_treino, y_treino):
        X = 100.0 * R_treino.to_numpy(dtype=float)
        a = 100.0 * y_treino.to_numpy(dtype=float)
        self.n, self.T = X.shape[1], len(a)
        self.X, self.a = X, a
        self.Q = X.T @ X / self.T          # erro médio = w'Qw - 2c'w + k0
        self.c = X.T @ a / self.T
        self.k0 = a @ a / self.T
        self.G = X.T @ X + PENALIDADE_SOMA ** 2   # forma de mínimos quadrados com a linha de penalização
        self.h = X.T @ a + PENALIDADE_SOMA ** 2

    def erro(self, idx, w):
        Qs = self.Q[np.ix_(idx, idx)]
        return max(float(w @ Qs @ w - 2 * self.c[idx] @ w + self.k0), 0.0)

    def qp(self, idx):
        # min ||X_S w - a||² com w >= 0 e soma(w) = 1, resolvido no espaço k x k.
        # Se a solução sem a restrição de sinal já for não negativa, ela é a ótima; senão, usa NNLS.
        idx = list(idx)
        Gs, hs = self.G[np.ix_(idx, idx)], self.h[idx]
        try:
            w = np.linalg.solve(Gs, hs)
            if w.min() < 0:
                L = np.linalg.cholesky(Gs)
                w, _ = nnls(L.T, solve_triangular(L, hs, lower=True))
        except np.linalg.LinAlgError:
            A = np.vstack([self.X[:, idx], PENALIDADE_SOMA * np.ones(len(idx))])
            w, _ = nnls(A, np.r_[self.a, PENALIDADE_SOMA])
        w = w / w.sum()
        Qs = (Gs - PENALIDADE_SOMA ** 2) / self.T
        return w, max(float(w @ Qs @ w - 2 * self.c[idx] @ w + self.k0), 0.0)


def selecao_gulosa(prob, kmax):
    # Adiciona, a cada passo, a ação que mais reduz o erro de treino. Guarda o caminho e o tempo acumulado.
    selecionadas, restantes, caminho = [], list(range(prob.n)), {}
    inicio = time.perf_counter()
    for k in range(1, min(kmax, prob.n) + 1):
        melhor_erro, melhor_j = np.inf, None
        for j in restantes:
            _, erro = prob.qp(selecionadas + [j])
            if erro < melhor_erro:
                melhor_erro, melhor_j = erro, j
        selecionadas.append(melhor_j)
        restantes.remove(melhor_j)
        caminho[k] = (list(selecionadas), time.perf_counter() - inicio)
    return caminho


def busca_trocas(prob, selecionadas, max_passos=50):
    # Melhor troca 1 por 1 (dentro x fora) até nenhuma troca reduzir o erro.
    sel = list(selecionadas)
    _, erro = prob.qp(sel)
    for _ in range(max_passos):
        melhor_erro, melhor_sel = erro, None
        fora = [j for j in range(prob.n) if j not in sel]
        for i in range(len(sel)):
            for j in fora:
                candidata = sel[:i] + [j] + sel[i + 1:]
                _, e = prob.qp(candidata)
                if e < melhor_erro - 1e-12:
                    melhor_erro, melhor_sel = e, candidata
        if melhor_sel is None:
            break
        erro, sel = melhor_erro, melhor_sel
    w, erro = prob.qp(sel)
    return sel, w, erro


def nome_status(status):
    mapa = {GRB.OPTIMAL: "ótimo provado", GRB.TIME_LIMIT: "limite de tempo", GRB.SUBOPTIMAL: "subótimo",
            GRB.INTERRUPTED: "interrompido", GRB.INFEASIBLE: "inviável", GRB.NUMERIC: "erro numérico"}
    return mapa.get(status, str(status))


def resolver_miqp(prob, K, w_inicial=None, tempo_max=10.0):
    # MIQP com binárias explícitas e partida a quente opcional.
    m = gp.Model(env=ambiente_gurobi())
    m.Params.TimeLimit = float(tempo_max)
    m.Params.MIPGap = MIP_GAP
    m.Params.Threads = THREADS_GUROBI
    m.Params.Seed = SEED
    w = m.addMVar(prob.n, lb=0.0, ub=1.0, name="w")
    z = m.addMVar(prob.n, vtype=GRB.BINARY, name="z")
    m.addConstr(w.sum() == 1.0, name="investimento_total")
    m.addConstr(w <= z, name="liga_peso_e_selecao")
    m.addConstr(z.sum() <= K, name="cardinalidade")
    m.setObjective(w @ prob.Q @ w - 2 * prob.c @ w + prob.k0, GRB.MINIMIZE)
    if w_inicial is not None:
        w0 = np.where(w_inicial > LIMIAR_PESO, w_inicial, 0.0)
        w0 = w0 / w0.sum()
        w.Start = w0
        z.Start = (w0 > 0).astype(float)
    m.optimize()
    info = {"status": nome_status(m.Status), "tempo solver (s)": m.Runtime, "nós": m.NodeCount,
            "limite inferior": np.nan, "gap (%)": np.nan}
    if m.SolCount == 0:
        return None, info
    info["limite inferior"] = m.ObjBound
    info["gap (%)"] = 100 * max(m.ObjVal - m.ObjBound, 0.0) / max(m.ObjVal, 1e-12)
    return np.asarray(w.X), info


# Teste rápido: problema pequeno em que a força bruta encontra o ótimo.
rng = np.random.default_rng(SEED)
fatores = rng.normal(0, 1, (300, 2))
R_teste = pd.DataFrame(fatores @ rng.normal(0.8, 0.3, (2, 8)) + rng.normal(0, 0.6, (300, 8))) / 100
y_teste = R_teste.iloc[:, :3].mean(axis=1) + rng.normal(0, 0.001, 300)
p_teste = ProblemaTracking(R_teste, y_teste)
forca_bruta = min(p_teste.qp(list(par))[1] for par in combinations(range(8), 2))
heur = busca_trocas(p_teste, selecao_gulosa(p_teste, 2)[2][0])
w_m, info_m = resolver_miqp(p_teste, 2, None, 10)
erro_m = p_teste.erro(list(np.where(w_m > LIMIAR_PESO)[0]), w_m[w_m > LIMIAR_PESO] / w_m[w_m > LIMIAR_PESO].sum())
print(f"Teste com 8 ações e K=2: força bruta {forca_bruta:.5f} | heurística {heur[2]:.5f} | MIQP {erro_m:.5f} ({info_m['status']})")
assert erro_m <= forca_bruta * (1 + MIP_GAP) + 1e-9, "MIQP não reproduziu a força bruta"
print("Solver e heurística prontos.")

## 11. Desenho do experimento (walk-forward)

A cada trimestre, repetimos o ciclo **treino → decisão → teste**:

1. usamos os pregões anteriores ao rebalanceamento: 504 (cerca de 2 anos) na configuração de referência e 252 (cerca de 1 ano) na comparação sugerida pela seção 8;
2. uma ação é candidata se tiver pelo menos 98% de dados no treino e dados nos 5 últimos pregões;
3. **ações que ainda não estavam no índice ficam de fora.** DELL, PANW, ANET e SNDK entraram no S&P 100 em 21/09/2026 (seção 6.3) e não são candidatas antes dessa data;
4. a lista de candidatas é a mesma para todos os métodos e valores de `K` na mesma janela;
5. a carteira é mantida por 63 pregões em compra-e-mantém (os pesos variam com os preços) e é recalculada no trimestre seguinte. Todas as configurações usam **os mesmos trimestres de teste**; o último vai até 30/09/2026 e é um pouco mais curto.

| Configuração | Treino (pregões) | K | Papel na análise |
|---|---|---|---|
| MIQP no Gurobi, com partida a quente | 504 | 5, 10, 15, 20, 30 | modelo principal |
| Heurística (gulosa + trocas) | 504 e 252 | 5, 10, 15, 20, 30 | fronteira completa e teste da janela |
| Sem limite de ações | 504 e 252 | todas as candidatas | piso de erro |
| Top-K por correlação | 504 e 252 | 5, 10, 15, 20, 30 | atalho ingênuo |

**Métricas fora da amostra** (calculadas só no trimestre seguinte, que o modelo não viu):
- **TE (tracking error), medida padrão de oscilação:** desvio-padrão anualizado da diferença diária entre carteira e índice (× √252). Mede o quanto a carteira oscila em torno do índice. A razão entre o TE fora e o TE dentro da amostra mede o quanto o erro piora quando a carteira enfrenta dados novos.
- **Tracking difference (TD), medida padrão de direção:** diferença média anualizada (× 252), em pontos percentuais. Positivo significa que a carteira rendeu mais que o índice. O TE não vê esse desvio constante; por isso as duas medidas andam juntas em todas as tabelas.
- **RMSE, o objetivo do modelo:** raiz da média das diferenças diárias ao quadrado, a fórmula do guia. É o que a otimização minimiza no treino e combina as duas medidas: RMSE² = TE² + viés²/252. Aparece nas tabelas como referência; nos nossos resultados, fica a centésimos do TE.
- **Giro:** fração da carteira trocada no rebalanceamento, comparando os pesos novos com os pesos que derivaram durante o trimestre.
- **Tempo de cálculo e gap** do MIQP.

In [ ]:
ENTRADAS_NO_INDICE = {
    (linha["indice"], t.strip()): pd.Timestamp(linha["data efetiva"])
    for _, linha in MUDANCAS_COMPOSICAO[MUDANCAS_COMPOSICAO["movimento"] == "entra"].iterrows()
    for t in linha["tickers"].split(",")
}
DIAS_MINIMOS_TESTE = 21


def construir_janelas(nome, J):
    b = bases[nome]
    y, R = b["y"], b["R"]
    # Mesmos trimestres de teste para todas as janelas de treino; o último pode ser mais curto.
    inicios = [s for s in range(JANELA_TREINO, len(y), HORIZONTE_TESTE) if len(y) - s >= DIAS_MINIMOS_TESTE]
    if CFG["n_janelas"]:
        inicios = inicios[-int(CFG["n_janelas"]):]
    janelas = []
    for numero, s in enumerate(inicios, 1):
        datas_treino = y.index[s - J:s]
        datas_teste = y.index[s:s + HORIZONTE_TESTE]
        Rt = R.reindex(datas_treino)
        cobertura = Rt.notna().mean()
        elegiveis = cobertura[(cobertura >= COBERTURA_MINIMA_TREINO) & Rt.tail(5).notna().all()].index.tolist()
        fora_do_indice = [t for t in elegiveis if datas_teste[0] < ENTRADAS_NO_INDICE.get((nome, t), pd.Timestamp.min)]
        janelas.append({
            "indice": nome, "janela treino": J, "numero": numero,
            "rebalanceamento": datas_teste[0], "fim_teste": datas_teste[-1],
            "datas_treino": datas_treino, "datas_teste": datas_teste,
            "candidatos": [t for t in elegiveis if t not in fora_do_indice],
            "excluídas por cobertura": R.shape[1] - len(elegiveis),
            "excluídas por entrada futura no índice": ", ".join(fora_do_indice),
        })
    return janelas


auditoria_janelas = pd.DataFrame([
    {k: v for k, v in j.items() if k not in ("datas_treino", "datas_teste", "candidatos")}
    | {"dias de teste": len(j["datas_teste"]), "candidatas": len(j["candidatos"]),
       "tickers candidatos": ";".join(j["candidatos"])}
    for nome in bases for J in (JANELA_TREINO, JANELA_ALTERNATIVA) for j in construir_janelas(nome, J)
])
if len(auditoria_janelas) == 0:
    raise ValueError("Não há janelas suficientes para o treino e o teste configurados.")
print("Trimestres de teste e candidatas por janela de treino:")
display(auditoria_janelas.groupby(["indice", "janela treino"]).agg(
    trimestres=("numero", "size"),
    primeiro_rebalanceamento=("rebalanceamento", "min"),
    último_fim_de_teste=("fim_teste", "max"),
    candidatas_mín=("candidatas", "min"),
    candidatas_máx=("candidatas", "max"),
))
excluidas = auditoria_janelas[auditoria_janelas["excluídas por entrada futura no índice"] != ""]
if len(excluidas):
    print("Ações com dados suficientes, mas excluídas por ainda não estarem no índice:",
          sorted(set(", ".join(excluidas["excluídas por entrada futura no índice"]).split(", "))))

## 12. Executar o experimento

Esta é a célula mais demorada. O tempo é dominado pelo MIQP, que roda até o limite de tempo em quase todas as carteiras, porque o gap raramente chega a 1%:

$$2\ \text{índices} \times 20\ \text{trimestres} \times 3\ \text{valores de K} \times 10\ \text{s} \approx 20\ \text{minutos},$$

mais 3 a 5 minutos de heurística (as duas janelas de treino). A célula mostra uma linha por trimestre. Os resultados ficam em cache em `finor_resultados_v7_5/`: se a célula for executada de novo com a mesma configuração, os cálculos não são repetidos (para forçar, defina `USAR_CACHE_MODELO = False`).

In [ ]:
def avaliar_carteira(b, pesos, datas_teste):
    # Compra-e-mantém no trimestre de teste: os pesos derivam com os preços.
    R_teste = b["R"].reindex(datas_teste)[pesos.index]
    ausentes = int(R_teste.isna().sum().sum())
    valores = (1 + R_teste.fillna(0.0)).cumprod().mul(pesos, axis=1)
    patrimonio = valores.sum(axis=1)
    rp = patrimonio.pct_change(fill_method=None)
    rp.iloc[0] = patrimonio.iloc[0] - 1.0
    par = pd.concat([rp.rename("carteira"), b["y"].reindex(datas_teste).rename("indice")], axis=1).dropna()
    erro = par["carteira"] - par["indice"]
    metricas = {
        "TE fora (% a.a.)": 100 * erro.std() * np.sqrt(ANUAL),
        "RMSE fora (% a.a.)": 100 * np.sqrt((erro ** 2).mean() * ANUAL),
        "TD fora (p.p. a.a.)": 100 * erro.mean() * ANUAL,
        "retorno carteira (%)": 100 * ((1 + par["carteira"]).prod() - 1),
        "retorno índice (%)": 100 * ((1 + par["indice"]).prod() - 1),
        "dias de teste": len(par),
        "retornos ausentes": ausentes,
    }
    metricas["diferença no trimestre (p.p.)"] = metricas["retorno carteira (%)"] - metricas["retorno índice (%)"]
    return metricas, par, valores.iloc[-1] / valores.iloc[-1].sum()


def giro(alvo, anterior):
    if anterior is None:
        return np.nan
    universo = alvo.index.union(anterior.index)
    return 0.5 * (alvo.reindex(universo, fill_value=0) - anterior.reindex(universo, fill_value=0)).abs().sum()


def carteiras_da_janela(prob, com_miqp):
    # Todas as carteiras de uma janela: (método, K) -> (índices, pesos, erro de treino, tempo, informações do solver)
    carteiras = {}
    t0 = time.perf_counter()
    w, erro = prob.qp(range(prob.n))
    carteiras[("Sem limite", 0)] = (list(range(prob.n)), w, erro, time.perf_counter() - t0, {})
    correlacao = np.nan_to_num([np.corrcoef(prob.X[:, i], prob.a)[0, 1] for i in range(prob.n)], nan=-1.0)
    ordem = list(np.argsort(-correlacao))
    caminho = selecao_gulosa(prob, max(K_HEURISTICA))
    for K in K_HEURISTICA:
        t0 = time.perf_counter()
        w, erro = prob.qp(ordem[:K])
        carteiras[("Top-K correlação", K)] = (ordem[:K], w, erro, time.perf_counter() - t0, {})
        t0 = time.perf_counter()
        sel, w, erro = busca_trocas(prob, caminho[K][0])
        tempo_heuristica = time.perf_counter() - t0 + caminho[K][1]
        carteiras[("Heurística", K)] = (sel, w, erro, tempo_heuristica, {})
        if com_miqp and K in CFG["K_MIQP"]:
            w0 = np.zeros(prob.n)
            w0[sel] = w
            w_m, info = resolver_miqp(prob, K, w0, CFG["time_limit_s"])
            idx, erro_m = sel, erro
            if w_m is not None:
                idx_m = list(np.where(w_m > LIMIAR_PESO)[0])
                w_m = w_m[idx_m] / w_m[idx_m].sum()
                if prob.erro(idx_m, w_m) <= erro:   # nunca pior que a partida a quente
                    idx, w, erro_m = idx_m, w_m, prob.erro(idx_m, w_m)
            info["melhora sobre a heurística (%)"] = 100 * (1 - erro_m / erro) if erro > 0 else 0.0
            carteiras[("MIQP", K)] = (idx, w, erro_m, tempo_heuristica + info["tempo solver (s)"], info)
    return carteiras


def rodar_bloco(nome, J, com_miqp, relogio):
    b = bases[nome]
    janelas = construir_janelas(nome, J)
    linhas, pesos_longos, retornos_longos, anteriores = [], [], [], {}
    for j in janelas:
        t_janela = time.perf_counter()
        cand = j["candidatos"]
        Rt = b["R"].reindex(j["datas_treino"])[cand]
        yt = b["y"].reindex(j["datas_treino"])
        ok = yt.notna()
        prob = ProblemaTracking(Rt.loc[ok].fillna(0.0), yt.loc[ok])
        for (metodo, K), (idx, w, erro, tempo, info) in carteiras_da_janela(prob, com_miqp).items():
            pesos = pd.Series(w, index=[cand[i] for i in idx])
            pesos = pesos[pesos > LIMIAR_PESO]
            pesos = pesos / pesos.sum()
            metricas, par, pesos_finais = avaliar_carteira(b, pesos, j["datas_teste"])
            g = giro(pesos, anteriores.get((metodo, K)))
            anteriores[(metodo, K)] = pesos_finais
            chave = {"indice": nome, "janela treino": J, "método": metodo, "K": K}
            linhas.append({**chave, "trimestre": j["numero"], "rebalanceamento": j["rebalanceamento"],
                           "fim do teste": j["fim_teste"], "candidatas": len(cand), "ações": len(pesos),
                           "maior peso (%)": 100 * pesos.max(), "número efetivo de ações": 1 / (pesos ** 2).sum(),
                           "erro dentro (% a.a.)": np.sqrt(erro * ANUAL), **metricas,
                           "giro (%)": 100 * g, "tempo (s)": tempo, **info})
            pesos_longos.append(pd.DataFrame({**chave, "rebalanceamento": j["rebalanceamento"],
                                              "ticker": pesos.index, "peso": pesos.values}))
            retornos_longos.append(par.rename(columns={"carteira": "retorno carteira", "indice": "retorno índice"})
                                   .rename_axis("data").reset_index().assign(**chave))
        print(f"[{NOMES[nome]} | treino {J}] {j['numero']:>2}/{len(janelas)} {j['rebalanceamento']:%d/%m/%Y}: "
              f"{len(cand)} candidatas | {time.perf_counter() - t_janela:5.1f} s | "
              f"decorrido {(time.perf_counter() - relogio) / 60:4.1f} min", flush=True)
    return pd.DataFrame(linhas), pd.concat(pesos_longos, ignore_index=True), pd.concat(retornos_longos, ignore_index=True)


assinatura = {
    "versão": "7.7", "modo": MODO_EXECUCAO, **{k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
    "K_HEURISTICA": list(K_HEURISTICA), "janelas": [JANELA_TREINO, JANELA_ALTERNATIVA],
    "horizonte": HORIZONTE_TESTE, "cobertura": COBERTURA_MINIMA_TREINO,
}
# Assinatura dos dados de cada índice: se só os dados de um índice mudam, só os blocos dele são refeitos.
dados_assinatura = {nome: [int(b["R"].shape[1]), str(b["y"].index[-1].date()), round(float(b["R"].sum().sum()), 2)]
                    for nome, b in bases.items()}
ARQUIVO_CACHE = DATA / f"cache_modelo_{MODO_EXECUCAO}.pkl"
cache = pd.read_pickle(ARQUIVO_CACHE) if (USAR_CACHE_MODELO and ARQUIVO_CACHE.exists()) else {}
if cache.get("assinatura") != assinatura:   # configuração diferente: refaz tudo
    cache = {"assinatura": assinatura, "blocos": {}, "minutos": {}, "dados": {}}
cache.setdefault("dados", {})
for nome in bases:
    if cache["dados"].get(nome) != dados_assinatura[nome]:
        for chave_bloco in [k for k in cache["blocos"] if k.startswith(nome + "|")]:
            cache["blocos"].pop(chave_bloco)
            cache["minutos"].pop(chave_bloco, None)
        cache["dados"][nome] = dados_assinatura[nome]

n_miqp = sum(len(construir_janelas(nome, JANELA_TREINO)) for nome in bases) * len(CFG["K_MIQP"])
print(f"Estimativa: até {n_miqp * CFG['time_limit_s'] / 60:.0f} min de MIQP ({n_miqp} carteiras × {CFG['time_limit_s']} s), "
      "mais alguns minutos de heurística.")
relogio = time.perf_counter()
for J in (JANELA_TREINO, JANELA_ALTERNATIVA):
    for nome in ["S&P100", "IBOV"]:
        chave = f"{nome}|{J}"
        if chave in cache["blocos"]:
            print(f"[{NOMES[nome]} | treino {J}] resultados carregados do cache.")
            continue
        inicio_bloco = time.perf_counter()
        cache["blocos"][chave] = rodar_bloco(nome, J, com_miqp=(J == JANELA_TREINO), relogio=relogio)
        cache["minutos"][chave] = (time.perf_counter() - inicio_bloco) / 60
        pd.to_pickle(cache, ARQUIVO_CACHE)

resultados_janelas = pd.concat([v[0] for v in cache["blocos"].values()], ignore_index=True)
resultados_pesos = pd.concat([v[1] for v in cache["blocos"].values()], ignore_index=True)
resultados_retornos = pd.concat([v[2] for v in cache["blocos"].values()], ignore_index=True)
tempo_total_min = sum(cache["minutos"].values())
print(f"Experimento: {tempo_total_min:.1f} minutos de cálculo; {len(resultados_janelas)} carteiras avaliadas.")

## 13. Resultados fora da amostra

### 13.1 Tabela-resumo

Cada linha resume uma configuração em todos os trimestres de teste. As medidas padrão são o **TE** (a oscilação) e a **TD** (a direção), calculados sobre a série diária fora da amostra completa (os trimestres encadeados), como se a carteira fosse um fundo rebalanceado a cada trimestre. O TE dentro da amostra é medido com os mesmos pesos sobre os dados de treino. O RMSE, objetivo do modelo, aparece como referência. A carteira de referência dos gráficos de acompanhamento é o **MIQP com `K = 10`** e treino de 504 pregões.

In [ ]:
MESES_PT = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]


def mes_ano(data):
    return f"{MESES_PT[data.month - 1]}/{data.year}"


CHAVES = ["indice", "janela treino", "método", "K"]
series_oos = {k: g.set_index("data")[["retorno carteira", "retorno índice"]].set_axis(["carteira", "indice"], axis=1).sort_index()
              for k, g in resultados_retornos.groupby(CHAVES, sort=False)}

# Erro dentro da amostra nas medidas padrão do projeto (TE e TD): recalculado com os pesos de cada carteira sobre os mesmos
# retornos de treino da otimização (pesos fixos no treino). Não refaz a otimização; só mede.
dentro = []
for nome in ["S&P100", "IBOV"]:
    b = bases[nome]
    for J in sorted(resultados_janelas["janela treino"].unique()):
        pesos_J = resultados_pesos[(resultados_pesos["indice"] == nome) & (resultados_pesos["janela treino"] == J)]
        por_data = {d: g for d, g in pesos_J.groupby("rebalanceamento")}
        for j in construir_janelas(nome, J):
            if j["rebalanceamento"] not in por_data:
                continue
            Rt = b["R"].reindex(j["datas_treino"])[j["candidatos"]]
            yt = b["y"].reindex(j["datas_treino"])
            ok = yt.notna()
            Rt, yt = Rt.loc[ok].fillna(0.0), yt.loc[ok].to_numpy()
            for (metodo, K), g in por_data[j["rebalanceamento"]].groupby(["método", "K"]):
                d = Rt[g["ticker"].tolist()].to_numpy() @ g["peso"].to_numpy() - yt
                dentro.append({"indice": nome, "janela treino": J, "método": metodo, "K": K, "rebalanceamento": j["rebalanceamento"],
                               "TE dentro (% a.a.)": 100 * d.std(ddof=1) * np.sqrt(ANUAL), "TD dentro (p.p. a.a.)": 100 * d.mean() * ANUAL})
resultados_janelas = (resultados_janelas.drop(columns=["TE dentro (% a.a.)", "TD dentro (p.p. a.a.)"], errors="ignore")
                      .merge(pd.DataFrame(dentro), on=["indice", "janela treino", "método", "K", "rebalanceamento"], how="left"))

linhas = []
for k, g in resultados_janelas.groupby(CHAVES, sort=False):
    par = series_oos[k]
    erro = par["carteira"] - par["indice"]
    te_dentro = np.sqrt((g["TE dentro (% a.a.)"] ** 2).mean())
    te_fora = 100 * erro.std() * np.sqrt(ANUAL)
    linhas.append({
        **dict(zip(CHAVES, k)), "trimestres": len(g), "ações (média)": g["ações"].mean(),
        "maior peso médio (%)": g["maior peso (%)"].mean(),
        "TE dentro (% a.a.)": te_dentro,
        "TE fora (% a.a.)": te_fora,
        "TD (p.p. a.a.)": 100 * erro.mean() * ANUAL,
        "razão fora/dentro": te_fora / te_dentro,
        "RMSE fora (% a.a.)": 100 * np.sqrt((erro ** 2).mean() * ANUAL),
        "diferença acumulada (p.p.)": 100 * ((1 + par["carteira"]).prod() - (1 + par["indice"]).prod()),
        "TE trimestral máximo (% a.a.)": g["TE fora (% a.a.)"].max(),
        "giro médio (%)": g["giro (%)"].mean(),
        "tempo médio (s)": g["tempo (s)"].mean(),
        "gap médio (%)": g["gap (%)"].mean() if "gap (%)" in g else np.nan,
        "melhora média sobre a heurística (%)": g["melhora sobre a heurística (%)"].mean() if "melhora sobre a heurística (%)" in g else np.nan,
    })
resumo_modelos = pd.DataFrame(linhas)
ORDEM_METODO = {"MIQP": 0, "Heurística": 1, "Top-K correlação": 2, "Sem limite": 3}
resumo_modelos = (resumo_modelos.assign(_o=resumo_modelos["método"].map(ORDEM_METODO))
                  .sort_values(["indice", "janela treino", "_o", "K"], ascending=[False, False, True, True])
                  .drop(columns="_o").reset_index(drop=True))

METODO_PRINCIPAL = "Heurística"   # a heurística é o modelo base (seção 13.3); o MIQP fica como verificação
PRINCIPAL = (METODO_PRINCIPAL, K_PRINCIPAL, JANELA_TREINO)
inicio_oos = resultados_janelas["rebalanceamento"].min()
fim_oos = resultados_janelas["fim do teste"].max()
PERIODO_OOS = f"{mes_ano(inicio_oos)} a {mes_ano(fim_oos)}"
N_TRIMESTRES = int(resultados_janelas.loc[resultados_janelas["janela treino"] == JANELA_TREINO, "trimestre"].max())


def resumo(nome, metodo, K, coluna, J=JANELA_TREINO):
    linha = resumo_modelos[(resumo_modelos["indice"] == nome) & (resumo_modelos["janela treino"] == J)
                           & (resumo_modelos["método"] == metodo) & (resumo_modelos["K"] == K)]
    return float(linha[coluna].iloc[0]) if len(linha) else np.nan


colunas_tabela = ["indice", "janela treino", "método", "K", "ações (média)", "TE dentro (% a.a.)", "TE fora (% a.a.)",
                  "TD (p.p. a.a.)", "razão fora/dentro", "RMSE fora (% a.a.)",
                  "giro médio (%)", "tempo médio (s)", "gap médio (%)"]
print(f"Fora da amostra: {PERIODO_OOS}, {N_TRIMESTRES} trimestres. Carteira de referência: {METODO_PRINCIPAL} com K = {K_PRINCIPAL}.")
print("K = 0 indica a carteira sem limite de ações.")
display(resumo_modelos[resumo_modelos["janela treino"] == JANELA_TREINO][colunas_tabela].round(2).reset_index(drop=True))
print("Diagnóstico do MIQP por trimestre (status do solver):")
display(resultados_janelas[resultados_janelas["método"] == "MIQP"].groupby(["indice", "K", "status"]).agg(
    carteiras=("status", "size"), gap_mediano=("gap (%)", "median"), gap_máximo=("gap (%)", "max"),
    melhora_mediana=("melhora sobre a heurística (%)", "median"), tempo_médio=("tempo solver (s)", "mean")).round(2))

### 13.2 Quantas ações são precisas?

A fronteira mostra o erro fora da amostra para cada número máximo de ações. Os quatro métodos aparecem assim: a **heurística** é uma linha, porque foi rodada para todos os K (5 a 30); o **MIQP** aparece como círculos, um por K, porque cada carteira leva até 10 s no Gurobi; o **top-K por correlação** é o atalho ingênuo; e a carteira **sem limite** é uma linha horizontal, porque não depende de K. A comparação com o top-K isola o valor da otimização: as duas carteiras têm o mesmo número de ações e pesos otimizados; muda só **quais** ações entram.

In [ ]:
ERRO = "TE fora (% a.a.)"   # medida padrão de erro do projeto (a TD de cada carteira está na tabela-resumo)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.6), sharey=True)
r504 = resumo_modelos[resumo_modelos["janela treino"] == JANELA_TREINO]
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    g = r504[r504["indice"] == nome]
    h = g[g["método"] == "Heurística"].sort_values("K")
    t = g[g["método"] == "Top-K correlação"].sort_values("K")
    m = g[g["método"] == "MIQP"].sort_values("K")
    s = g[g["método"] == "Sem limite"].iloc[0]
    ax.axhline(s[ERRO], color=TINTA_2, lw=1.2, ls=(0, (4, 3)), zorder=1, label=f"Sem limite (todas as ~{s['ações (média)']:.0f} candidatas)")
    ax.annotate(f"Sem limite: {pct(s[ERRO], 1)}", (K_HEURISTICA[-1] + 6.5, s[ERRO]), xytext=(0, -6), textcoords="offset points",
                ha="right", va="top", fontsize=10, color=TINTA_2)
    ax.plot(t["K"], t[ERRO], color=CONTEXTO_ESC, lw=2, marker="o", ms=6, mec=FUNDO, mew=1.5, zorder=2, label="Top-K por correlação")
    ax.plot(h["K"], h[ERRO], color=CORES[nome], lw=2.4, marker="o", ms=7, mec=FUNDO, mew=1.5, zorder=4, label="Heurística (todos os K)")
    if len(m):
        ax.scatter(m["K"], m[ERRO], s=190, facecolor="none", edgecolor=TINTA, linewidth=1.3, zorder=5, label="MIQP no Gurobi")
        for _, l in m[m["K"] == K_PRINCIPAL].iterrows():
            ax.annotate(f"MIQP: {pct(l[ERRO], 1)}", (l["K"], l[ERRO]), xytext=(-9, -9), textcoords="offset points",
                        ha="right", va="top", fontsize=10, color=TINTA, fontweight="semibold")
    for _, l in t[t["K"] == K_PRINCIPAL].iterrows():
        ax.annotate(pct(l[ERRO], 1), (l["K"], l[ERRO]), xytext=(0, 10), textcoords="offset points", ha="center", fontsize=10, color=TINTA_2)
    ax.set_xticks(list(K_HEURISTICA))
    ax.set_xlim(K_HEURISTICA[0] - 2, K_HEURISTICA[-1] + 7)
    ax.set_ylim(0, r504[ERRO].max() * 1.12)
    eixo_pct(ax)
    ax.set_xlabel("Número máximo de ações (K)")
    ax.set_title(NOMES[nome])
axes[0].set_ylabel("Erro fora da amostra (TE, % ao ano)")
legenda(axes[1], loc="upper right", ncol=1)
m_sp, m_ib = resumo("S&P100", METODO_PRINCIPAL, K_PRINCIPAL, ERRO), resumo("IBOV", METODO_PRINCIPAL, K_PRINCIPAL, ERRO)
t_sp, t_ib = resumo("S&P100", "Top-K correlação", K_PRINCIPAL, ERRO), resumo("IBOV", "Top-K correlação", K_PRINCIPAL, ERRO)
if m_sp < t_sp and m_ib < t_ib:
    titulo = "Escolher as ações por otimização erra bem menos do que escolher as mais correlacionadas"
else:
    titulo = "Erro fora da amostra conforme o número máximo de ações"
titulos(fig, titulo,
        f"Erro fora da amostra (TE, % ao ano), {PERIODO_OOS}, carteiras refeitas a cada trimestre com treino de 2 anos. Com {K_PRINCIPAL} ações, o MIQP erra "
        f"{pct(m_sp, 1)} no S&P 100 e {pct(m_ib, 1)} no Ibovespa, contra {pct(t_sp, 1)} e {pct(t_ib, 1)} com as {K_PRINCIPAL} mais correlacionadas. "
        "O MIQP só foi rodado para K = 5, 10 e 20 (círculos) e fica quase em cima da heurística. A linha tracejada usa todas as candidatas: "
        "é o menor erro no treino, não um piso garantido fora dele.")
finalizar_figura("22_fronteira_erro_por_k.png", fig)

### 13.3 Dentro × fora da amostra

O erro de treino é sempre otimista: a carteira foi escolhida justamente para aquele período. A distância entre o erro dentro e o erro fora da amostra mede o quanto o que a carteira "aprendeu" deixa de valer no trimestre seguinte.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.8), sharey=True)
LARG = 0.38
giros = {}
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    g = r504[(r504["indice"] == nome) & (r504["método"] == "Heurística")].sort_values("K")
    m = r504[(r504["indice"] == nome) & (r504["método"] == "MIQP")].sort_values("K")
    s = resumo(nome, "Sem limite", 0, "giro médio (%)")
    x = np.arange(len(K_HEURISTICA))
    ax.bar(x - LARG / 2, g["giro médio (%)"].to_numpy(), LARG, color=CORES[nome], zorder=3, label="Heurística")
    ax.bar(x + LARG / 2, m["giro médio (%)"].to_numpy(), LARG, color=CONTEXTO_ESC, zorder=3, label="MIQP no Gurobi")
    for xi, vh, vm in zip(x, g["giro médio (%)"].to_numpy(), m["giro médio (%)"].to_numpy()):
        ax.text(xi - LARG / 2, vh + 0.6, pct(vh), ha="center", va="bottom", fontsize=9.5, color=CORES[nome], fontweight="semibold")
        ax.text(xi + LARG / 2, vm + 0.6, pct(vm), ha="center", va="bottom", fontsize=9.5, color=TINTA_2)
    ax.axhline(s, color=TINTA, lw=1.3, ls=(0, (4, 3)), zorder=4, label=f"Sem limite de ações: {pct(s)}")
    ax.set_xticks(x)
    ax.set_xticklabels([str(k) for k in K_HEURISTICA])
    ax.set_xlim(-0.7, len(x) - 0.3)
    ax.set_xlabel("Número máximo de ações (K)")
    ax.set_title(NOMES[nome])
    eixo_pct(ax)
    giros[nome] = g.set_index("K")["giro médio (%)"]
ax.set_ylim(0, max(r504["giro médio (%)"].max() * 1.2, 10))
axes[0].set_ylabel("Giro médio por trimestre (% da carteira)")
for ax in axes:
    legenda(ax, loc="upper right", ncol=1)
gk = {n: giros[n].get(K_PRINCIPAL, np.nan) for n in giros}
titulos(fig, f"Com {K_PRINCIPAL} ações, o modelo troca {pct(gk['S&P100'])} da carteira por trimestre no S&P 100 e {pct(gk['IBOV'])} no Ibovespa",
        "Giro médio por rebalanceamento (metade da soma das diferenças entre os pesos novos e os pesos que derivaram no trimestre), treino de 2 anos. "
        "Com poucas ações, cada uma pesa muito e trocar uma delas move bastante a carteira; com todas as ações, o giro é só o da própria composição do índice. "
        "O giro não é penalizado nesta versão: controlá-lo é o próximo passo.", base_pol=1.05, espaco_h=0.42)
fig.subplots_adjust(top=fig.subplotpars.top - 0.06)
finalizar_figura("29_giro_por_k.png", fig)


### 13.4 O erro ao longo do tempo

O EDA indicou que, desde 2023, as ações do S&P 100 andam muito menos juntas (seção 7.10). Se isso dificulta a replicação com poucas ações, o erro trimestral do S&P deve subir nos trimestres mais recentes. O painel de cima mostra o TE de cada trimestre (a oscilação); o de baixo, a TD do trimestre (a direção): o TE não diz se a carteira ficou acima ou abaixo do índice.

In [ ]:
ERRO = "TE fora (% a.a.)"
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(12, 8.4), sharex=True, gridspec_kw={"height_ratios": [1.35, 1]})
metodo, K, J = PRINCIPAL
medias, testes_t = {}, {}
largura_barra = pd.Timedelta(days=24)
for i, nome in enumerate(["S&P100", "IBOV"]):
    g = resultados_janelas[(resultados_janelas["indice"] == nome) & (resultados_janelas["método"] == metodo)
                           & (resultados_janelas["K"] == K) & (resultados_janelas["janela treino"] == J)].sort_values("rebalanceamento")
    meio = g["rebalanceamento"] + (g["fim do teste"] - g["rebalanceamento"]) / 2
    ax.plot(meio, g[ERRO], color=CORES[nome], lw=2.2, marker="o", ms=6.5, mec=FUNDO, mew=1.5, label=NOMES[nome], zorder=3)
    rotulo(ax, meio.iloc[-1], g[ERRO].iloc[-1], NOMES[nome], cor=None, marcador=False, dx=10)
    pior = g.loc[g[ERRO].idxmax()]
    ax.annotate(f"{NOMES[nome]}: {pct(pior[ERRO], 1)}\n({mes_ano(pior['rebalanceamento'])}–{mes_ano(pior['fim do teste'])})",
                (pior["rebalanceamento"] + (pior["fim do teste"] - pior["rebalanceamento"]) / 2, pior[ERRO]),
                xytext=(0, 12), textcoords="offset points", ha="center", va="bottom", fontsize=9.5, color=TINTA_2)
    antes, depois = g.loc[g["rebalanceamento"] < "2024-01-01", ERRO], g.loc[g["rebalanceamento"] >= "2024-01-01", ERRO]
    medias[nome] = (antes.mean(), depois.mean())
    testes_t[nome] = (depois.mean() - antes.mean()) / np.sqrt(antes.var() / len(antes) + depois.var() / len(depois)) if len(antes) > 1 else np.nan
    # direção: diferença de rentabilidade entre carteira e índice em cada trimestre
    deslocamento = (i - 0.5) * largura_barra
    ax2.bar(meio + deslocamento, g["diferença no trimestre (p.p.)"], width=largura_barra, color=CORES[nome], zorder=2)
for a in (ax, ax2):
    if inicio_oos < pd.Timestamp("2024-01-01"):
        a.axvspan(pd.Timestamp("2024-01-01"), fim_oos + pd.Timedelta(days=15), color=FAIXA, zorder=0)
ax.text(pd.Timestamp("2024-01-20"), 0.97, "2024 em diante", transform=ax.get_xaxis_transform(), va="top", fontsize=10, color=APAGADO)
ax2.axhline(0, color=TINTA_2, lw=0.8, zorder=3)
eixo_anos(ax2)
ax.set_ylim(0, resultados_janelas[(resultados_janelas["método"] == metodo) & (resultados_janelas["K"] == K)
                                  & (resultados_janelas["janela treino"] == J)][ERRO].max() * 1.3)
ax.set_xlim(inicio_oos - pd.Timedelta(days=30), fim_oos + (fim_oos - inicio_oos) * 0.12)
eixo_pct(ax)
ax2.set_xticks([pd.Timestamp(f"{a}-01-01") for a in range(inicio_oos.year + 1, fim_oos.year + 1)])
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: num(v, 0, sinal=True) + " p.p." if v else "0"))
ax.set_title("TE de cada trimestre (% ao ano): a oscilação em torno do índice", fontsize=11.5)
ax2.set_title("Viés no trimestre: rentabilidade da carteira − rentabilidade do índice (p.p.), a direção", fontsize=11.5)
legenda(ax, loc="upper left")
(a_sp, d_sp), (a_ib, d_ib) = medias["S&P100"], medias["IBOV"]
if pd.notna(a_sp) and d_sp > a_sp and d_sp > d_ib:
    titulo = "Desde 2024, o S&P 100 ficou mais difícil de replicar que o Ibovespa; a direção do erro muda de um trimestre para outro"
else:
    titulo = "O erro trimestral varia com o regime de mercado nos dois índices"
titulos(fig, titulo,
        f"Carteira {metodo} com {K} ações e treino de 2 anos. TE médio antes e depois de 2024: {pct(a_sp, 1)} e {pct(d_sp, 1)} no S&P 100; "
        f"{pct(a_ib, 1)} e {pct(d_ib, 1)} no Ibovespa. O TE não mostra se a carteira ficou acima ou abaixo do índice: "
        "isso é o viés, nas barras.", direita=0.9, espaco_h=0.32)
finalizar_figura("24_erro_por_trimestre.png", fig)
print("Diferença de média depois × antes de 2024 (estatística t aproximada):", {NOMES[n]: round(v, 2) for n, v in testes_t.items()})

### 13.5 A carteira acompanha o índice no acumulado?

O TE mede a oscilação dia a dia, e a TD, a direção média. Para quem investe, importa também a diferença **acumulada**: um desvio pequeno, mas sempre na mesma direção, vira uma distância grande ao longo do tempo. O gráfico mostra a evolução de 100 investidos na carteira e no índice; a tabela separa a diferença em **efeito do universo** (a carteira sem limite contra o índice: viés de sobrevivência) e **efeito da concentração** (a carteira de 10 ações contra a sem limite).

In [ ]:
metodo, K, J = PRINCIPAL


def marcar_anos(ax, inicio, fim):
    # Marca só os anos com dados (o espaço à direita fica para os rótulos).
    import matplotlib.dates as mdates
    ax.set_xticks([pd.Timestamp(f"{a}-01-01") for a in range(inicio.year + 1, fim.year + 1)])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))


fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.2))
acumulados, acumulados_ref = {}, {}
for ax, nome in zip(axes, ["S&P100", "IBOV"]):
    par = series_oos[(nome, J, metodo, K)]
    cart, idx = 100 * (1 + par["carteira"]).cumprod(), 100 * (1 + par["indice"]).cumprod()
    ref = 100 * (1 + series_oos[(nome, J, "Sem limite", 0)]["carteira"]).cumprod()
    ax.plot(idx.index, idx, color=CONTEXTO_ESC, lw=2.8, zorder=2)
    ax.plot(cart.index, cart, color=CORES[nome], lw=2, zorder=3)
    acima = cart.iloc[-1] >= idx.iloc[-1]
    rotulo(ax, cart.index[-1], cart.iloc[-1], f"Carteira ({K} ações): {pct(cart.iloc[-1] - 100, 0, sinal=True)}", cor=CORES[nome],
           dy=8 if acima else -8, tam=10.5)
    rotulo(ax, idx.index[-1], idx.iloc[-1], f"Índice: {pct(idx.iloc[-1] - 100, 0, sinal=True)}", cor=CONTEXTO_ESC,
           dy=-8 if acima else 8, tam=10.5, peso="normal")
    ax.axhline(100, color=TINTA_2, lw=0.8, zorder=1)
    eixo_num(ax, casas=0)
    ax.set_xlim(idx.index[0], idx.index[-1] + (idx.index[-1] - idx.index[0]) * 0.42)
    marcar_anos(ax, idx.index[0], idx.index[-1])
    ax.set_title(NOMES[nome])
    acumulados[nome] = cart.iloc[-1] - idx.iloc[-1]
    acumulados_ref[nome] = ref.iloc[-1] - idx.iloc[-1]


def acima_abaixo(v):
    return f"{num(abs(v), 0)} p.p. {'acima' if v >= 0 else 'abaixo'}"


titulos(fig, f"A carteira de {K} ações seguiu o índice e terminou {acima_abaixo(acumulados['S&P100'])} no S&P 100 e {acima_abaixo(acumulados['IBOV'])} no Ibovespa",
        f"Evolução de 100 investidos no início do período fora da amostra ({PERIODO_OOS}), carteira {metodo} refeita a cada trimestre, sem custos. "
        f"Linha cinza: o índice. A carteira com todas as ações também terminou acima ({acima_abaixo(acumulados_ref['S&P100'])} e "
        f"{acima_abaixo(acumulados_ref['IBOV'])}): boa parte da distância vem do universo de sobreviventes.", direita=0.96)
finalizar_figura("25_carteira_vs_indice.png", fig)

# De onde vem a diferença: efeito do universo (sem limite − índice) e da concentração (carteira − sem limite)
decomposicao_vies = pd.DataFrame([{
    "indice": nome,
    f"TD da carteira de {K} ações (p.p. a.a.)": resumo(nome, metodo, K, "TD (p.p. a.a.)"),
    "efeito do universo: sem limite − índice (p.p. a.a.)": resumo(nome, "Sem limite", 0, "TD (p.p. a.a.)"),
    f"efeito da concentração: {K} ações − sem limite (p.p. a.a.)": resumo(nome, metodo, K, "TD (p.p. a.a.)") - resumo(nome, "Sem limite", 0, "TD (p.p. a.a.)"),
} for nome in ["S&P100", "IBOV"]])
display(decomposicao_vies.round(2))


## 14. Parte 2 (Guia 2): oito modelos, giro, custos e robustez

A seção 13 mostrou quantas ações são precisas, justificou K = 20 e mostrou que o modelo escolhe bem. Esta seção responde ao Guia Parte 2: o modelo **vence alternativas mais simples**? O resultado vale **fora da amostra, em crises e com custos**? Comparamos oito modelos nos mesmos 20 trimestres, com K = 20 ações e a mesma função de avaliação.

| Família | Modelo | Como escolhe as ações | Pesos |
|---|---|---|---|
| Baseline | Top K por peso | As K maiores por valor de mercado (pesos da seção 7.14) | Proporcionais ao valor de mercado |
| Baseline | Top K por correlação, 1/K | As K mais correlacionadas com o índice | 1/K |
| Seleção + pesos | Top K por correlação + pesos otimizados | As mesmas K | Otimizados |
| Seleção + pesos | Lasso positivo | A penalidade L1 zera pesos até sobrarem K | Reotimizados |
| Seleção + pesos | Clustering hierárquico | Um representante por grupo de ações parecidas | Otimizados |
| Busca | Heurística gulosa + trocas | O método da seção 10 | Otimizados |
| Busca | Heurística com controle de giro | A mesma busca, com uma penalidade λ sobre o que precisa ser vendido da carteira anterior | No mesmo problema |
| Exato | MIQP com controle de giro | A heurística com controle de giro como ponto de partida e, depois, o Gurobi por 10 s | No mesmo problema |

Custo de negociação: 0,05% (EUA) e 0,15% (B3) por unidade de giro, o que cobre a compra e a venda (cerca de 0,025% e 0,075% por lado). S&P 100 (dólar) e Ibovespa (real) nunca se misturam. As funções ficam em `src/` e os resultados em `results/`; com `RECALCULAR_PARTE2 = False`, o notebook só lê os arquivos.

In [ ]:
### 14.1 Divisão temporal e features (Guia 2, seções 2.1 a 2.3)

Em vez de um corte fixo em treino, validação e teste, usamos o *walk-forward* da seção 11: 20 rodadas, cada uma com 504 pregões de treino e 63 de teste. Em cada rodada, **tudo** o que o modelo usa é calculado só com o treino. As features de cada ação (correlação com o índice, beta, volatilidade, correlação média com as demais, drawdown máximo e setor) estão em `src/features.py`. A tabela abaixo mostra as features da última janela de treino, com as vinte ações mais correlacionadas, e o efeito do filtro de redundância (pares com correlação acima de 0,90).

### 13.7 Treinar com 1 ano ou com 2 anos?

A seção 8 mostrou que, sem limite de ações, janelas mais curtas erram menos e giram mais. Aqui repetimos a comparação com a heurística para cada `K`, nos mesmos trimestres de teste.

In [ ]:
### 14.2 Tabela comparativa dos oito modelos (Guia 2, seção 4.3)

Medidas fora da amostra nos 20 trimestres, com K = 20 ações. **TE** é a oscilação em torno do índice, **TD** é a direção (a diferença média anualizada) e **giro** é a fração da carteira trocada por trimestre. O retorno líquido já desconta o custo de negociação. O retorno e o alfa do *Top K por peso* **não valem como resultado**: ele escolhe as ações pelo valor de mercado de set/2026, ou seja, já sabe quais cresceram (viés de olhar o futuro). No S&P 100, a linha do MIQP coincide com a da heurística com controle de giro: em 10 s o Gurobi não encontrou carteira melhor que a da heurística (ver a seção 14.3).

ARQ_PARTE2 = [f"tabela_comparativa_modelos_K{K_PRINCIPAL}_sp100.csv", f"tabela_comparativa_modelos_K{K_PRINCIPAL}_ibov.csv",
              f"series_oos_modelos_K{K_PRINCIPAL}.csv", f"te_por_regime_modelos_K{K_PRINCIPAL}.csv",
              f"modelo_nulo_monte_carlo_K{K_PRINCIPAL}.csv", f"estatistica_modelos_K{K_PRINCIPAL}.csv",
              f"estresse_remocao_maior_acao_sp100_K{K_PRINCIPAL}.csv", f"estresse_remocao_maior_acao_ibov_K{K_PRINCIPAL}.csv"]

if RECALCULAR_PARTE2 or not all((PASTA_SAIDA / a).exists() for a in ARQ_PARTE2):
    from src.compare_all_models import executar_indice, executar_sensibilidade_k, montar
    from src.estatistica import analisar
    for nome in ["S&P100", "IBOV"]:
        executar_indice(nome, K=K_PRINCIPAL, bases=bases)
        executar_sensibilidade_k(nome, K=K_PRINCIPAL, bases=bases)
    montar(K_PRINCIPAL)
    analisar(K_PRINCIPAL)
else:
    print("Comparação dos 8 modelos carregada de results/ (RECALCULAR_PARTE2 = True para refazer).")

comparativo = pd.concat([
    pd.read_csv(PASTA_SAIDA / f"tabela_comparativa_modelos_K{K_PRINCIPAL}_sp100.csv").assign(indice="S&P100"),
    pd.read_csv(PASTA_SAIDA / f"tabela_comparativa_modelos_K{K_PRINCIPAL}_ibov.csv").assign(indice="IBOV"),
], ignore_index=True)
colunas_cmp = ["Modelo", "TE (% a.a.)", "TD (p.p. a.a.)", "Correlação", "R²", "Giro Médio (% trim)", "Retorno Anual Líq (%)"]
for nome in ["S&P100", "IBOV"]:
    print(f"\n{NOMES[nome]} (K = {K_PRINCIPAL}, custo de {'5' if nome == 'S&P100' else '15'} bps)")
    display(comparativo[comparativo["indice"] == nome][colunas_cmp].round(2).reset_index(drop=True))

estat = pd.read_csv(PASTA_SAIDA / f"estatistica_modelos_K{K_PRINCIPAL}.csv")
print("\nIncerteza e testes pareados contra o modelo final (heurística com controle de giro) (bootstrap por trimestre e Wilcoxon):")
display(estat.round(2))

In [ ]:
### 14.3 Controle de giro e custos

O modelo da Parte 2 inclui no objetivo uma penalidade λ sobre a fração da carteira anterior que precisa ser **vendida** (com investimento total de 100%, o que se vende é o que se compra: o próprio giro). λ é a penalidade sobre o giro anualizado; o multiplicador usado pelo solver é λ × rebalanceamentos por ano ÷ 252. A seção 14.4.1 escolhe λ em um período separado de validação, para cada K.

O motor é a **heurística com penalidade**: a mesma busca gulosa e por trocas da seção 10, avaliando cada conjunto de ações pelo erro de réplica mais a penalidade, com duas partidas (carteira vazia e carteira anterior). O Gurobi entra depois, por 10 s, partindo da carteira da heurística, como verificação. A licença restrita do Gurobi limita o modelo a 200 variáveis; com a penalidade o MIQP precisa de 2N + K variáveis, então o universo de candidatas é reduzido a 90 ações com K = 20 (85 com K = 30). A tabela abaixo mostra o objetivo final da heurística e do Gurobi, rodada a rodada.

rodadas_modelos = {n: pd.read_csv(PASTA_SAIDA / f"rodadas_modelos_K{K_PRINCIPAL}_{'sp100' if n == 'S&P100' else 'ibov'}.csv") for n in ["S&P100", "IBOV"]}
for nome, d in rodadas_modelos.items():
    giro = d[d["modelo"].isin(["Heurística com controle de giro", "MIQP Gurobi (Turnover)"])]
    if {"objetivo_heuristica", "objetivo_final"} <= set(giro.columns):
        m = giro[giro["modelo"] == "MIQP Gurobi (Turnover)"]
        melhora = 100 * (1 - m["objetivo_final"] / m["objetivo_heuristica"])
        print(f"  Gurobi (10 s) melhorou o objetivo da heurística em mais de 0,001% em {(melhora > 1e-3).sum()} de {len(m)} rodadas; melhora mediana {melhora.median():.2f}%, máxima {melhora.max():.2f}%.")

In [ ]:
### 14.4 Os oito modelos em gráficos: erro e giro, acumulado, regimes e modelo nulo

Quatro figuras: o **erro e o giro** de cada modelo; o **modelo nulo** (nenhuma de 1.000 carteiras aleatórias, nos mesmos 20 trimestres, erra menos que o modelo final); o **erro por regime** (a Covid, em 2020, fica antes do primeiro trimestre de teste, set/2021, então a crise medida fora da amostra é 2022); e a **diferença acumulada** entre carteira e índice (a do *Top K por peso* é viés de olhar o futuro). A relação entre giro e erro por valor de λ está na seção 14.4.1.

FIGURAS_PARTE2 = ["31_comparativo_modelos.png", "33_modelo_nulo.png", "34_te_por_regime.png", "35_diferenca_acumulada.png"]
if RECALCULAR_PARTE2 or not all((PASTA_FIGURAS / f).exists() for f in FIGURAS_PARTE2):
    from src.figuras_parte2 import gerar_figuras, preparar_fonte
    preparar_fonte()
    gerar_figuras(bases, K=K_PRINCIPAL)
for arquivo in FIGURAS_PARTE2:
    mostrar_png(arquivo)

### 14.4.1 Escolher λ em um período separado (Etapa 12 do relatório)

Escolher λ nos mesmos trimestres em que depois medimos o resultado favoreceria a escolha. Por isso separamos a escolha da medida.

**Desenho, declarado antes dos resultados.** O modelo com controle de giro roda nos 20 trimestres de sempre, com os oito valores de λ da grade, para cada K da sensibilidade (10, 20 e 30). Os **8 primeiros trimestres** (set/2021 a cerca de set/2023) são a **validação**; os **12 últimos**, o **teste**. Cada rodada parte da carteira da anterior, então os trimestres de teste usam a carteira que o λ escolhido teria produzido desde o início, sem usar dados de teste para escolher λ. O motor é determinístico, então não há repetição por semente.

**Regra de escolha**, fixada antes de olhar os resultados: o **menor λ** da grade que (i) reduz o giro de validação em pelo menos **75%** em relação a λ = 0 e (ii) mantém o TE de validação a até **0,2 p.p.** do menor TE de validação da grade. O teste só é lido depois. Medidas: TE e giro médio por trimestre (sem a primeira rodada, em que a carteira é montada do zero).

In [ ]:
if RECALCULAR_VALIDACAO_LAMBDA:
    from src.validacao_lambda import executar_validacao
    for k_val in (10, 20, 30):
        executar_validacao(K=k_val, bases=bases)

from src.validacao_lambda import arquivo_rodadas, lambda_escolhido, resumir_e_escolher

if not all(arquivo_rodadas(K_PRINCIPAL, n).exists() for n in ["S&P100", "IBOV"]):
    print("Validação de λ ainda não rodada: defina RECALCULAR_VALIDACAO_LAMBDA = True na configuração (retoma de onde parou).")
else:
    for k_val in (10, 20, 30):
        if all(arquivo_rodadas(k_val, n).exists() for n in ["S&P100", "IBOV"]):
            medias_val, escolhido_val = resumir_e_escolher(k_val)
            print(f"K = {k_val}: λ escolhido só com a validação: {{{', '.join(f'{NOMES[n]}: {v:g}' for n, v in escolhido_val.items())}}}")
    medias_val = pd.read_csv(PASTA_SAIDA / f"validacao_lambda_K{K_PRINCIPAL}.csv")
    tabela_val = (medias_val[medias_val["periodo"].isin(["validação", "teste"])]
                  .pivot(index=["indice", "lambda_anual"], columns="periodo", values=["TE (% a.a.)", "Giro médio (% trim)"]))
    tabela_val.columns = [f"{m} – {p}" for m, p in tabela_val.columns]
    print(f"\nK = {K_PRINCIPAL}: TE e giro por λ, na validação e no teste")
    display(tabela_val.round(2))

    from src.figuras_parte2 import aplicar_estilo, figura_validacao_lambda, preparar_fonte
    preparar_fonte(); aplicar_estilo()
    figura_validacao_lambda(K_PRINCIPAL)
    mostrar_png("38_validacao_lambda.png")

In [ ]:
### 14.5 Rebalanceamento e janela de treino (Etapa 12 do relatório)

Duas decisões de desenho que o guia pede para documentar: **com que frequência rebalancear** (trimestral, 63 pregões, ou mensal, 21) e **quanto histórico usar** (2 anos, 504 pregões, ou 1 ano, 252). A seção 13.7 já mostrou que 1 ano reduz o erro da heurística, mas aumenta o giro; aqui as quatro combinações rodam com a **heurística com controle de giro**.

**Desenho, declarado antes dos resultados.**
- Quatro configurações por índice: janela de treino {504, 252} × rebalanceamento {63, 21}, com K = 20. A referência é a configuração base (504, 63), refeita aqui com o mesmo código.
- **Mesmos pregões de teste** em todas as configurações (set/2021 a set/2026, o intervalo dos 20 trimestres): a janela e a frequência mudam só o que o modelo vê e quando troca a carteira. No mensal são 59 a 60 rodadas, em vez de 20.
- **Penalidade λ em {0, 3, 6, 12} mais o λ escolhido na validação**, a mesma em todas as configurações. λ é a penalidade sobre o giro anualizado: o multiplicador do solver é rebalanceamentos por ano × λ ÷ 252, então λ tem o mesmo significado nas duas frequências.
- Medidas: TE fora da amostra, **giro anualizado** (giro médio por rodada × rebalanceamentos por ano, para comparar frequências), custo de transação (5 bps nos EUA, 15 bps na B3) e retorno líquido.
- **Regra de decisão:** só trocamos a configuração base se a alternativa for melhor, nos dois índices e nos dois períodos (validação e teste), por mais que a variação típica entre períodos. O motor é determinístico, então a régua de ruído vem da comparação entre períodos e índices, não de sementes.

if RECALCULAR_REBALANCEAMENTO:
    from src.rebalanceamento_experiment import CONFIGURACOES, executar as executar_rebalanceamento
    for nome in ["S&P100", "IBOV"]:
        for janela, horizonte in CONFIGURACOES:
            executar_rebalanceamento(K=K_PRINCIPAL, indice=nome, janela=janela, horizonte=horizonte, bases=bases)

from src.rebalanceamento_experiment import ler_resultados

rebal = ler_resultados(K_PRINCIPAL)
if not len(rebal):
    print("Experimento de rebalanceamento ainda não rodado: defina RECALCULAR_REBALANCEAMENTO = True na configuração (retoma de onde parou).")
else:
    nome_janela = {504: "2 anos", 252: "1 ano"}
    nome_freq = {63: "trimestral", 21: "mensal"}
    rebal["janela"] = rebal["janela_treino"].map(nome_janela)
    rebal["rebalanceamento"] = rebal["horizonte_teste"].map(nome_freq)
    escolhido_reb = lambda_escolhido(K_PRINCIPAL)
    colunas_reb = ["janela", "rebalanceamento", "rodadas", "TE (% a.a.)", "Giro médio (% trim)", "Giro anualizado (%)",
                   "Custo fricção (% a.a.)", "Alfa líquido (p.p. a.a.)"]
    for nome in ["S&P100", "IBOV"]:
        g = rebal[(rebal["indice"] == nome) & (rebal["lambda_anual"] == escolhido_reb[nome])]
        print(f"\n{NOMES[nome]}, λ = {escolhido_reb[nome]:g}: as quatro configurações")
        display(g[colunas_reb].round(2).reset_index(drop=True))
        for medida in ["TE (% a.a.)", "Giro anualizado (%)"]:
            print(f"{NOMES[nome]}: {medida} por λ")
            display(rebal[rebal["indice"] == nome].pivot_table(index=["janela", "rebalanceamento"], columns="lambda_anual", values=medida).round(2))

    from src.figuras_parte2 import aplicar_estilo, figura_rebalanceamento
    aplicar_estilo()
    figura_rebalanceamento(K_PRINCIPAL)
    mostrar_png("36_rebalanceamento_janela.png")

In [ ]:
### 14.6 Robustez, verificação e custos (Guia 2, seções 5.1, 5.4 a 5.6)

Quatro testes de robustez: (1) **modelo nulo** (1.000 carteiras aleatórias); (2) sensibilidade ao K (K − 5, K e K + 5, e as seções 13.2 e 13.5); (3) janela de treino de 2 anos × 1 ano (seções 13.7 e 14.5); (4) remoção da ação de maior peso. A repetição do solver com várias sementes deixou de ser necessária: o motor é a heurística, que é determinística, e o Gurobi só verifica o resultado.

**Verificação (5.1):** abaixo, as checagens de sanidade das carteiras: pesos somam 1, não negativos, no máximo K ações, sem tickers repetidos.

nulo = pd.read_csv(PASTA_SAIDA / f"modelo_nulo_monte_carlo_K{K_PRINCIPAL}.csv")
print("Modelo nulo (TE fora da amostra, 20 trimestres):")
display(nulo.round(2))

estresse = pd.concat([pd.read_csv(PASTA_SAIDA / f"estresse_remocao_maior_acao_{c}_K{K_PRINCIPAL}.csv").assign(indice=n)
                      for n, c in [("S&P100", "sp100"), ("IBOV", "ibov")]], ignore_index=True)
print("Remoção da ação de maior peso (último trimestre):")
display(estresse[["indice", "Ativo Removido", "Tracking Error (% a.a.)", "R²", "Correlação"]].round(3))

tamanho = pd.concat([pd.read_csv(PASTA_SAIDA / f"robustez_tamanho_K{K_PRINCIPAL}_{c}.csv").assign(indice=n)
                     for n, c in [("S&P100", "sp100"), ("IBOV", "ibov")]], ignore_index=True)
print(f"Sensibilidade ao tamanho da carteira (K − 5, K, K + 5):")
display(tamanho.round(2))

# Verificação das carteiras avaliadas na seção 13 (pesos de todas as carteiras)
soma_pesos = resultados_pesos.groupby(["indice", "janela treino", "método", "K", "rebalanceamento"])["peso"].agg(["sum", "min", "count"])
assert (soma_pesos["sum"] - 1).abs().max() < 1e-6, "pesos não somam 1"
assert (soma_pesos["min"] >= 0).all(), "peso negativo"
limite = soma_pesos.reset_index()
limite = limite[limite["K"] > 0]
assert (limite["count"] <= limite["K"]).all(), "mais ações que K"
assert not resultados_pesos.duplicated(["indice", "janela treino", "método", "K", "rebalanceamento", "ticker"]).any(), "ticker repetido"
print(f"Verificação OK em {len(soma_pesos)} carteiras: pesos somam 1, não negativos, no máximo K ações, sem tickers repetidos.")

te_regimes = pd.read_csv(PASTA_SAIDA / f"te_por_regime_modelos_K{K_PRINCIPAL}.csv")
print("TE por regime de mercado, modelo final e heurística sem controle de giro (% a.a.):")
display(te_regimes[te_regimes["modelo"].isin(["Heurística com controle de giro", "Heurística Gulosa + Swaps"])]
        .pivot_table(index=["indice", "modelo"], columns="regime", values="TE (% a.a.)").round(2))

**Custos.** Com custos de 0,05% (EUA) e 0,15% (B3) por unidade de giro, cobrindo compra e venda, o giro pesa pouco no retorno: a heurística sem controle custa 0,06 ponto por ano no S&P 100 e 0,14 no Ibovespa; o modelo final (heurística com controle de giro), 0,02 nos dois. O ganho é operacional: o modelo final faz entre um sétimo e um quarto das ordens. O custo fixo não captura spread, impacto de preço nem imposto sobre ganho de capital.

**Limitações da Parte 2.** (1) Viés de sobrevivência (seção 6.3); (2) pesos do índice aproximados pelo valor de mercado atual, o que faz o *Top K por peso* olhar o futuro; (3) período de 7 anos, com 2022 como única crise medida fora da amostra; (4) K, λ e a regra de decisão escolhidos dentro dos mesmos 20 trimestres (λ com validação nos 8 primeiros); (5) os modelos com controle de giro usam no máximo 90 candidatas (85 com K = 30), pelo limite de 200 variáveis da licença do Gurobi; (6) custos simplificados; (7) dados do Yahoo Finance, com a base inteira ainda não conferida com a B3.

In [ ]:
import src.features as feat

linhas_feat = []
for nome in ["S&P100", "IBOV"]:
    j = construir_janelas(nome, JANELA_TREINO)[-1]                      # última janela de treino
    R_tr = bases[nome]["R"].reindex(j["datas_treino"])[j["candidatos"]].fillna(0.0)
    y_tr = bases[nome]["y"].reindex(j["datas_treino"]).fillna(0.0)
    tabela_feat = feat.construir_features(R_tr, y_tr, mapa_setores=dict(zip(setores["ticker"], setores["setor"])))
    escolhidas, _ = feat.selecionar_acoes_filtro_redundancia(R_tr, y_tr, top_n=40, limite_corr_par=0.90,
                                                              mapa_setores=dict(zip(setores["ticker"], setores["setor"])))
    print(f"{NOMES[nome]}: treino de {j['datas_treino'][0]:%d/%m/%Y} a {j['datas_treino'][-1]:%d/%m/%Y}; "
          f"{len(j['candidatos'])} candidatas; filtro de redundância: 40 mais correlacionadas -> {len(escolhidas)}")
    display(tabela_feat.sort_values("corr_idx", ascending=False).head(10).round(3))

### 14.2 Tabela comparativa dos sete modelos (Guia 2, seção 4.3)

Medidas fora da amostra nos 20 trimestres, com 10 ações. **TE** é a oscilação em torno do índice, **TD** é a direção (a diferença média anualizada) e **giro** é a fração da carteira trocada por trimestre. O retorno líquido já desconta o custo de negociação. O retorno e o alfa do *Top K por peso* **não valem como resultado**: ele escolhe as ações pelo valor de mercado de set/2026, ou seja, já sabe quais cresceram (viés de olhar o futuro).

In [ ]:
ARQ_PARTE2 = ["tabela_comparativa_modelos_sp100.csv", "tabela_comparativa_modelos_ibov.csv", "series_oos_modelos.csv",
              "te_por_regime_modelos.csv", "modelo_nulo_monte_carlo.csv", "estresse_remocao_maior_acao_sp100.csv",
              "estresse_remocao_maior_acao_ibov.csv"]
ARQ_PARETO = ["tabela_pareto_sp100.csv", "tabela_pareto_ibov.csv"]

if RECALCULAR_PARTE2 or not all((PASTA_SAIDA / a).exists() for a in ARQ_PARTE2):
    from src.compare_all_models import executar_comparativo_completo
    executar_comparativo_completo(bases)
else:
    print("Comparação dos 7 modelos carregada de results/ (RECALCULAR_PARTE2 = True para refazer).")

comparativo = pd.concat([
    pd.read_csv(PASTA_SAIDA / "tabela_comparativa_modelos_sp100.csv").assign(indice="S&P100"),
    pd.read_csv(PASTA_SAIDA / "tabela_comparativa_modelos_ibov.csv").assign(indice="IBOV"),
], ignore_index=True)
colunas_cmp = ["Modelo", "TE (% a.a.)", "TD (p.p. a.a.)", "Correlação", "R²", "Giro Médio (% trim)", "Retorno Anual Líq (%)"]
for nome in ["S&P100", "IBOV"]:
    print(f"\n{NOMES[nome]} (K = 10, custo de {'5' if nome == 'S&P100' else '15'} bps)")
    display(comparativo[comparativo["indice"] == nome][colunas_cmp].round(2).reset_index(drop=True))

### 14.3 Controle de giro e custos

O MIQP da Parte 2 inclui no objetivo uma penalidade λ sobre a fração da carteira anterior que precisa ser vendida. A grade abaixo varia λ de 0 a 50. O giro cai de cerca de 25% para poucos por cento por trimestre, enquanto o TE quase não muda até λ = 25. Escolhemos λ = 6 no S&P 100 e λ = 3 no Ibovespa. **Ressalva:** λ foi escolhido nos mesmos 20 trimestres de teste, sem um período de validação separado.

In [ ]:
if RECALCULAR_PARTE2 or not all((PASTA_SAIDA / a).exists() for a in ARQ_PARETO):
    from src.pareto_experiment import executar_grade
    executar_grade(bases)
else:
    print("Grade de Pareto carregada de results/.")

pareto = {nome: pd.read_csv(PASTA_SAIDA / arq) for nome, arq in [("S&P100", "tabela_pareto_sp100.csv"), ("IBOV", "tabela_pareto_ibov.csv")]}
colunas_par = ["lambda_anual", "TE (% a.a.)", "TD (p.p. a.a.)", "RMSE (% a.a.)", "Giro médio (% trim)", "Custo fricção (% a.a.)", "Alfa líquido (p.p. a.a.)"]
for nome, d in pareto.items():
    print(f"\n{NOMES[nome]}")
    display(d[colunas_par].round(2).reset_index(drop=True))

### 14.4 Os sete modelos em gráficos: erro e giro, giro × erro, acumulado, regimes e modelo nulo

- **Erro e giro:** heurística e MIQP erram cerca de 5% ao ano, metade dos demais; só o MIQP mantém o giro baixo.
- **Fronteira de giro:** a penalidade λ corta as trocas sem piorar o erro.
- **Modelo nulo:** nenhuma de 1.000 carteiras aleatórias, nos mesmos 20 trimestres, erra menos que o MIQP.
- **Regimes:** a Covid (2020) fica antes do primeiro trimestre de teste (set/2021), então a crise medida fora da amostra é 2022.
- **Diferença acumulada:** carteira − índice; a do *Top K por peso* é viés de olhar o futuro.

In [ ]:
FIGURAS_PARTE2 = ["31_comparativo_modelos.png", "32_fronteira_giro.png", "33_modelo_nulo.png", "34_te_por_regime.png", "35_diferenca_acumulada.png"]
if RECALCULAR_PARTE2 or not all((PASTA_FIGURAS / f).exists() for f in FIGURAS_PARTE2):
    from src.figuras_parte2 import gerar_figuras
    gerar_figuras(bases)
for arquivo in FIGURAS_PARTE2:
    mostrar_png(arquivo)

### 14.5 Rebalanceamento e janela de treino (etapa 13 do relatório)

Duas decisões de desenho que o guia pede para documentar: **com que frequência rebalancear** (trimestral, 63 pregões, ou mensal, 21) e **quanto histórico usar** (2 anos, 504 pregões, ou 1 ano, 252). A seção 13.7 já mostrou que 1 ano reduz o erro da heurística, mas aumenta o giro; aqui as quatro combinações rodam com o **MIQP com controle de giro**.

**Desenho, declarado antes dos resultados.**
- Quatro configurações por índice: janela de treino {504, 252} × rebalanceamento {63, 21}, com K = 10. A referência é a atual (504, 63), refeita aqui com o mesmo código.
- **Mesmos pregões de teste** em todas as configurações (set/2021 a set/2026, o intervalo dos 20 trimestres): a janela e a frequência mudam só o que o modelo vê e quando troca a carteira. No mensal são 59 a 60 rodadas, em vez de 20.
- **Penalidade λ em {0, 3, 6, 12}**, a mesma em todas as configurações. λ é a penalidade sobre o giro anualizado: o multiplicador do solver é rebalanceamentos por ano × λ ÷ 252, então λ tem o mesmo significado nas duas frequências. A grade mostra se λ = 6 (S&P 100) e λ = 3 (Ibovespa) continuam bons nas novas configurações.
- Medidas: TE fora da amostra, **giro anualizado** (giro médio por rodada × rebalanceamentos por ano, para comparar frequências), custo de transação (5 bps nos EUA, 15 bps na B3) e retorno líquido.
- **Ruído do solver:** o Gurobi para em 1,5 s por carteira. A referência é repetida com três sementes e a faixa de variação do TE serve de régua: diferenças menores que ela não são conclusão.


In [ ]:
ARQ_REBAL = PASTA_SAIDA / "rebalanceamento_janela.csv"
if RECALCULAR_REBALANCEAMENTO:
    from src.rebalanceamento_experiment import executar_experimento
    executar_experimento(bases)

if not ARQ_REBAL.exists():
    print("Experimento da etapa 13 ainda não rodado: defina RECALCULAR_REBALANCEAMENTO = True na configuração (~35 min; retoma de onde parou).")
else:
    from src.rebalanceamento_experiment import LAMBDA_ESCOLHIDO, REFERENCIA, SEED_PADRAO
    rebal = pd.read_csv(ARQ_REBAL)
    nome_janela = {504: "2 anos", 252: "1 ano"}
    nome_freq = {63: "trimestral", 21: "mensal"}
    rebal["janela"] = rebal["janela_treino"].map(nome_janela)
    rebal["rebalanceamento"] = rebal["horizonte_teste"].map(nome_freq)
    principal = rebal[rebal["seed"] == SEED_PADRAO]
    colunas_reb = ["janela", "rebalanceamento", "rodadas", "TE (% a.a.)", "Giro médio (% trim)", "Giro anualizado (%)",
                   "Custo fricção (% a.a.)", "Alfa líquido (p.p. a.a.)"]
    for nome in ["S&P100", "IBOV"]:
        g = principal[(principal["indice"] == nome) & (principal["lambda_anual"] == LAMBDA_ESCOLHIDO[nome])]
        print(f"\n{NOMES[nome]}, λ = {LAMBDA_ESCOLHIDO[nome]:g}: as quatro configurações")
        display(g[colunas_reb].round(2).reset_index(drop=True))
        for medida in ["TE (% a.a.)", "Giro anualizado (%)"]:
            print(f"{NOMES[nome]}: {medida} por λ")
            display(principal[principal["indice"] == nome].pivot_table(index=["janela", "rebalanceamento"], columns="lambda_anual", values=medida).round(2))

    # ruído do solver: a referência com várias sementes, no λ escolhido
    ref = rebal[(rebal["janela_treino"] == REFERENCIA[0]) & (rebal["horizonte_teste"] == REFERENCIA[1])]
    ruido = pd.concat([ref[(ref["indice"] == n) & (ref["lambda_anual"] == LAMBDA_ESCOLHIDO[n])] for n in ["S&P100", "IBOV"]])
    faixa = ruido.groupby("indice").agg(sementes=("seed", "nunique"), TE_min=("TE (% a.a.)", "min"), TE_max=("TE (% a.a.)", "max"),
                                         giro_min=("Giro anualizado (%)", "min"), giro_max=("Giro anualizado (%)", "max"))
    faixa["faixa do TE (p.p.)"] = faixa["TE_max"] - faixa["TE_min"]
    print("Ruído do solver na configuração de referência:")
    display(faixa.round(2))

    from src.figuras_parte2 import aplicar_estilo, figura_rebalanceamento
    aplicar_estilo()
    figura_rebalanceamento()
    mostrar_png("36_rebalanceamento_janela.png")


### 14.6 Robustez, verificação e custos (Guia 2, seções 5.1, 5.4 a 5.6)

Seis testes de robustez: (1) **modelo nulo** (1.000 carteiras aleatórias); (2) sensibilidade ao K (seção 13.2); (3) janela de treino de 2 anos × 1 ano (seção 13.7); (4) remoção da ação de maior peso; (5) extremos amortecidos (seção 13.10); (6) repetição do solver, porque o Gurobi para em 1,5 s por carteira e o TE do MIQP varia até 0,4 ponto entre execuções (4,9% a 5,3% no S&P 100; 4,5% a 4,7% no Ibovespa).

**Verificação (5.1):** abaixo, as checagens de sanidade das carteiras: pesos somam 1, não negativos, no máximo K ações, sem tickers repetidos.

In [ ]:
nulo = pd.read_csv(PASTA_SAIDA / "modelo_nulo_monte_carlo.csv")
print("Modelo nulo (TE fora da amostra, 20 trimestres):")
display(nulo.round(2))

estresse = pd.concat([pd.read_csv(PASTA_SAIDA / f"estresse_remocao_maior_acao_{c}.csv").assign(indice=n)
                      for n, c in [("S&P100", "sp100"), ("IBOV", "ibov")]], ignore_index=True)
print("Remoção da ação de maior peso (último trimestre):")
display(estresse[["indice", "Ativo Removido", "Tracking Error (% a.a.)", "R²", "Correlação"]].round(3))

# Verificação das carteiras avaliadas na seção 13 (pesos de todas as carteiras)
soma_pesos = resultados_pesos.groupby(["indice", "janela treino", "método", "K", "rebalanceamento"])["peso"].agg(["sum", "min", "count"])
assert (soma_pesos["sum"] - 1).abs().max() < 1e-6, "pesos não somam 1"
assert (soma_pesos["min"] >= 0).all(), "peso negativo"
limite = soma_pesos.reset_index()
limite = limite[limite["K"] > 0]
assert (limite["count"] <= limite["K"]).all(), "mais ações que K"
assert not resultados_pesos.duplicated(["indice", "janela treino", "método", "K", "rebalanceamento", "ticker"]).any(), "ticker repetido"
print(f"Verificação OK em {len(soma_pesos)} carteiras: pesos somam 1, não negativos, no máximo K ações, sem tickers repetidos.")

te_regimes = pd.read_csv(PASTA_SAIDA / "te_por_regime_modelos.csv")
print("TE por regime de mercado, modelo final e heurística sem controle de giro (% a.a.):")
display(te_regimes[te_regimes["modelo"].isin(["Heurística com controle de giro", "Heurística Gulosa + Swaps"])]
        .pivot_table(index=["indice", "modelo"], columns="regime", values="TE (% a.a.)").round(2))

**Custos.** Com custos de 0,05% (EUA) e 0,15% (B3) sobre o valor negociado, o giro pesa pouco no retorno (a heurística custa 0,07 ponto por ano no S&P 100 e 0,17 no Ibovespa; o MIQP, 0,01 e 0,05), mas muito na operação: o MIQP faz cerca de um quinto das ordens. O custo fixo não captura spread, impacto de preço nem imposto sobre ganho de capital.

**Limitações da Parte 2.** (1) Viés de sobrevivência (seção 6.3); (2) pesos do índice aproximados pelo valor de mercado atual, o que faz o *Top K por peso* olhar o futuro; (3) período de 7 anos, com 2022 como única crise medida fora da amostra; (4) λ escolhido nos mesmos trimestres de teste; (5) solver com tempo curto, com variação de até 0,4 ponto no TE do MIQP; (6) custos simplificados; (7) dados do Yahoo Finance, com a base inteira ainda não conferida com a B3.

## 15. Conclusões preliminares

O texto abaixo é gerado a partir dos resultados desta execução, para que os números citados estejam sempre atualizados. As conclusões desta seção são da versão preliminar (seção 13): valem para o universo fixo de 2026, com rebalanceamento trimestral e sem custos de transação. A seção 14 acrescenta o controle de giro, os custos e os testes de robustez.

In [ ]:
metodo, K, J = PRINCIPAL


def r(nome, met, k, col, j=JANELA_TREINO):
    return resumo(nome, met, k, col, j)


linhas_md = ["**Leitura automática dos resultados do modelo**", ""]
k_lista = [k for k in (5, 10, 20) if k in K_HEURISTICA]
for nome in ["S&P100", "IBOV"]:
    tes = ", ".join(f"{pct(r(nome, 'Heurística', k, 'TE fora (% a.a.)'), 1)} com {k}" for k in k_lista)
    linhas_md.append(f"- **{NOMES[nome]}:** erro fora da amostra de {tes} ações; a carteira sem limite (~{r(nome, 'Sem limite', 0, 'ações (média)'):.0f} ações) "
                     f"erra {pct(r(nome, 'Sem limite', 0, 'TE fora (% a.a.)'), 1)}. Escolher as {K} mais correlacionadas erra "
                     f"{pct(r(nome, 'Top-K correlação', K, 'TE fora (% a.a.)'), 1)}.")
if len(comparacao_miqp):
    linhas_md.append(f"- **MIQP × heurística:** em {CFG['time_limit_s']} s por carteira, o Gurobi reduziu o erro de treino da heurística em "
                     f"{num(comparacao_miqp['melhora sobre a heurística (%)'].median(), 2)}% na mediana (máximo de {num(comparacao_miqp['melhora sobre a heurística (%)'].max(), 1)}%) "
                     f"e terminou com gap mediano de {num(comparacao_miqp['gap (%)'].median(), 0)}%. Fora da amostra, a diferença média de TE foi de "
                     f"{num(comparacao_miqp['diferença TE fora (p.p.)'].mean(), 2, sinal=True)} p.p. A heurística já entrega quase todo o ganho; o MIQP serve como verificação.")
rz = {n: r(n, "Heurística", K, "razão fora/dentro") for n in ["S&P100", "IBOV"]}
linhas_md.append(f"- **Dentro × fora da amostra (K={K}):** o erro fora é {num(rz['S&P100'], 1)} vez o de treino no S&P 100 e {num(rz['IBOV'], 1)} no Ibovespa.")
linhas_md.append(f"- **Diferença acumulada:** a carteira {metodo} com {K} ações ficou {num(r('S&P100', metodo, K, 'TD (p.p. a.a.)'), 1, sinal=True)} p.p. por ano em relação ao S&P 100 "
                 f"e {num(r('IBOV', metodo, K, 'TD (p.p. a.a.)'), 1, sinal=True)} p.p. em relação ao Ibovespa. "
                 f"A carteira sem limite também fica acima dos índices ({num(r('S&P100', 'Sem limite', 0, 'TD (p.p. a.a.)'), 1, sinal=True)} e "
                 f"{num(r('IBOV', 'Sem limite', 0, 'TD (p.p. a.a.)'), 1, sinal=True)} p.p. por ano): parte da diferença vem do viés de sobrevivência, não do modelo.")
linhas_md.append(f"- **Giro:** com {K} ações, {pct(r('S&P100', metodo, K, 'giro médio (%)'))} da carteira é trocada por trimestre no S&P 100 e "
                 f"{pct(r('IBOV', metodo, K, 'giro médio (%)'))} no Ibovespa, contra {pct(r('S&P100', 'Sem limite', 0, 'giro médio (%)'))} e "
                 f"{pct(r('IBOV', 'Sem limite', 0, 'giro médio (%)'))} sem limite de ações.")
linhas_md.append(f"- **Janela de treino:** com K={K}, treinar com {JANELA_ALTERNATIVA} pregões leva o TE de {pct(te_janela['S&P100'][0], 1)} para "
                 f"{pct(te_janela['S&P100'][1], 1)} no S&P 100 e de {pct(te_janela['IBOV'][0], 1)} para {pct(te_janela['IBOV'][1], 1)} no Ibovespa.")
linhas_md.append("")
linhas_md.append("**Hipóteses do EDA:** " + "; ".join(f"{h.split('.')[0]} {res}" for h, res in zip(hipoteses["hipótese do EDA"], hipoteses["resultado"])) + ".")
display(Markdown("\n".join(linhas_md)))

## 16. Limitações

1. **Viés de sobrevivência:** o universo de 30/09/2026 é usado em todo o período. As ações de hoje sobreviveram e, em média, cresceram mais que o índice, o que deixa a diferença acumulada das carteiras positiva (seções 6.3 e 13.5). Só as quatro entradas do S&P 100 de 21/09/2026 são tratadas.
2. **Sem composição histórica (*point-in-time*):** as saídas dos índices não estão no universo, e as demais entradas do período não são controladas.
3. **MIQP sem prova de ótimo:** com o limite de tempo, o Gurobi termina com gap alto. As carteiras do MIQP são boas soluções factíveis, não ótimos comprovados. O gap alto vem sobretudo do limite inferior fraco da formulação. Como o limite é de tempo, outra máquina pode chegar a carteiras um pouco diferentes (a heurística é determinística).
4. **Giro e custos:** na seção 13 o giro é medido, mas não entra no objetivo. Na seção 14 ele entra (penalidade λ) e há custo fixo de 0,05% (EUA) e 0,15% (B3), sem spread, impacto de preço, impostos ou lotes.
5. **Rebalanceamento fixo:** trimestral, sem ajustes no meio do trimestre.
6. **Convenções diferentes:** S&P 100 em retorno de preços (USD) e Ibovespa em retorno total aproximado (BRL); os níveis de erro dos dois mercados não são diretamente comparáveis.
7. **Amostra curta:** 20 trimestres fora da amostra e poucos regimes de mercado.
8. **Fonte pública:** Yahoo Finance, adequado para um estudo acadêmico, mas sem a auditoria de uma base institucional.
9. **Verificação dos extremos por amostra:** dos 272 extremos com origem na empresa, 50 foram conferidos (os 30 maiores e 20 sorteados), todos reais. Os demais não foram conferidos um a um: a taxa de erro entre eles fica abaixo de 14% (95% de confiança). Erros pequenos, como um provento pequeno lançado no dia errado, ficam abaixo dos limites da triagem.
10. **Restrições simplificadas:** sem peso máximo por ação, caixa ou limites de liquidez (a seção 7.13 indica que liquidez não seria restrição).

## 17. Próximos passos

1. **Validar K e λ em período separado:** o controle de giro (seção 14) já está no modelo, mas λ foi escolhido nos mesmos 20 trimestres de teste.
2. **Reduzir o viés de sobrevivência:** reconstruir a composição histórica dos índices (carteiras teóricas da B3; mudanças do S&P 100) e incluir as ações que saíram.
3. **Fortalecer o MIQP:** formulações mais apertadas (por exemplo, a reformulação perspectiva), mais tempo de solver ou um limite inferior melhor, para reduzir o gap; e um limite de trabalho (`WorkLimit`) no lugar do limite de tempo, para que o resultado não dependa da máquina.
4. **Testar a frequência de rebalanceamento** (mensal × trimestral) e a janela de treino adaptativa por mercado.
5. **Restrições de negócio:** peso máximo por ação, lotes mínimos e custos com spread e impacto de preço.
6. **Conferir a base inteira com a B3:** o caso da Copasa mostra que o Yahoo pode lançar um provento no dia errado. Comparar todos os retornos com as séries históricas oficiais (COTAHIST) e a tabela de proventos da B3 pegaria também os erros pequenos, abaixo dos limites da triagem.
7. **Referência de mercado:** comparar com a réplica parcial das K maiores ações nos pesos oficiais (B3 e ETF OEF), com *free float*.

## 18. Aderência aos guias do projeto

- **Dados e EDA:** sete anos de preços diários, auditoria de faltantes e outliers com tabela de decisões, performance normalizada, estatísticas descritivas, retornos mensais e anuais, distribuição dos retornos, correlação com o índice e entre ações, volatilidade móvel, comparação entre índices, análise setorial, regimes, PCA, liquidez e concentração.
- **Modelo:** formulação matemática (seção 9), solução em Python com Gurobi e heurística (seção 10), validação walk-forward com 20 carteiras fora da amostra por configuração (o briefing pede ao menos cinco), comparação com referências e análise de sensibilidade a `K` e à janela de treino.
- **Avaliação:** TE e TD (as medidas padrão; o MSE, fórmula do guia, é o objetivo do modelo), diferença acumulada, giro, concentração, tempo de cálculo e gap.

Referências metodológicas indicadas no briefing:

- Cornuéjols e Tütüncü (2006), *Optimization Methods in Finance*;
- Sant'Anna, Filomena e Borenstein (2014), *Index Tracking com Controle do Número de Ativos*;
- Sant'Anna et al. (2017), *Index tracking with controlled number of assets using a hybrid heuristic*.
- **Parte 2 (Guia 2, seção 14):** divisão temporal e features só com o treino, dois baselines (Top K por peso e por correlação), Lasso, clustering e MIQP com controle de giro, função de avaliação única, tabela comparativa, gráficos do guia, TE por regime (2022 como crise fora da amostra), custos, modelo nulo, outros cinco testes de robustez e limitações.

## 19. Exportar os resultados

O arquivo final contém dados suficientes para revisar o experimento sem depender das saídas visuais do Colab: contrato dos dados, EDA, janelas, resultados por trimestre, pesos, retornos fora da amostra, resumo, hipóteses e figuras.

In [ ]:
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

config_exportada = {
    "versão": "notebook único (Parte 1 v7.7 + Parte 2)",
    "data_fotografia_universo": DATA_FOTOGRAFIA_UNIVERSO,
    "data_inicial": str(DATA_INICIAL.date()),
    "data_final": str(DATA_FINAL.date()),
    "modo_execucao": MODO_EXECUCAO,
    "janela_treino": JANELA_TREINO,
    "janela_alternativa": JANELA_ALTERNATIVA,
    "horizonte_teste": HORIZONTE_TESTE,
    "cobertura_minima_treino": COBERTURA_MINIMA_TREINO,
    "K_MIQP": list(CFG["K_MIQP"]),
    "K_heuristica": list(K_HEURISTICA),
    "carteira_de_referencia": f"{PRINCIPAL[0]} K={PRINCIPAL[1]} treino {PRINCIPAL[2]}",
    "time_limit_s": CFG["time_limit_s"],
    "mip_gap": MIP_GAP,
    "trimestres_fora_da_amostra": N_TRIMESTRES,
    "periodo_fora_da_amostra": PERIODO_OOS,
    "tempo_total_modelo_min": tempo_total_min,
    "fonte_precos": "Yahoo Finance",
    "universo": "fixo em 30/09/2026; DELL, PANW, ANET e SNDK excluídas antes de 21/09/2026",
}
(PASTA_SAIDA / "configuracao.json").write_text(json.dumps(config_exportada, ensure_ascii=False, indent=2), encoding="utf-8")

tabelas = {
    # dados e EDA
    "componentes_atuais.csv": componentes_atuais,
    "contrato_dados.csv": contrato_dados,
    "qualidade_cobertura.csv": qualidade,
    "auditoria_fator_ajuste.csv": auditoria_fator_ajuste,
    "log_correcoes.csv": log_correcoes,
    "integridade_series.csv": integridade_series,
    "triagem_erros.csv": triagem,
    "outliers_classificados.csv": outliers,
    "verificacao_outliers.csv": verificacao,
    "tabela_decisoes_outliers.csv": tabela_decisoes_outliers.reset_index(),
    "consistencia_universo.csv": consistencia_universo,
    "mudancas_composicao.csv": MUDANCAS_COMPOSICAO,
    "metadados_yahoo.csv": metadados,
    "estatisticas_univariadas.csv": estatisticas,
    "correlacoes_com_indice.csv": correlacoes,
    "pares_redundantes.csv": pares_redundantes,
    "resumo_indices.csv": resumo_indices,
    "retornos_anuais_indices.csv": retornos_anuais_indices.reset_index(),
    "analise_setorial.csv": analise_setorial,
    "beta_risco_idiossincratico.csv": beta_idio,
    "estabilidade_ranking.csv": estabilidade_ranking,
    "correlacao_regimes.csv": correlacao_regimes,
    "assimetria_correlacao.csv": assimetria,
    "pca_variancia.csv": pca_variancia,
    "liquidez.csv": liquidez,
    "pesos_aproximados.csv": pesos_aprox,
    "concentracao.csv": concentracao,
    "checagem_numeros_eda.csv": checagem.reset_index(),
    "sensibilidade_janela.csv": sensibilidade_janela,
    "resumo_janela.csv": resumo_janela.reset_index(),
    # modelo
    "auditoria_janelas.csv": auditoria_janelas,
    "resultados_por_trimestre.csv": resultados_janelas,
    "pesos_carteiras.csv": resultados_pesos,
    "retornos_fora_amostra.csv": resultados_retornos,
    "resumo_modelos.csv": resumo_modelos,
    "comparacao_miqp_heuristica.csv": comparacao_miqp,
    "comparacao_janelas_treino.csv": comparacao_janelas,
    "frequencia_acoes_escolhidas.csv": frequencia,
    "composicao_setorial_carteiras.csv": composicao_setorial_carteiras,
    "hipoteses_eda.csv": hipoteses,
    "sensibilidade_outliers.csv": sensibilidade_outliers,
    "decomposicao_vies.csv": decomposicao_vies,
}
for nome_arquivo, tabela in tabelas.items():
    tabela.to_csv(PASTA_SAIDA / nome_arquivo, index=False)

for nome, b in bases.items():
    chave = nome.lower().replace("&", "").replace(" ", "")
    b["R"].to_csv(PASTA_SAIDA / f"retornos_componentes_{chave}.csv")
    b["y"].to_csv(PASTA_SAIDA / f"retornos_benchmark_{chave}.csv")

# Insumos textuais para a documentação e os slides
def tabela_md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return df.to_string(index=False)


linhas_doc = [
    "# FINOR - insumos para documentação e apresentação (v7.7)",
    "",
    f"- Período dos dados: {DATA_INICIAL.date()} a {DATA_FINAL.date()}; universo fixo de {DATA_FOTOGRAFIA_UNIVERSO}.",
    f"- Modo executado: {MODO_EXECUCAO}; {N_TRIMESTRES} trimestres fora da amostra ({PERIODO_OOS}).",
    f"- Carteira de referência: {PRINCIPAL[0]} com K={PRINCIPAL[1]} e treino de {PRINCIPAL[2]} pregões.",
    f"- MIQP: limite de {CFG['time_limit_s']} s por carteira, partida a quente pela heurística; tempo total do modelo {tempo_total_min:.1f} min.",
    "",
    "## Resultados do modelo",
    "",
] + linhas_md[2:] + [
    "",
    "## Resumo (treino de 504 pregões)",
    "",
    tabela_md(resumo_modelos[resumo_modelos["janela treino"] == JANELA_TREINO][colunas_tabela].round(2)),
    "",
    "## Hipóteses do EDA",
    "",
    tabela_md(hipoteses),
    "",
    "## Alertas para a interpretação",
    "- O universo final é conhecido antecipadamente: viés de sobrevivência (diferença acumulada positiva).",
    "- O S&P 100 usa retorno de preços; o Ibovespa, retorno total aproximado.",
    "- MIQP com gap alto: soluções factíveis, sem prova de ótimo global.",
    "- Giro penalizado só na seção 14; custos simplificados (taxa fixa).",
    "",
    "## Figuras para os slides (pasta figuras/)",
] + [f"- {arquivo.name}" for arquivo in sorted(PASTA_FIGURAS.glob("*.png"))]
(PASTA_SAIDA / "INSUMOS_DOCUMENTACAO.md").write_text("\n".join(linhas_doc), encoding="utf-8")

ambiente = {
    "python": __import__("sys").version,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "yfinance": yf.__version__ if yf else "n/d",
    "gurobi": ".".join(map(str, gp.gurobi.version())) if gp else "n/d",
}
(PASTA_SAIDA / "ambiente.json").write_text(json.dumps(ambiente, ensure_ascii=False, indent=2), encoding="utf-8")

leia_me = f'''FINOR - resultados v7.7
Universo: componentes congelados em 30/09/2026
Fonte de preços: Yahoo Finance
Período: {DATA_INICIAL.date()} a {DATA_FINAL.date()}
Modo: {MODO_EXECUCAO} ({N_TRIMESTRES} trimestres fora da amostra, {PERIODO_OOS})
Tempo do modelo: {tempo_total_min:.1f} minutos

Atenção: esta versão usa o universo final em todo o período e possui viés de sobrevivência.
'''
(PASTA_SAIDA / "LEIA-ME.txt").write_text(leia_me, encoding="utf-8")

print("Resultados gravados em", PASTA_SAIDA, "e figuras em", PASTA_FIGURAS)